In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, roc_curve, classification_report, confusion_matrix
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
TEST_SIZE = 0.2

DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"
print("Exists:", os.path.exists(DATA_PATH), "| Path:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head(3))
print("\nColumns:\n", df.columns.tolist())

# basic missingness
missing = df.isna().mean().sort_values(ascending=False)
print("\nTop missing columns:")
display(missing.head(15))

def detect_target_column(df: pd.DataFrame):
    candidates = ["stroke", "Stroke", "STROKE", "target", "Target", "label", "Label", "y", "Y"]
    for c in candidates:
        if c in df.columns:
            return c

    # fallback: pick a column with 2 unique values (excluding obvious IDs)
    possible = []
    for c in df.columns:
        if "id" in c.lower():
            continue
        nunq = df[c].nunique(dropna=True)
        if nunq == 2:
            possible.append(c)
    # Prefer a column whose name hints at stroke
    for c in possible:
        if "stroke" in c.lower():
            return c
    return possible[0] if possible else None

TARGET_COL = detect_target_column(df)
print("Detected TARGET_COL:", TARGET_COL)

if TARGET_COL is None:
    raise ValueError("Could not auto-detect target column. Please tell me the correct target column name.")

# Show class distribution
y_raw = df[TARGET_COL]
print("\nTarget value counts:")
display(y_raw.value_counts(dropna=False))
print("\nTarget value counts (%):")
display((y_raw.value_counts(normalize=True, dropna=False) * 100).round(3))

# Try to coerce to 0/1 if needed
def coerce_binary_target(y: pd.Series):
    if y.dtype == "O":
        y2 = y.astype(str).str.strip().str.lower()
        mapping = {"yes":1, "no":0, "true":1, "false":0, "stroke":1, "no_stroke":0, "1":1, "0":0}
        if set(y2.unique()) <= set(mapping.keys()):
            return y2.map(mapping).astype(int)
        # if it's two arbitrary strings, map to 0/1
        uniq = list(pd.unique(y2.dropna()))
        if len(uniq) == 2:
            return y2.map({uniq[0]:0, uniq[1]:1}).astype(int)
        return y
    # numeric
    uniq = sorted(pd.unique(y.dropna()))
    if len(uniq) == 2 and set(uniq) <= {0, 1}:
        return y.astype(int)
    if len(uniq) == 2:
        # map min->0, max->1
        return y.map({uniq[0]:0, uniq[1]:1}).astype(int)
    return y

df[TARGET_COL] = coerce_binary_target(df[TARGET_COL])
print("\nAfter coercion, target unique:", sorted(df[TARGET_COL].dropna().unique()))

def find_first_existing(cols, df_cols):
    for c in cols:
        if c in df_cols:
            return c
    return None

# Common variants
SEX_COL = find_first_existing(["sex", "Sex", "gender", "Gender"], df.columns)
RACE_COL = find_first_existing(["race", "Race", "ethnicity", "Ethnicity"], df.columns)
AGE_COL  = find_first_existing(["age", "Age"], df.columns)

print("Protected columns detected:")
print("  SEX_COL :", SEX_COL)
print("  RACE_COL:", RACE_COL)
print("  AGE_COL :", AGE_COL)

# Drop ID-like columns (safe default)
id_like = [c for c in df.columns if "id" in c.lower() and c != TARGET_COL]
print("Dropping ID-like columns:", id_like)
df = df.drop(columns=id_like, errors="ignore")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

# Split (stratified because dataset is imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Identify column types
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Num cols:", len(num_cols), "| Cat cols:", len(cat_cols))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

lr = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs"
)

lr_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", lr)
])

lr_pipe.fit(X_train, y_train)

proba = lr_pipe.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)

roc = roc_auc_score(y_test, proba)
pr  = average_precision_score(y_test, proba)
f1  = f1_score(y_test, pred)

print(f"LogReg | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f} | F1@0.5: {f1:.4f}")
print("\nClassification report:\n", classification_report(y_test, pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

def subgroup_metrics(df_X, y_true, proba, group_col):
    out = []
    if group_col is None or group_col not in df_X.columns:
        return None
    tmp = df_X[[group_col]].copy()
    tmp["y"] = y_true.values
    tmp["p"] = proba

    for g, d in tmp.groupby(group_col):
        if d["y"].nunique() < 2:
            continue
        out.append({
            "group_col": group_col,
            "group": g,
            "n": len(d),
            "pos_rate": float(d["y"].mean()),
            "roc_auc": float(roc_auc_score(d["y"], d["p"])),
            "pr_auc": float(average_precision_score(d["y"], d["p"])),
        })
    return pd.DataFrame(out).sort_values(["group_col", "n"], ascending=[True, False])

audit_frames = []
for col in [SEX_COL, RACE_COL]:
    sm = subgroup_metrics(X_test, y_test, proba, col)
    if sm is not None and len(sm):
        audit_frames.append(sm)

# Age: make age bins if AGE_COL exists
if AGE_COL is not None and AGE_COL in X_test.columns:
    age_bins = pd.cut(X_test[AGE_COL], bins=[0, 40, 55, 65, 200], right=False, include_lowest=True)
    X_test_age = X_test.copy()
    X_test_age["age_bin"] = age_bins.astype(str)
    sm_age = subgroup_metrics(X_test_age, y_test, proba, "age_bin")
    if sm_age is not None and len(sm_age):
        audit_frames.append(sm_age)

if audit_frames:
    audit = pd.concat(audit_frames, ignore_index=True)
    display(audit)
else:
    print("No protected attribute columns detected for subgroup audit (that's okay).")

Exists: True | Path: /kaggle/input/stroke-dataset/Stroke.csv
Shape: (4603, 36)


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233



Columns:
 ['stroke', 'gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'sleep time', 'diabetes', 'hypertension', 'high cholesterol', 'Minutes sedentary activity', 'Coronary Heart Disease', 'Body Mass Index', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein', 'Fasting Glucose', 'Glycohemoglobin', 'energy', 'protein', 'Carbohydrate', 'Dietary fiber', 'Total fat', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Total polyunsaturated fatty acids', 'Potassium', 'Sodium']

Top missing columns:


stroke                      0.0
gender                      0.0
age                         0.0
Race                        0.0
Marital status              0.0
alcohol                     0.0
smoke                       0.0
sleep disorder              0.0
Health Insurance            0.0
General health condition    0.0
depression                  0.0
sleep time                  0.0
diabetes                    0.0
hypertension                0.0
high cholesterol            0.0
dtype: float64

Detected TARGET_COL: stroke

Target value counts:


stroke
0    4241
1     362
Name: count, dtype: int64


Target value counts (%):


stroke
0    92.136
1     7.864
Name: proportion, dtype: float64


After coercion, target unique: [np.int64(0), np.int64(1)]
Protected columns detected:
  SEX_COL : gender
  RACE_COL: Race
  AGE_COL : age
Dropping ID-like columns: ['Triglyceride', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Total polyunsaturated fatty acids']
Num cols: 31 | Cat cols: 0
LogReg | ROC-AUC: 0.6229 | PR-AUC: 0.1555 | F1@0.5: 0.1772

Classification report:
               precision    recall  f1-score   support

           0     0.9381    0.6608    0.7754       849
           1     0.1084    0.4861    0.1772        72

    accuracy                         0.6471       921
   macro avg     0.5232    0.5734    0.4763       921
weighted avg     0.8733    0.6471    0.7286       921

Confusion matrix:
 [[561 288]
 [ 37  35]]


,group_col,group,n,pos_rate,roc_auc,pr_auc
0,gender,2,504,0.077381,0.592225,0.125546
1,gender,1,417,0.079137,0.658933,0.200233
2,Race,3,467,0.094218,0.655599,0.194422
3,Race,4,199,0.060302,0.554813,0.072749
4,Race,1,105,0.085714,0.530093,0.098810
5,Race,2,101,0.029703,0.340136,0.029064
6,Race,5,49,0.081633,0.872222,0.458929
7,age_bin,"[0, 40)",921,0.078176,0.622890,0.155470


In [2]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"
df = pd.read_csv(DATA_PATH)

# Clean column names: strip spaces and normalize multiple spaces
df.columns = [c.strip().replace("  ", " ") for c in df.columns]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

TARGET_COL = "stroke"
assert TARGET_COL in df.columns

# Confirm protected attributes
SEX_COL  = "gender"
RACE_COL = "Race"
AGE_COL  = "age"
print("Protected:", SEX_COL, RACE_COL, AGE_COL)

def show_uniques(col, n=20):
    vals = df[col].dropna().unique()
    vals_sorted = np.sort(vals) if np.issubdtype(df[col].dtype, np.number) else vals
    print(f"{col}: nunique={df[col].nunique()} | sample={vals_sorted[:n]}")

for col in [SEX_COL, RACE_COL, "Marital status", "alcohol", "smoke", "sleep disorder",
            "Health Insurance", "General health condition", "depression",
            "diabetes", "hypertension", "high cholesterol", "Coronary Heart Disease"]:
    if col in df.columns:
        show_uniques(col)

# Check target distribution
print("\nTarget distribution:")
print(df[TARGET_COL].value_counts(), "\n")
print((df[TARGET_COL].value_counts(normalize=True)*100).round(3))

# --- Concept groups (edit later if you want) ---
CONCEPTS = {
    "socio_demo": [
        "gender", "age", "Race", "Marital status", "Health Insurance"
    ],
    "lifestyle_sleep": [
        "alcohol", "smoke", "sleep disorder", "sleep time", "Minutes sedentary activity",
        "General health condition", "depression"
    ],
    "cardio": [
        "hypertension", "Coronary Heart Disease",
        "Systolic blood pressure", "Diastolic blood pressure",
        "Body Mass Index", "Waist Circumference"
    ],
    "metabolic_labs_diet": [
        "diabetes", "high cholesterol",
        "High-density lipoprotein", "Low-density lipoprotein", "Triglyceride",
        "Fasting Glucose", "Glycohemoglobin",
        "energy", "protein", "Carbohydrate", "Dietary fiber",
        "Total fat", "Total saturated fatty acids",
        "Total monounsaturated fatty acids", "Total polyunsaturated fatty acids",
        "Potassium", "Sodium"
    ]
}

# Verify existence
all_used = []
missing = []
for k, cols in CONCEPTS.items():
    for c in cols:
        if c not in df.columns:
            missing.append(c)
        else:
            all_used.append(c)

print("Missing columns in concept mapping:", missing)
print("Total used features:", len(all_used))

# Define feature set used for modeling
FEATURE_COLS = all_used
X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int).copy()

print("X shape:", X.shape, "| y mean (pos rate):", y.mean().round(4))

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Impute + scale
imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc  = scaler.transform(X_test_imp)

print("Scaled shapes:", X_train_sc.shape, X_test_sc.shape)

feature_index = {c:i for i,c in enumerate(FEATURE_COLS)}

concept_slices = {}
for cname, cols in CONCEPTS.items():
    idxs = [feature_index[c] for c in cols if c in feature_index]
    concept_slices[cname] = idxs

print("Concept sizes:")
for k,v in concept_slices.items():
    print(f"  {k}: {len(v)}")

Shape: (4603, 36)
Columns: ['stroke', 'gender', 'age', 'Race', 'Marital status', 'alcohol', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'sleep time', 'diabetes', 'hypertension', 'high cholesterol', 'Minutes sedentary activity', 'Coronary Heart Disease', 'Body Mass Index', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein', 'Fasting Glucose', 'Glycohemoglobin', 'energy', 'protein', 'Carbohydrate', 'Dietary fiber', 'Total fat', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Total polyunsaturated fatty acids', 'Potassium', 'Sodium']
Protected: gender Race age
gender: nunique=2 | sample=[1 2]
Race: nunique=5 | sample=[1 2 3 4 5]
Marital status: nunique=6 | sample=[1 2 3 4 5 6]
alcohol: nunique=2 | sample=[0 1]
smoke: nunique=2 | sample=[0 1]
sleep disorder: nunique=2 | sample=[1 2]
Health Insurance: nunique=2 | sample=[1 2]
Gene

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

class TabDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np.values if hasattr(y_np, "values") else y_np, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 256

train_ds = TabDataset(X_train_sc, y_train)
test_ds  = TabDataset(X_test_sc,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# class imbalance weight: pos_weight = N_neg / N_pos
n_pos = int(y_train.sum())
n_neg = int((y_train == 0).sum())
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)

print("Train size:", len(train_ds), "Test size:", len(test_ds))
print("Pos:", n_pos, "Neg:", n_neg, "pos_weight:", float(pos_weight))

class MLPBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.net(x)

class ConceptMLP(nn.Module):
    def __init__(self, concept_slices, hidden_dim=32, concept_dim=16, head_hidden=32, dropout=0.1):
        super().__init__()
        self.concept_names = list(concept_slices.keys())
        self.concept_slices = concept_slices

        self.concept_blocks = nn.ModuleDict()
        for cname in self.concept_names:
            in_dim = len(self.concept_slices[cname])
            self.concept_blocks[cname] = MLPBlock(in_dim, hidden_dim, concept_dim, dropout=dropout)

        total_concept_dim = concept_dim * len(self.concept_names)

        self.head = nn.Sequential(
            nn.Linear(total_concept_dim, head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1)  # logits
        )

    def forward(self, x):
        concept_vecs = []
        for cname in self.concept_names:
            idxs = self.concept_slices[cname]
            xb = x[:, idxs]
            cb = self.concept_blocks[cname](xb)
            concept_vecs.append(cb)
        z = torch.cat(concept_vecs, dim=1)
        logits = self.head(z).squeeze(1)
        return logits

model = ConceptMLP(concept_slices, hidden_dim=32, concept_dim=16, head_hidden=32, dropout=0.15).to(DEVICE)
print(model)

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def evaluate_model(model, loader):
    model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits)
            all_p.append(probs.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())
    p = np.concatenate(all_p)
    y = np.concatenate(all_y).astype(int)
    roc = roc_auc_score(y, p)
    pr  = average_precision_score(y, p)
    pred = (p >= 0.5).astype(int)
    f1  = f1_score(y, pred)
    return roc, pr, f1, p, y

EPOCHS = 40
LR = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_pr = -1.0
best_state = None
patience = 7
pat = 0

for epoch in range(1, EPOCHS+1):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    roc, pr, f1, _, _ = evaluate_model(model, test_loader)
    print(f"Epoch {epoch:02d} | train_loss={np.mean(losses):.4f} | ROC-AUC={roc:.4f} | PR-AUC={pr:.4f} | F1@0.5={f1:.4f}")

    # early stopping on PR-AUC (better for imbalance)
    if pr > best_pr + 1e-4:
        best_pr = pr
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

# load best
if best_state is not None:
    model.load_state_dict(best_state)
print("Best PR-AUC:", best_pr)

from sklearn.metrics import classification_report, confusion_matrix

roc, pr, f1, p_test, y_test_np = evaluate_model(model, test_loader)
pred = (p_test >= 0.5).astype(int)

print(f"ConceptMLP (clean) | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f} | F1@0.5: {f1:.4f}")
print("\nClassification report:\n", classification_report(y_test_np, pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test_np, pred))

def subgroup_report(X_test_raw, y_true, proba, group_col):
    rows = []
    for g, idx in X_test_raw.groupby(group_col).groups.items():
        yt = y_true[idx]
        pt = proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        rows.append({
            "group_col": group_col,
            "group": g,
            "n": len(idx),
            "pos_rate": float(yt.mean()),
            "roc_auc": float(roc_auc_score(yt, pt)),
            "pr_auc": float(average_precision_score(yt, pt)),
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False)

# Make sure indices align: y_test is a Series with original indices
p_series = pd.Series(p_test, index=y_test.index)

reports = []
for col in [SEX_COL, RACE_COL]:
    rep = subgroup_report(X_test, y_test, p_series, col)
    reports.append(rep)

# age bins
age_bins = pd.cut(X_test[AGE_COL], bins=[0, 40, 55, 65, 200], right=False, include_lowest=True)
X_test_age = X_test.copy()
X_test_age["age_bin"] = age_bins.astype(str)
rep_age = subgroup_report(X_test_age, y_test, p_series, "age_bin")
reports.append(rep_age)

audit = pd.concat(reports, ignore_index=True)
display(audit)

DEVICE: cpu
Train size: 3682 Test size: 921
Pos: 290 Neg: 3392 pos_weight: 11.696551322937012
ConceptMLP(
  (concept_blocks): ModuleDict(
    (socio_demo): MLPBlock(
      (net): Sequential(
        (0): Linear(in_features=5, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.15, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
        (4): ReLU()
      )
    )
    (lifestyle_sleep): MLPBlock(
      (net): Sequential(
        (0): Linear(in_features=7, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.15, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
        (4): ReLU()
      )
    )
    (cardio): MLPBlock(
      (net): Sequential(
        (0): Linear(in_features=6, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.15, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
        (4): ReLU()
      )
    )
    (metabolic_labs_diet): 

,group_col,group,n,pos_rate,roc_auc,pr_auc
0,gender,2,504,0.077381,0.522856,0.092769
1,gender,1,417,0.079137,0.610717,0.165803
2,Race,3,467,0.094218,0.625779,0.177375
3,Race,4,199,0.060302,0.527629,0.064668
4,Race,1,105,0.085714,0.408565,0.090652
5,Race,2,101,0.029703,0.231293,0.025059
6,Race,5,49,0.081633,0.705556,0.358323
7,age_bin,"[0, 40)",921,0.078176,0.564160,0.121642


In [4]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    balanced_accuracy_score, precision_score, recall_score, confusion_matrix
)

def eval_with_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "thr": thr,
        "acc": (pred == y_true).mean(),
        "bal_acc": balanced_accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "tn_fp_fn_tp": confusion_matrix(y_true, pred).ravel()
    }

def find_best_threshold(y_true, proba, metric="f1"):
    thrs = np.linspace(0.01, 0.99, 99)
    rows = [eval_with_threshold(y_true, proba, t) for t in thrs]
    dfm = pd.DataFrame(rows)
    best = dfm.iloc[dfm[metric].astype(float).values.argmax()]
    return best, dfm

def overall_metrics(y_true, proba):
    return {
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba)
    }

# Use the probabilities you already computed: p_test, y_test_np
# If you re-ran, recompute quickly:
roc, pr, f1, p_test, y_test_np = evaluate_model(model, test_loader)

print("ConceptMLP ranking metrics:", overall_metrics(y_test_np, p_test))

best_f1, grid = find_best_threshold(y_test_np, p_test, metric="f1")
best_bal, _   = find_best_threshold(y_test_np, p_test, metric="bal_acc")

print("\nBest threshold by F1:")
print(best_f1.to_dict())

print("\nBest threshold by Balanced Accuracy:")
print(best_bal.to_dict())

display(grid.sort_values("f1", ascending=False).head(10))

!pip -q install xgboost

from xgboost import XGBClassifier

# We'll use the same imputed/scaled arrays (X_train_sc, X_test_sc)
xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_weight=2,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb.fit(X_train_sc, y_train)

p_xgb = xgb.predict_proba(X_test_sc)[:, 1]

print("XGB ranking metrics:", overall_metrics(y_test, p_xgb))

best_xgb_f1, grid_xgb = find_best_threshold(y_test.values, p_xgb, metric="f1")
best_xgb_bal, _       = find_best_threshold(y_test.values, p_xgb, metric="bal_acc")

print("\nXGB best threshold by F1:")
print(best_xgb_f1.to_dict())

print("\nXGB best threshold by Balanced Accuracy:")
print(best_xgb_bal.to_dict())

display(grid_xgb.sort_values("f1", ascending=False).head(10))

p_xgb_series = pd.Series(p_xgb, index=y_test.index)

def subgroup_report_df(X_test_raw, y_true_ser, proba_ser, group_col):
    rows = []
    for g, idx in X_test_raw.groupby(group_col).groups.items():
        yt = y_true_ser.loc[idx].values
        pt = proba_ser.loc[idx].values
        if len(np.unique(yt)) < 2:
            continue
        rows.append({
            "group_col": group_col,
            "group": g,
            "n": len(idx),
            "pos_rate": float(yt.mean()),
            "roc_auc": float(roc_auc_score(yt, pt)),
            "pr_auc": float(average_precision_score(yt, pt)),
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False)

rep_gender = subgroup_report_df(X_test, y_test, p_xgb_series, SEX_COL)
rep_race   = subgroup_report_df(X_test, y_test, p_xgb_series, RACE_COL)

age_bins = pd.cut(X_test[AGE_COL], bins=[0, 40, 55, 65, 200], right=False, include_lowest=True)
X_test_age = X_test.copy()
X_test_age["age_bin"] = age_bins.astype(str)
rep_age    = subgroup_report_df(X_test_age, y_test, p_xgb_series, "age_bin")

audit_xgb = pd.concat([rep_gender, rep_race, rep_age], ignore_index=True)
display(audit_xgb)

ConceptMLP ranking metrics: {'roc_auc': np.float64(0.5641604502028531), 'pr_auc': np.float64(0.1216420613627141)}

Best threshold by F1:
{'thr': 0.53, 'acc': 0.8371335504885994, 'bal_acc': 0.5430408323517865, 'f1': 0.15730337078651685, 'precision': 0.1320754716981132, 'recall': 0.19444444444444445, 'tn_fp_fn_tp': array([757,  92,  58,  14])}

Best threshold by Balanced Accuracy:
{'thr': 0.53, 'acc': 0.8371335504885994, 'bal_acc': 0.5430408323517865, 'f1': 0.15730337078651685, 'precision': 0.1320754716981132, 'recall': 0.19444444444444445, 'tn_fp_fn_tp': array([757,  92,  58,  14])}


,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
52,0.53,0.837134,0.543041,0.157303,0.132075,0.194444,"[757, 92, 58, 14]"
46,0.47,0.269273,0.540096,0.155583,0.085517,0.861111,"[186, 663, 10, 62]"
51,0.52,0.726384,0.540170,0.154362,0.101770,0.319444,"[646, 203, 49, 23]"
47,0.48,0.320304,0.535998,0.154054,0.085329,0.791667,"[238, 611, 15, 57]"
45,0.46,0.224756,0.535017,0.154028,0.084197,0.902778,"[142, 707, 7, 65]"
49,0.50,0.485342,0.536538,0.153571,0.088115,0.597222,"[404, 445, 29, 43]"
42,0.43,0.137894,0.526036,0.151709,0.082176,0.986111,"[56, 793, 1, 71]"
43,0.44,0.154180,0.522158,0.150491,0.081657,0.958333,"[73, 776, 3, 69]"
41,0.42,0.123779,0.518379,0.149631,0.080958,0.986111,"[43, 806, 1, 71]"
44,0.45,0.182410,0.512048,0.147225,0.080148,0.902778,"[103, 746, 7, 65]"


XGB ranking metrics: {'roc_auc': np.float64(0.5680375605287266), 'pr_auc': np.float64(0.10410177804415605)}

XGB best threshold by F1:
{'thr': 0.09, 'acc': 0.758957654723127, 'bal_acc': 0.5705486847271299, 'f1': 0.18382352941176472, 'precision': 0.125, 'recall': 0.3472222222222222, 'tn_fp_fn_tp': array([674, 175,  47,  25])}

XGB best threshold by Balanced Accuracy:
{'thr': 0.09, 'acc': 0.758957654723127, 'bal_acc': 0.5705486847271299, 'f1': 0.18382352941176472, 'precision': 0.125, 'recall': 0.3472222222222222, 'tn_fp_fn_tp': array([674, 175,  47,  25])}


,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
8,0.09,0.758958,0.570549,0.183824,0.125000,0.347222,"[674, 175, 47, 25]"
9,0.10,0.779587,0.562672,0.178138,0.125714,0.305556,"[696, 153, 50, 22]"
4,0.05,0.627579,0.562844,0.169492,0.102639,0.486111,"[543, 306, 37, 35]"
7,0.08,0.730727,0.555237,0.167785,0.110619,0.347222,"[648, 201, 47, 25]"
6,0.07,0.703583,0.553224,0.165138,0.105882,0.375000,"[621, 228, 45, 27]"
10,0.11,0.801303,0.549028,0.164384,0.122449,0.250000,"[720, 129, 54, 18]"
2,0.03,0.482085,0.553838,0.161687,0.092555,0.638889,"[398, 451, 26, 46]"
0,0.01,0.247557,0.534673,0.153846,0.084337,0.875000,"[165, 684, 9, 63]"
3,0.04,0.550489,0.533741,0.151639,0.088942,0.513889,"[470, 379, 35, 37]"
5,0.06,0.659066,0.535434,0.151351,0.093960,0.388889,"[579, 270, 44, 28]"


,group_col,group,n,pos_rate,roc_auc,pr_auc
0,gender,2,504,0.077381,0.585277,0.105792
1,gender,1,417,0.079137,0.558081,0.126258
2,Race,3,467,0.094218,0.617612,0.149538
3,Race,4,199,0.060302,0.467914,0.058257
4,Race,1,105,0.085714,0.444444,0.083603
5,Race,2,101,0.029703,0.190476,0.023821
6,Race,5,49,0.081633,0.855556,0.287990
7,age_bin,"[0, 40)",921,0.078176,0.568038,0.104102


In [5]:
!pip -q install catboost

from catboost import CatBoostClassifier

# CatBoost likes raw/unscaled features; we'll use imputed (not scaled) arrays
X_train_cb = X_train_imp.copy()
X_test_cb  = X_test_imp.copy()

cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=6.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=200
)

cb.fit(X_train_cb, y_train, eval_set=(X_test_cb, y_test), use_best_model=True)

p_cb = cb.predict_proba(X_test_cb)[:, 1]

print("CatBoost ranking metrics:", overall_metrics(y_test.values, p_cb))

best_cb_f1, grid_cb = find_best_threshold(y_test.values, p_cb, metric="f1")
best_cb_bal, _      = find_best_threshold(y_test.values, p_cb, metric="bal_acc")

print("\nCatBoost best threshold by F1:")
print(best_cb_f1.to_dict())

print("\nCatBoost best threshold by Balanced Accuracy:")
print(best_cb_bal.to_dict())

display(grid_cb.sort_values("f1", ascending=False).head(10))

p_cb_series = pd.Series(p_cb, index=y_test.index)

rep_gender_cb = subgroup_report_df(X_test, y_test, p_cb_series, SEX_COL)
rep_race_cb   = subgroup_report_df(X_test, y_test, p_cb_series, RACE_COL)

age_bins = pd.cut(X_test[AGE_COL], bins=[0, 40, 55, 65, 200], right=False, include_lowest=True)
X_test_age = X_test.copy()
X_test_age["age_bin"] = age_bins.astype(str)
rep_age_cb  = subgroup_report_df(X_test_age, y_test, p_cb_series, "age_bin")

audit_cb = pd.concat([rep_gender_cb, rep_race_cb, rep_age_cb], ignore_index=True)
display(audit_cb)

class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, reduction="mean"):
        """
        alpha: weight for positive class (0..1). Higher -> emphasize positives.
        gamma: focusing parameter
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # targets: (B,)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = torch.where(targets == 1, p, 1 - p)
        focal = (1 - pt).pow(self.gamma) * bce
        alpha_t = torch.where(targets == 1, torch.tensor(self.alpha, device=logits.device), torch.tensor(1-self.alpha, device=logits.device))
        loss = alpha_t * focal
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

class MLPBlockBN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dp  = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dp(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        return x

class ConceptMLP_BN(nn.Module):
    def __init__(self, concept_slices, hidden_dim=64, concept_dim=24, head_hidden=64, dropout=0.1):
        super().__init__()
        self.concept_names = list(concept_slices.keys())
        self.concept_slices = concept_slices

        self.concept_blocks = nn.ModuleDict()
        for cname in self.concept_names:
            in_dim = len(self.concept_slices[cname])
            self.concept_blocks[cname] = MLPBlockBN(in_dim, hidden_dim, concept_dim, dropout=dropout)

        total_dim = concept_dim * len(self.concept_names)
        self.head = nn.Sequential(
            nn.Linear(total_dim, head_hidden),
            nn.BatchNorm1d(head_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1)
        )

    def forward(self, x):
        zs = []
        for cname in self.concept_names:
            idxs = self.concept_slices[cname]
            zs.append(self.concept_blocks[cname](x[:, idxs]))
        z = torch.cat(zs, dim=1)
        return self.head(z).squeeze(1)

model2 = ConceptMLP_BN(concept_slices, hidden_dim=64, concept_dim=24, head_hidden=64, dropout=0.15).to(DEVICE)
print(model2)

def train_model(model, train_loader, test_loader, epochs=50, lr=2e-3, patience=8):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = FocalLossWithLogits(alpha=0.80, gamma=2.0)  # you can tune alpha 0.7-0.9

    best_pr = -1.0
    best_state = None
    pat = 0

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        roc, pr, f1, p, y = evaluate_model(model, test_loader)
        print(f"Epoch {ep:02d} | loss={np.mean(losses):.4f} | ROC={roc:.4f} | PR={pr:.4f} | F1@0.5={f1:.4f}")

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

model2 = train_model(model2, train_loader, test_loader, epochs=60, lr=2e-3, patience=10)

roc2, pr2, f12, p2, y2 = evaluate_model(model2, test_loader)
print("\nConceptMLP++ ranking metrics:", overall_metrics(y2, p2))

best2_f1, grid2 = find_best_threshold(y2, p2, metric="f1")
best2_bal, _    = find_best_threshold(y2, p2, metric="bal_acc")
print("\nConceptMLP++ best threshold by F1:", best2_f1.to_dict())
print("ConceptMLP++ best threshold by BalAcc:", best2_bal.to_dict())
display(grid2.sort_values("f1", ascending=False).head(10))

0:	test: 0.4437574	best: 0.4437574 (0)	total: 57.9ms	remaining: 1m 55s
200:	test: 0.6237567	best: 0.6287135 (159)	total: 968ms	remaining: 8.66s
400:	test: 0.6288771	best: 0.6311020 (360)	total: 1.87s	remaining: 7.46s
600:	test: 0.6168204	best: 0.6311020 (360)	total: 2.79s	remaining: 6.49s
800:	test: 0.6052545	best: 0.6311020 (360)	total: 3.71s	remaining: 5.55s
1000:	test: 0.6019173	best: 0.6311020 (360)	total: 4.64s	remaining: 4.63s
1200:	test: 0.5988909	best: 0.6311020 (360)	total: 5.63s	remaining: 3.75s
1400:	test: 0.5962898	best: 0.6311020 (360)	total: 6.57s	remaining: 2.81s
1600:	test: 0.5972059	best: 0.6311020 (360)	total: 7.49s	remaining: 1.87s
1800:	test: 0.5970913	best: 0.6311020 (360)	total: 8.41s	remaining: 929ms
1999:	test: 0.5989563	best: 0.6311020 (360)	total: 9.32s	remaining: 0us

bestTest = 0.63110195
bestIteration = 360

Shrink model to first 361 iterations.
CatBoost ranking metrics: {'roc_auc': np.float64(0.6311019500065437), 'pr_auc': np.float64(0.12476218982183299)}


,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
5,0.06,0.564604,0.617663,0.196393,0.114754,0.680556,"[471, 378, 23, 49]"
4,0.05,0.498371,0.619871,0.192308,0.110000,0.763889,"[404, 445, 17, 55]"
6,0.07,0.621064,0.591088,0.186480,0.112045,0.555556,"[532, 317, 32, 40]"
3,0.04,0.418024,0.601713,0.180428,0.101375,0.819444,"[326, 523, 13, 59]"
2,0.03,0.311618,0.588486,0.172324,0.095101,0.916667,"[221, 628, 6, 66]"
10,0.11,0.788274,0.554672,0.170213,0.122699,0.277778,"[706, 143, 52, 20]"
7,0.08,0.664495,0.557445,0.167116,0.103679,0.430556,"[581, 268, 41, 31]"
8,0.09,0.711183,0.550991,0.163522,0.105691,0.361111,"[629, 220, 46, 26]"
11,0.12,0.809989,0.547384,0.162679,0.124088,0.236111,"[729, 120, 55, 17]"
9,0.10,0.748100,0.545593,0.159420,0.107843,0.305556,"[667, 182, 50, 22]"


,group_col,group,n,pos_rate,roc_auc,pr_auc
0,gender,2,504,0.077381,0.608437,0.112161
1,gender,1,417,0.079137,0.665483,0.181471
2,Race,3,467,0.094218,0.666452,0.155854
3,Race,4,199,0.060302,0.526738,0.067054
4,Race,1,105,0.085714,0.549769,0.105684
5,Race,2,101,0.029703,0.309524,0.027515
6,Race,5,49,0.081633,0.872222,0.639205
7,age_bin,"[0, 40)",921,0.078176,0.631102,0.124762


ConceptMLP_BN(
  (concept_blocks): ModuleDict(
    (socio_demo): MLPBlockBN(
      (fc1): Linear(in_features=5, out_features=64, bias=True)
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dp): Dropout(p=0.15, inplace=False)
      (fc2): Linear(in_features=64, out_features=24, bias=True)
      (bn2): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (lifestyle_sleep): MLPBlockBN(
      (fc1): Linear(in_features=7, out_features=64, bias=True)
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dp): Dropout(p=0.15, inplace=False)
      (fc2): Linear(in_features=64, out_features=24, bias=True)
      (bn2): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (cardio): MLPBlockBN(
      (fc1): Linear(in_features=6, out_features=64, bias=True)
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_ru

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
47,0.48,0.783931,0.584094,0.200803,0.141243,0.347222,"[697, 152, 47, 25]"
48,0.49,0.834962,0.573641,0.200000,0.161017,0.263889,"[750, 99, 53, 19]"
46,0.47,0.716612,0.592069,0.196923,0.126482,0.444444,"[628, 221, 40, 32]"
45,0.46,0.640608,0.601688,0.194647,0.117994,0.555556,"[550, 299, 32, 40]"
44,0.45,0.559175,0.608363,0.191235,0.111628,0.666667,"[467, 382, 24, 48]"
43,0.44,0.474484,0.600559,0.182432,0.103846,0.750000,"[383, 466, 18, 54]"
42,0.43,0.403909,0.587701,0.174436,0.097808,0.805556,"[314, 535, 14, 58]"
49,0.50,0.862106,0.550231,0.169935,0.160494,0.180556,"[781, 68, 59, 13]"
41,0.42,0.317047,0.566009,0.164675,0.091043,0.861111,"[230, 619, 10, 62]"
40,0.41,0.258415,0.559629,0.161963,0.088829,0.916667,"[172, 677, 6, 66]"


In [6]:
from torch.optim.lr_scheduler import OneCycleLR

def train_model_focal(model, train_loader, test_loader, alpha=0.75, gamma=2.0, epochs=40, max_lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-4)
    sched = OneCycleLR(opt, max_lr=max_lr, steps_per_epoch=len(train_loader), epochs=epochs)
    crit = FocalLossWithLogits(alpha=alpha, gamma=gamma)

    best_pr = -1.0
    best_state = None
    pat, patience = 0, 8

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())

        roc, pr, f1, _, _ = evaluate_model(model, test_loader)
        print(f"ep {ep:02d} | loss={np.mean(losses):.4f} | ROC={roc:.4f} | PR={pr:.4f} | F1={f1:.4f}")

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    roc, pr, f1, p, y = evaluate_model(model, test_loader)
    return model, {"roc_auc": roc, "pr_auc": pr, "f1@0.5": f1}, p, y

alphas = [0.65, 0.75, 0.85]
alpha_results = []

for a in alphas:
    print("\n==== Training ConceptMLP++ with focal alpha =", a, "====")
    m = ConceptMLP_BN(concept_slices, hidden_dim=96, concept_dim=32, head_hidden=96, dropout=0.10).to(DEVICE)
    m, met, p, y = train_model_focal(m, train_loader, test_loader, alpha=a, gamma=2.0, epochs=45, max_lr=3e-3)
    alpha_results.append({"alpha": a, **met})
    print("Final:", met)

alpha_df = pd.DataFrame(alpha_results).sort_values("pr_auc", ascending=False)
display(alpha_df)


==== Training ConceptMLP++ with focal alpha = 0.65 ====
ep 01 | loss=0.0480 | ROC=0.5852 | PR=0.1107 | F1=0.0476
ep 02 | loss=0.0427 | ROC=0.5881 | PR=0.1258 | F1=0.1188
ep 03 | loss=0.0391 | ROC=0.5912 | PR=0.1494 | F1=0.1474
ep 04 | loss=0.0348 | ROC=0.5966 | PR=0.1458 | F1=0.1319
ep 05 | loss=0.0346 | ROC=0.5979 | PR=0.1348 | F1=0.0750
ep 06 | loss=0.0324 | ROC=0.5835 | PR=0.1282 | F1=0.0482
ep 07 | loss=0.0313 | ROC=0.5792 | PR=0.1182 | F1=0.0471
ep 08 | loss=0.0304 | ROC=0.5885 | PR=0.1190 | F1=0.0476
ep 09 | loss=0.0285 | ROC=0.5832 | PR=0.1034 | F1=0.0440
ep 10 | loss=0.0281 | ROC=0.5705 | PR=0.1092 | F1=0.0625
ep 11 | loss=0.0269 | ROC=0.5803 | PR=0.0985 | F1=0.0488
Early stop.
Final: {'roc_auc': np.float64(0.5912347860227719), 'pr_auc': np.float64(0.14935018453788496), 'f1@0.5': 0.14736842105263157}

==== Training ConceptMLP++ with focal alpha = 0.75 ====
ep 01 | loss=0.0419 | ROC=0.5273 | PR=0.1071 | F1=0.0000
ep 02 | loss=0.0373 | ROC=0.5724 | PR=0.1167 | F1=0.0460
ep 03 | 

,alpha,roc_auc,pr_auc,f1@0.5
0,0.65,0.591235,0.149350,0.147368
1,0.75,0.636468,0.142231,0.190476
2,0.85,0.589632,0.128478,0.151786


In [7]:
from sklearn.model_selection import train_test_split

# Start from original cleaned dataframe df and FEATURE_COLS
X_all = df[FEATURE_COLS].copy()
y_all = df[TARGET_COL].astype(int).copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=RANDOM_STATE
)

print("Train/Val/Test:", X_tr.shape, X_va.shape, X_te.shape)
print("Pos rates:", y_tr.mean().round(4), y_va.mean().round(4), y_te.mean().round(4))

# Categorical-coded columns (treat as categorical)
CAT_COLS = [
    "gender", "Race", "Marital status", "sleep disorder",
    "Health Insurance", "General health condition", "depression"
]

# Binary numeric columns (0/1) – keep numeric
BIN_COLS = [
    "alcohol", "smoke", "diabetes", "hypertension", "high cholesterol", "Coronary Heart Disease"
]

# Everything else is continuous numeric
CONT_COLS = [c for c in FEATURE_COLS if (c not in CAT_COLS) and (c not in BIN_COLS)]
print("CAT:", len(CAT_COLS), CAT_COLS)
print("BIN:", len(BIN_COLS), BIN_COLS)
print("CONT:", len(CONT_COLS))

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Preprocess:
# - CAT: impute most_frequent + onehot
# - BIN: impute most_frequent (keep numeric)
# - CONT: impute median + scale
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore"))
])

bin_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent"))
])

cont_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler())
])

prep_nn = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, CAT_COLS),
        ("bin", bin_pipe, BIN_COLS),
        ("cont", cont_pipe, CONT_COLS),
    ],
    remainder="drop"
)

Xtr_np = prep_nn.fit_transform(X_tr)
Xva_np = prep_nn.transform(X_va)
Xte_np = prep_nn.transform(X_te)

# Ensure dense numpy arrays (OneHot gives sparse)
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

Xtr_np = to_dense(Xtr_np).astype(np.float32)
Xva_np = to_dense(Xva_np).astype(np.float32)
Xte_np = to_dense(Xte_np).astype(np.float32)

print("NN matrices:", Xtr_np.shape, Xva_np.shape, Xte_np.shape)

# Build feature names after preprocessing
cat_feature_names = list(prep_nn.named_transformers_["cat"].named_steps["oh"].get_feature_names_out(CAT_COLS))
bin_feature_names = BIN_COLS[:]  # unchanged
cont_feature_names = CONT_COLS[:]  # scaled but same names
NN_FEATURES = cat_feature_names + bin_feature_names + cont_feature_names

feat2idx = {f:i for i,f in enumerate(NN_FEATURES)}

def concept_cols_to_indices(concept_cols):
    idxs = []
    for c in concept_cols:
        if c in CAT_COLS:
            # add all one-hot features starting with "col_"
            prefix = c + "_"
            idxs.extend([feat2idx[f] for f in NN_FEATURES if f.startswith(prefix)])
        elif c in BIN_COLS:
            idxs.append(feat2idx[c])
        else:
            idxs.append(feat2idx[c])
    return sorted(list(set(idxs)))

CONCEPTS_NN = {
    "socio_demo": ["gender", "age", "Race", "Marital status", "Health Insurance"],
    "lifestyle_sleep": ["alcohol", "smoke", "sleep disorder", "sleep time", "Minutes sedentary activity",
                        "General health condition", "depression"],
    "cardio": ["hypertension", "Coronary Heart Disease", "Systolic blood pressure", "Diastolic blood pressure",
               "Body Mass Index", "Waist Circumference"],
    "metabolic_labs_diet": ["diabetes", "high cholesterol", "High-density lipoprotein", "Low-density lipoprotein",
                            "Triglyceride", "Fasting Glucose", "Glycohemoglobin",
                            "energy", "protein", "Carbohydrate", "Dietary fiber", "Total fat",
                            "Total saturated fatty acids", "Total monounsaturated fatty acids",
                            "Total polyunsaturated fatty acids", "Potassium", "Sodium"]
}

concept_slices_nn = {k: concept_cols_to_indices(v) for k,v in CONCEPTS_NN.items()}
print("Concept sizes after one-hot:")
for k,v in concept_slices_nn.items():
    print(k, len(v))

D_nn = len(NN_FEATURES)

from torch.utils.data import WeightedRandomSampler

class NNDataset(Dataset):
    def __init__(self, X_np, y_ser):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_ser.values, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = NNDataset(Xtr_np, y_tr)
val_ds   = NNDataset(Xva_np, y_va)
test_ds  = NNDataset(Xte_np, y_te)

# weights for sampler
ytr_np = y_tr.values
class_counts = np.bincount(ytr_np)
w0 = 1.0 / class_counts[0]
w1 = 1.0 / class_counts[1]
weights = np.where(ytr_np == 1, w1, w0)
sampler = WeightedRandomSampler(weights=torch.tensor(weights, dtype=torch.float32),
                                num_samples=len(weights),
                                replacement=True)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Counts train:", class_counts, "| sampler weights:", (w0, w1))

# Recreate model on the new feature space
model_nn = ConceptMLP_BN(concept_slices_nn, hidden_dim=96, concept_dim=32, head_hidden=96, dropout=0.10).to(DEVICE)

def evaluate_probs(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            ps.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.cpu().numpy())
    p = np.concatenate(ps)
    y = np.concatenate(ys).astype(int)
    return p, y

def train_focal(model, train_loader, val_loader, epochs=50, max_lr=3e-3, alpha=0.85, gamma=2.0):
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-4)
    sched = OneCycleLR(opt, max_lr=max_lr, steps_per_epoch=len(train_loader), epochs=epochs)
    crit = FocalLossWithLogits(alpha=alpha, gamma=gamma)

    best_pr = -1.0
    best_state = None
    pat, patience = 0, 10

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())

        p_val, y_val = evaluate_probs(model, val_loader)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

model_nn = train_focal(model_nn, train_loader, val_loader, epochs=60, max_lr=3e-3, alpha=0.85, gamma=2.0)

# Evaluate on TEST
p_test, y_test = evaluate_probs(model_nn, test_loader)
print("\nTEST ranking metrics:", {"roc_auc": roc_auc_score(y_test, p_test), "pr_auc": average_precision_score(y_test, p_test)})

best_thr, grid = find_best_threshold(y_test, p_test, metric="f1")
print("Best thr by F1:", best_thr.to_dict())
display(grid.sort_values("f1", ascending=False).head(10))

Train/Val/Test: (2945, 35) (737, 35) (921, 35)
Pos rates: 0.0788 0.0787 0.0782
CAT: 7 ['gender', 'Race', 'Marital status', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression']
BIN: 6 ['alcohol', 'smoke', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease']
CONT: 22
NN matrices: (2945, 53) (737, 53) (921, 53)
Concept sizes after one-hot:
socio_demo 16
lifestyle_sleep 14
cardio 6
metabolic_labs_diet 17
Counts train: [2713  232] | sampler weights: (np.float64(0.00036859565057132326), np.float64(0.004310344827586207))
ep 01 | loss=0.1064 | VAL ROC=0.5752 | VAL PR=0.1093
ep 02 | loss=0.0867 | VAL ROC=0.6494 | VAL PR=0.1523
ep 03 | loss=0.0718 | VAL ROC=0.6806 | VAL PR=0.1516
ep 04 | loss=0.0579 | VAL ROC=0.7015 | VAL PR=0.1592
ep 05 | loss=0.0466 | VAL ROC=0.7089 | VAL PR=0.1594
ep 06 | loss=0.0408 | VAL ROC=0.7021 | VAL PR=0.1571
ep 07 | loss=0.0386 | VAL ROC=0.6879 | VAL PR=0.1483
ep 08 | loss=0.0352 | VAL ROC=0.7022 | VAL PR=0.1595
ep 09 | 

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
52,0.53,0.504886,0.597983,0.182796,0.104938,0.708333,"[414, 435, 21, 51]"
54,0.55,0.558089,0.588707,0.181087,0.105882,0.625000,"[469, 380, 27, 45]"
53,0.54,0.526602,0.590695,0.180451,0.104348,0.666667,"[437, 412, 24, 48]"
51,0.52,0.473398,0.587260,0.176570,0.100580,0.722222,"[384, 465, 20, 52]"
50,0.51,0.451683,0.581836,0.173486,0.098330,0.736111,"[363, 486, 19, 53]"
48,0.49,0.398480,0.578401,0.170659,0.095638,0.791667,"[310, 539, 15, 57]"
47,0.48,0.377850,0.573567,0.168360,0.094003,0.805556,"[290, 559, 14, 58]"
49,0.50,0.416938,0.569346,0.167442,0.094241,0.750000,"[330, 519, 18, 54]"
55,0.56,0.578719,0.561764,0.167382,0.098985,0.541667,"[494, 355, 33, 39]"
44,0.45,0.318132,0.572953,0.167109,0.092375,0.875000,"[230, 619, 9, 63]"


In [8]:
# Rebuild loaders without sampler
train_loader_ns = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
val_loader      = DataLoader(val_ds, batch_size=256, shuffle=False, drop_last=False)
test_loader     = DataLoader(test_ds, batch_size=256, shuffle=False, drop_last=False)

# pos_weight for BCE (moderate)
n_pos = int(y_tr.sum())
n_neg = int((y_tr == 0).sum())
pos_weight_bce = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print("pos_weight_bce:", float(pos_weight_bce))

from torch.optim.lr_scheduler import OneCycleLR

def train_bce_posweight(model, train_loader, val_loader, epochs=50, max_lr=2e-3, weight_decay=5e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
    sched = OneCycleLR(opt, max_lr=max_lr, steps_per_epoch=len(train_loader), epochs=epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight_bce)

    best_score = -1.0  # val ROC
    best_state = None
    pat, patience = 0, 10

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())

        p_val, y_val = evaluate_probs(model, val_loader)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_score + 1e-4:
            best_score = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# Slightly smaller model + stronger dropout (less overfit)
model_nn2 = ConceptMLP_BN(concept_slices_nn, hidden_dim=64, concept_dim=24, head_hidden=64, dropout=0.20).to(DEVICE)

model_nn2 = train_bce_posweight(model_nn2, train_loader_ns, val_loader, epochs=60, max_lr=2e-3, weight_decay=8e-4)

# TEST eval
p_test2, y_test2 = evaluate_probs(model_nn2, test_loader)
print("\nTEST ranking metrics:", {"roc_auc": roc_auc_score(y_test2, p_test2), "pr_auc": average_precision_score(y_test2, p_test2)})

best_thr2, grid2 = find_best_threshold(y_test2, p_test2, metric="f1")
print("Best thr by F1:", best_thr2.to_dict())
display(grid2.sort_values("f1", ascending=False).head(10))

def train_one_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    m = ConceptMLP_BN(concept_slices_nn, hidden_dim=64, concept_dim=24, head_hidden=64, dropout=0.20).to(DEVICE)
    m = train_bce_posweight(m, train_loader_ns, val_loader, epochs=50, max_lr=2e-3, weight_decay=8e-4)
    p_te, y_te = evaluate_probs(m, test_loader)
    return p_te, y_te

seeds = [1, 2, 3]
ps = []
for s in seeds:
    print("\nTraining seed:", s)
    p_te, y_te = train_one_seed(s)
    ps.append(p_te)

p_ens = np.mean(ps, axis=0)
print("\nENSEMBLE TEST metrics:", {"roc_auc": roc_auc_score(y_te, p_ens), "pr_auc": average_precision_score(y_te, p_ens)})

best_thrE, gridE = find_best_threshold(y_te, p_ens, metric="f1")
print("Ensemble best thr by F1:", best_thrE.to_dict())
display(gridE.sort_values("f1", ascending=False).head(10))

pos_weight_bce: 11.693965911865234
ep 01 | loss=1.3420 | VAL ROC=0.4364 | VAL PR=0.0910
ep 02 | loss=1.3047 | VAL ROC=0.4658 | VAL PR=0.0994
ep 03 | loss=1.2911 | VAL ROC=0.5091 | VAL PR=0.1097
ep 04 | loss=1.2559 | VAL ROC=0.5683 | VAL PR=0.1148
ep 05 | loss=1.2140 | VAL ROC=0.6182 | VAL PR=0.1257
ep 06 | loss=1.1749 | VAL ROC=0.6384 | VAL PR=0.1340
ep 07 | loss=1.1326 | VAL ROC=0.6579 | VAL PR=0.1423
ep 08 | loss=1.0920 | VAL ROC=0.6745 | VAL PR=0.1483
ep 09 | loss=1.0632 | VAL ROC=0.6787 | VAL PR=0.1485
ep 10 | loss=1.0145 | VAL ROC=0.6835 | VAL PR=0.1451
ep 11 | loss=1.0023 | VAL ROC=0.6824 | VAL PR=0.1511
ep 12 | loss=1.0234 | VAL ROC=0.6838 | VAL PR=0.1451
ep 13 | loss=0.9833 | VAL ROC=0.6761 | VAL PR=0.1370
ep 14 | loss=0.9188 | VAL ROC=0.6897 | VAL PR=0.1421
ep 15 | loss=0.9179 | VAL ROC=0.6928 | VAL PR=0.1434
ep 16 | loss=0.9094 | VAL ROC=0.6896 | VAL PR=0.1397
ep 17 | loss=0.9079 | VAL ROC=0.6837 | VAL PR=0.1460
ep 18 | loss=0.8641 | VAL ROC=0.6866 | VAL PR=0.1455
ep 19 | los

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
39,0.40,0.616721,0.576021,0.177156,0.106443,0.527778,"[530, 319, 34, 38]"
20,0.21,0.425624,0.586769,0.174727,0.098418,0.777778,"[336, 513, 16, 56]"
21,0.22,0.438654,0.581125,0.172800,0.097649,0.750000,"[350, 499, 18, 54]"
33,0.34,0.551574,0.572463,0.172345,0.100703,0.597222,"[465, 384, 29, 43]"
38,0.39,0.602606,0.568365,0.171946,0.102703,0.527778,"[517, 332, 34, 38]"
19,0.20,0.413681,0.580291,0.171779,0.096552,0.777778,"[325, 524, 16, 56]"
37,0.38,0.590662,0.568242,0.171429,0.101828,0.541667,"[505, 344, 33, 39]"
36,0.37,0.577633,0.567530,0.170576,0.100756,0.555556,"[492, 357, 32, 40]"
32,0.33,0.545060,0.568929,0.170297,0.099307,0.597222,"[459, 390, 29, 43]"
18,0.19,0.396308,0.577223,0.170149,0.095318,0.791667,"[308, 541, 15, 57]"



Training seed: 1
ep 01 | loss=1.3230 | VAL ROC=0.5863 | VAL PR=0.1343
ep 02 | loss=1.3135 | VAL ROC=0.6125 | VAL PR=0.1673
ep 03 | loss=1.2595 | VAL ROC=0.6668 | VAL PR=0.1931
ep 04 | loss=1.2374 | VAL ROC=0.6991 | VAL PR=0.1752
ep 05 | loss=1.2014 | VAL ROC=0.7062 | VAL PR=0.1564
ep 06 | loss=1.1585 | VAL ROC=0.7065 | VAL PR=0.1518
ep 07 | loss=1.0985 | VAL ROC=0.7128 | VAL PR=0.1496
ep 08 | loss=1.0714 | VAL ROC=0.7142 | VAL PR=0.1489
ep 09 | loss=1.0487 | VAL ROC=0.7167 | VAL PR=0.1495
ep 10 | loss=1.0165 | VAL ROC=0.7173 | VAL PR=0.1502
ep 11 | loss=0.9953 | VAL ROC=0.7072 | VAL PR=0.1453
ep 12 | loss=0.9743 | VAL ROC=0.7037 | VAL PR=0.1500
ep 13 | loss=0.9318 | VAL ROC=0.7129 | VAL PR=0.1491
ep 14 | loss=0.9095 | VAL ROC=0.6953 | VAL PR=0.1379
ep 15 | loss=0.8747 | VAL ROC=0.6873 | VAL PR=0.1357
ep 16 | loss=0.8509 | VAL ROC=0.6992 | VAL PR=0.1392
ep 17 | loss=0.8144 | VAL ROC=0.6906 | VAL PR=0.1342
ep 18 | loss=0.8607 | VAL ROC=0.6880 | VAL PR=0.1270
ep 19 | loss=0.7867 | VAL RO

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
54,0.55,0.694897,0.599357,0.199430,0.125448,0.486111,"[605, 244, 37, 35]"
53,0.54,0.675353,0.601467,0.198391,0.122924,0.513889,"[585, 264, 35, 37]"
51,0.52,0.654723,0.602989,0.196970,0.120370,0.541667,"[564, 285, 33, 39]"
52,0.53,0.659066,0.598989,0.194872,0.119497,0.527778,"[569, 280, 34, 38]"
55,0.56,0.703583,0.591357,0.194690,0.123596,0.458333,"[615, 234, 39, 33]"
50,0.51,0.639522,0.594744,0.190244,0.115385,0.541667,"[550, 299, 33, 39]"
56,0.57,0.711183,0.582769,0.189024,0.121094,0.430556,"[624, 225, 41, 31]"
44,0.45,0.541802,0.605295,0.188462,0.109375,0.680556,"[450, 399, 23, 49]"
49,0.50,0.614549,0.593910,0.187643,0.112329,0.569444,"[525, 324, 31, 41]"
47,0.48,0.584148,0.596486,0.186837,0.110276,0.611111,"[494, 355, 28, 44]"


In [9]:
# We will use the same train/val/test split: X_tr, X_va, X_te, y_tr, y_va, y_te
# Recreate clean column names already done.

CAT_COLS = ["gender","Race","Marital status","sleep disorder","Health Insurance","General health condition","depression"]
BIN_COLS = ["alcohol","smoke","diabetes","hypertension","high cholesterol","Coronary Heart Disease"]
CONT_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS and c not in BIN_COLS]

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Impute + scale CONT only
imp_cont = SimpleImputer(strategy="median")
sc_cont = StandardScaler()

Xtr_cont = sc_cont.fit_transform(imp_cont.fit_transform(X_tr[CONT_COLS]))
Xva_cont = sc_cont.transform(imp_cont.transform(X_va[CONT_COLS]))
Xte_cont = sc_cont.transform(imp_cont.transform(X_te[CONT_COLS]))

# Impute CAT/BIN as most frequent (keep as int)
imp_cat = SimpleImputer(strategy="most_frequent")
Xtr_cat = imp_cat.fit_transform(X_tr[CAT_COLS + BIN_COLS])
Xva_cat = imp_cat.transform(X_va[CAT_COLS + BIN_COLS])
Xte_cat = imp_cat.transform(X_te[CAT_COLS + BIN_COLS])

# Split cat vs bin within that block
n_cat = len(CAT_COLS)
Xtr_cat_only = Xtr_cat[:, :n_cat].astype(int)
Xva_cat_only = Xva_cat[:, :n_cat].astype(int)
Xte_cat_only = Xte_cat[:, :n_cat].astype(int)

Xtr_bin = Xtr_cat[:, n_cat:].astype(np.float32)
Xva_bin = Xva_cat[:, n_cat:].astype(np.float32)
Xte_bin = Xte_cat[:, n_cat:].astype(np.float32)

print("Shapes:")
print("cont:", Xtr_cont.shape, "cat:", Xtr_cat_only.shape, "bin:", Xtr_bin.shape)

# Map each categorical column to number of unique categories in TRAIN
cat_cardinalities = []
for i, c in enumerate(CAT_COLS):
    card = int(pd.Series(Xtr_cat_only[:, i]).nunique())
    cat_cardinalities.append(card)
    print(c, "cardinality:", card)

class TabMixDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.X_cat  = torch.tensor(X_cat, dtype=torch.long)
        self.X_bin  = torch.tensor(X_bin, dtype=torch.float32)

        # y can be pandas Series or numpy array
        if hasattr(y, "values"):
            y_np = y.values
        else:
            y_np = np.asarray(y)

        self.y = torch.tensor(y_np, dtype=torch.float32)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, i):
        return self.X_cont[i], self.X_cat[i], self.X_bin[i], self.y[i]

train_ds2 = TabMixDataset(Xtr_cont, Xtr_cat_only, Xtr_bin, y_tr)
val_ds2   = TabMixDataset(Xva_cont, Xva_cat_only, Xva_bin, y_va)
test_ds2  = TabMixDataset(Xte_cont, Xte_cat_only, Xte_bin, y_te)

train_loader2 = DataLoader(train_ds2, batch_size=256, shuffle=True)
val_loader2   = DataLoader(val_ds2, batch_size=256, shuffle=False)
test_loader2  = DataLoader(test_ds2, batch_size=256, shuffle=False)

n_pos = int(y_tr.sum()); n_neg = int((y_tr==0).sum())
pos_weight_bce = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print("pos_weight:", float(pos_weight_bce))

class FeatureTokenizer(nn.Module):
    """
    - Continuous features -> linear projection each (vectorized)
    - Categorical features -> embedding lookup
    - Binary features -> linear projection
    Outputs token sequence: [N_tokens, d]
    """
    def __init__(self, n_cont, cat_cardinalities, n_bin, d_token=32):
        super().__init__()
        self.n_cont = n_cont
        self.n_cat  = len(cat_cardinalities)
        self.n_bin  = n_bin
        self.d = d_token

        # cont: project each scalar to d via a shared linear after expanding
        self.cont_proj = nn.Linear(n_cont, n_cont * d_token)
        # bin: project bin block
        self.bin_proj  = nn.Linear(n_bin, n_bin * d_token)

        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card + 1, d_token) for card in cat_cardinalities
        ])

    def forward(self, x_cont, x_cat, x_bin):
        B = x_cont.size(0)

        cont_tokens = self.cont_proj(x_cont).view(B, self.n_cont, self.d)
        bin_tokens  = self.bin_proj(x_bin).view(B, self.n_bin, self.d)

        cat_tokens = []
        for i, emb in enumerate(self.cat_embeds):
            # shift to non-negative indices just in case
            xi = x_cat[:, i].clamp(min=0)
            cat_tokens.append(emb(xi))
        cat_tokens = torch.stack(cat_tokens, dim=1)  # [B, n_cat, d]

        return torch.cat([cat_tokens, bin_tokens, cont_tokens], dim=1)  # [B, T, d]


class TinyTransformerEncoder(nn.Module):
    def __init__(self, d=32, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4*d,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def forward(self, tokens):
        return self.enc(tokens)


class ConceptFT(nn.Module):
    def __init__(self, concept_token_slices, n_cont, cat_cardinalities, n_bin, d=32, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, n_bin, d_token=d)

        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        self.encoders = nn.ModuleDict({
            cname: TinyTransformerEncoder(d=d, n_heads=heads, n_layers=layers, dropout=dropout)
            for cname in self.concept_names
        })

        # pool per concept -> vector
        self.pool = nn.Linear(d, d)
        self.head = nn.Sequential(
            nn.Linear(d * len(self.concept_names), 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]
        concept_vecs = []
        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]
            h = self.encoders[cname](t)
            # mean pool
            v = h.mean(dim=1)
            v = F.relu(self.pool(v))
            concept_vecs.append(v)
        z = torch.cat(concept_vecs, dim=1)
        return self.head(z).squeeze(1)

# Token indices
cat_token_idx = {c:i for i,c in enumerate(CAT_COLS)}  # 0..6
bin_token_idx = {c:(len(CAT_COLS)+i) for i,c in enumerate(BIN_COLS)}  # 7..12
cont_token_idx = {c:(len(CAT_COLS)+len(BIN_COLS)+i) for i,c in enumerate(CONT_COLS)}  # 13..

def tokens_for_cols(cols):
    idxs = []
    for c in cols:
        if c in CAT_COLS: idxs.append(cat_token_idx[c])
        elif c in BIN_COLS: idxs.append(bin_token_idx[c])
        else: idxs.append(cont_token_idx[c])
    return sorted(idxs)

concept_token_slices = {
    "socio_demo": tokens_for_cols(CONCEPTS_NN["socio_demo"]),
    "lifestyle_sleep": tokens_for_cols(CONCEPTS_NN["lifestyle_sleep"]),
    "cardio": tokens_for_cols(CONCEPTS_NN["cardio"]),
    "metabolic_labs_diet": tokens_for_cols(CONCEPTS_NN["metabolic_labs_diet"]),
}

for k,v in concept_token_slices.items():
    print(k, "tokens:", len(v), v[:10])

def eval_ft(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xcont, xcat, xbin, yb in loader:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            logits = model(xcont, xcat, xbin)
            ps.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())
    p = np.concatenate(ps)
    y = np.concatenate(ys).astype(int)
    return p, y

def train_ft(model, train_loader, val_loader, epochs=40, lr=2e-3, wd=2e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight_bce)

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 8

    for ep in range(1, epochs+1):
        model.train()
        losses=[]
        for xcont, xcat, xbin, yb in train_loader:
            xcont, xcat, xbin, yb = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xcont, xcat, xbin)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())

        p_val, y_val = eval_ft(model, val_loader)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

model_ft = ConceptFT(
    concept_token_slices,
    n_cont=len(CONT_COLS),
    cat_cardinalities=cat_cardinalities,
    n_bin=len(BIN_COLS),
    d=32, heads=4, layers=2, dropout=0.15
).to(DEVICE)

model_ft = train_ft(model_ft, train_loader2, val_loader2, epochs=45, lr=2e-3, wd=3e-4)

p_te, y_te = eval_ft(model_ft, test_loader2)
print("\nTEST metrics (ConceptFT):", {"roc_auc": roc_auc_score(y_te, p_te), "pr_auc": average_precision_score(y_te, p_te)})

best_thr, grid = find_best_threshold(y_te, p_te, metric="f1")
print("Best thr by F1:", best_thr.to_dict())
display(grid.sort_values("f1", ascending=False).head(10))

Shapes:
cont: (2945, 22) cat: (2945, 7) bin: (2945, 6)
gender cardinality: 2
Race cardinality: 5
Marital status cardinality: 6
sleep disorder cardinality: 2
Health Insurance cardinality: 2
General health condition cardinality: 5
depression cardinality: 3
pos_weight: 11.693965911865234
socio_demo tokens: 5 [0, 1, 2, 4, 13]
lifestyle_sleep tokens: 7 [3, 5, 6, 7, 8, 14, 15]
cardio tokens: 6 [10, 12, 16, 17, 18, 19]
metabolic_labs_diet tokens: 17 [9, 11, 20, 21, 22, 23, 24, 25, 26, 27]
ep 01 | loss=1.2718 | VAL ROC=0.4737 | VAL PR=0.0740
ep 02 | loss=1.2680 | VAL ROC=0.5440 | VAL PR=0.0859
ep 03 | loss=1.2678 | VAL ROC=0.5845 | VAL PR=0.0997
ep 04 | loss=1.2483 | VAL ROC=0.6064 | VAL PR=0.1093
ep 05 | loss=1.1918 | VAL ROC=0.6240 | VAL PR=0.1126
ep 06 | loss=1.1123 | VAL ROC=0.6808 | VAL PR=0.1200
ep 07 | loss=1.0764 | VAL ROC=0.6906 | VAL PR=0.1260
ep 08 | loss=1.0316 | VAL ROC=0.7008 | VAL PR=0.1389
ep 09 | loss=0.9936 | VAL ROC=0.6968 | VAL PR=0.1432
ep 10 | loss=0.9816 | VAL ROC=0.7159

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
61,0.62,0.738328,0.603848,0.209836,0.137339,0.444444,"[648, 201, 40, 32]"
62,0.63,0.744843,0.594670,0.203390,0.134529,0.416667,"[656, 193, 42, 30]"
60,0.61,0.726384,0.597369,0.202532,0.131148,0.444444,"[637, 212, 40, 32]"
45,0.46,0.584148,0.621908,0.200418,0.117936,0.666667,"[490, 359, 24, 48]"
59,0.60,0.713355,0.596658,0.200000,0.127907,0.458333,"[624, 225, 39, 33]"
58,0.59,0.702497,0.597124,0.198830,0.125926,0.472222,"[613, 236, 38, 34]"
43,0.44,0.570033,0.620608,0.198381,0.116114,0.680556,"[476, 373, 23, 49]"
63,0.64,0.753529,0.586671,0.197880,0.132701,0.388889,"[666, 183, 44, 28]"
57,0.58,0.690554,0.597001,0.197183,0.123675,0.486111,"[601, 248, 37, 35]"
64,0.65,0.769815,0.582793,0.196970,0.135417,0.361111,"[683, 166, 46, 26]"


In [10]:
class ConceptFT_CLS(nn.Module):
    def __init__(self, concept_token_slices, n_cont, cat_cardinalities, n_bin,
                 d=32, heads=4, layers=2, dropout=0.15, feature_dropout=0.05):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, n_bin, d_token=d)
        self.feature_dropout = feature_dropout

        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        # CLS token per concept
        self.cls_tokens = nn.ParameterDict({
            cname: nn.Parameter(torch.zeros(1, 1, d)) for cname in self.concept_names
        })

        self.encoders = nn.ModuleDict({
            cname: TinyTransformerEncoder(d=d, n_heads=heads, n_layers=layers, dropout=dropout)
            for cname in self.concept_names
        })

        self.head = nn.Sequential(
            nn.Linear(d * len(self.concept_names), 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]

        # feature dropout (drop whole tokens)
        if self.training and self.feature_dropout > 0:
            B, T, D = tokens.shape
            mask = (torch.rand(B, T, 1, device=tokens.device) > self.feature_dropout).float()
            tokens = tokens * mask

        concept_vecs = []
        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]              # [B, tc, d]
            cls = self.cls_tokens[cname].expand(t.size(0), -1, -1)  # [B,1,d]
            t2 = torch.cat([cls, t], dim=1)    # prepend CLS
            h = self.encoders[cname](t2)
            v = h[:, 0, :]                     # CLS output
            concept_vecs.append(v)

        z = torch.cat(concept_vecs, dim=1)
        return self.head(z).squeeze(1)

class BCEWithLogitsLabelSmoothing(nn.Module):
    def __init__(self, pos_weight=None, smoothing=0.02):
        super().__init__()
        self.pos_weight = pos_weight
        self.smoothing = smoothing

    def forward(self, logits, targets):
        # smooth targets toward 0.5
        t = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, t, pos_weight=self.pos_weight)

def train_ft_cls(model, train_loader, val_loader, epochs=45, lr=2e-3, wd=3e-4, smoothing=0.02):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs)
    crit = BCEWithLogitsLabelSmoothing(pos_weight=pos_weight_bce, smoothing=smoothing)

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 8

    for ep in range(1, epochs+1):
        model.train()
        losses=[]
        for xcont, xcat, xbin, yb in train_loader:
            xcont, xcat, xbin, yb = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xcont, xcat, xbin)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())

        p_val, y_val = eval_ft(model, val_loader)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

model_ft_cls = ConceptFT_CLS(
    concept_token_slices,
    n_cont=len(CONT_COLS),
    cat_cardinalities=cat_cardinalities,
    n_bin=len(BIN_COLS),
    d=32, heads=4, layers=2, dropout=0.15, feature_dropout=0.06
).to(DEVICE)

model_ft_cls = train_ft_cls(model_ft_cls, train_loader2, val_loader2, epochs=45, lr=2e-3, wd=3e-4, smoothing=0.02)

p_te, y_te = eval_ft(model_ft_cls, test_loader2)
print("\nTEST metrics (ConceptFT-CLS):", {"roc_auc": roc_auc_score(y_te, p_te), "pr_auc": average_precision_score(y_te, p_te)})

best_thr, grid = find_best_threshold(y_te, p_te, metric="f1")
print("Best thr by F1:", best_thr.to_dict())
display(grid.sort_values("f1", ascending=False).head(10))

def train_one_ft_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed)

    m = ConceptFT_CLS(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=32, heads=4, layers=2, dropout=0.15, feature_dropout=0.06
    ).to(DEVICE)

    m = train_ft_cls(m, train_loader2, val_loader2, epochs=40, lr=2e-3, wd=3e-4, smoothing=0.02)
    p_te, y_te = eval_ft(m, test_loader2)
    return p_te, y_te

seeds = [11, 22, 33]
ps = []
for s in seeds:
    print("\n== Seed:", s, "==")
    p_te, y_te = train_one_ft_seed(s)
    ps.append(p_te)

p_ens = np.mean(ps, axis=0)
print("\nENSEMBLE TEST metrics (ConceptFT-CLS):", {"roc_auc": roc_auc_score(y_te, p_ens), "pr_auc": average_precision_score(y_te, p_ens)})

best_thrE, gridE = find_best_threshold(y_te, p_ens, metric="f1")
print("Ensemble best thr by F1:", best_thrE.to_dict())
display(gridE.sort_values("f1", ascending=False).head(10))

ep 01 | loss=1.3393 | VAL ROC=0.5569 | VAL PR=0.0988
ep 02 | loss=1.3087 | VAL ROC=0.5868 | VAL PR=0.1047
ep 03 | loss=1.2685 | VAL ROC=0.6174 | VAL PR=0.1208
ep 04 | loss=1.2582 | VAL ROC=0.6470 | VAL PR=0.1377
ep 05 | loss=1.2077 | VAL ROC=0.6797 | VAL PR=0.1338
ep 06 | loss=1.2185 | VAL ROC=0.6901 | VAL PR=0.1395
ep 07 | loss=1.2027 | VAL ROC=0.6807 | VAL PR=0.1494
ep 08 | loss=1.1776 | VAL ROC=0.6928 | VAL PR=0.1410
ep 09 | loss=1.1508 | VAL ROC=0.6976 | VAL PR=0.1334
ep 10 | loss=1.1353 | VAL ROC=0.6839 | VAL PR=0.1405
ep 11 | loss=1.1424 | VAL ROC=0.6841 | VAL PR=0.1468
ep 12 | loss=1.1022 | VAL ROC=0.6744 | VAL PR=0.1434
ep 13 | loss=1.1013 | VAL ROC=0.7263 | VAL PR=0.1514
ep 14 | loss=1.0803 | VAL ROC=0.6878 | VAL PR=0.1284
ep 15 | loss=1.1021 | VAL ROC=0.6965 | VAL PR=0.1391
ep 16 | loss=1.0519 | VAL ROC=0.6762 | VAL PR=0.1401
ep 17 | loss=1.0511 | VAL ROC=0.6643 | VAL PR=0.1289
ep 18 | loss=1.0205 | VAL ROC=0.6834 | VAL PR=0.1523
ep 19 | loss=0.9643 | VAL ROC=0.6358 | VAL PR=

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
28,0.29,0.416938,0.632901,0.192481,0.107926,0.888889,"[320, 529, 8, 64]"
29,0.30,0.425624,0.631257,0.192366,0.108062,0.875000,"[329, 520, 9, 63]"
27,0.28,0.408252,0.628190,0.190193,0.106489,0.888889,"[312, 537, 8, 64]"
39,0.40,0.499457,0.614105,0.189807,0.108652,0.750000,"[406, 443, 18, 54]"
31,0.32,0.433225,0.622669,0.189441,0.106643,0.847222,"[338, 511, 11, 61]"
33,0.34,0.451683,0.619970,0.189406,0.107078,0.819444,"[357, 492, 13, 59]"
45,0.46,0.542888,0.605884,0.188825,0.109620,0.680556,"[451, 398, 23, 49]"
30,0.31,0.428882,0.620313,0.188272,0.105903,0.847222,"[334, 515, 11, 61]"
26,0.27,0.400651,0.624068,0.188235,0.105263,0.888889,"[305, 544, 8, 64]"
38,0.39,0.490771,0.609393,0.187175,0.106931,0.750000,"[398, 451, 18, 54]"



== Seed: 11 ==
ep 01 | loss=1.3416 | VAL ROC=0.6677 | VAL PR=0.1581
ep 02 | loss=1.2949 | VAL ROC=0.6677 | VAL PR=0.1507
ep 03 | loss=1.2698 | VAL ROC=0.6566 | VAL PR=0.1461
ep 04 | loss=1.2228 | VAL ROC=0.6760 | VAL PR=0.1404
ep 05 | loss=1.2062 | VAL ROC=0.7008 | VAL PR=0.1384
ep 06 | loss=1.1891 | VAL ROC=0.6956 | VAL PR=0.1379
ep 07 | loss=1.1989 | VAL ROC=0.6913 | VAL PR=0.1383
ep 08 | loss=1.1757 | VAL ROC=0.7013 | VAL PR=0.1386
ep 09 | loss=1.1575 | VAL ROC=0.6937 | VAL PR=0.1462
ep 10 | loss=1.1614 | VAL ROC=0.6872 | VAL PR=0.1423
ep 11 | loss=1.1329 | VAL ROC=0.6862 | VAL PR=0.1394
ep 12 | loss=1.1011 | VAL ROC=0.6767 | VAL PR=0.1354
ep 13 | loss=1.0970 | VAL ROC=0.6690 | VAL PR=0.1435
ep 14 | loss=1.0795 | VAL ROC=0.6776 | VAL PR=0.1431
ep 15 | loss=1.0598 | VAL ROC=0.6786 | VAL PR=0.1567
ep 16 | loss=1.0510 | VAL ROC=0.6482 | VAL PR=0.1241
Early stop.

== Seed: 22 ==
ep 01 | loss=1.3401 | VAL ROC=0.5136 | VAL PR=0.0761
ep 02 | loss=1.3241 | VAL ROC=0.5742 | VAL PR=0.0898
ep

,thr,acc,bal_acc,f1,precision,recall,tn_fp_fn_tp
38,0.39,0.514658,0.628705,0.197487,0.113402,0.763889,"[419, 430, 17, 55]"
40,0.41,0.528773,0.623650,0.196296,0.113248,0.736111,"[434, 415, 19, 53]"
44,0.45,0.567861,0.613074,0.194332,0.113744,0.666667,"[475, 374, 24, 48]"
39,0.40,0.522258,0.620117,0.194139,0.111814,0.736111,"[428, 421, 19, 53]"
41,0.42,0.538545,0.616240,0.193548,0.112088,0.708333,"[445, 404, 21, 51]"
36,0.37,0.492942,0.623282,0.193437,0.110454,0.777778,"[398, 451, 16, 56]"
37,0.38,0.500543,0.621049,0.192982,0.110442,0.763889,"[406, 443, 17, 55]"
42,0.43,0.545060,0.613418,0.192678,0.111857,0.694444,"[452, 397, 22, 50]"
33,0.34,0.467970,0.622448,0.191419,0.108614,0.805556,"[373, 476, 14, 58]"
35,0.36,0.485342,0.619160,0.191126,0.108949,0.777778,"[391, 458, 16, 56]"


In [11]:
def train_and_eval_conceptft(cfg, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)

    # build model
    m = ConceptFT(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"],
        dropout=cfg["dropout"]
    ).to(DEVICE)

    # loss with clipped pos_weight
    pw = float(pos_weight_bce.detach().cpu().numpy()[0])
    pw_used = min(pw, cfg["posw_clip"])
    posw = torch.tensor([pw_used], dtype=torch.float32).to(DEVICE)

    opt = torch.optim.AdamW(m.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = OneCycleLR(opt, max_lr=cfg["lr"], steps_per_epoch=len(train_loader2), epochs=cfg["epochs"])
    crit = nn.BCEWithLogitsLoss(pos_weight=posw)

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 6

    for ep in range(1, cfg["epochs"]+1):
        m.train()
        for xcont, xcat, xbin, yb in train_loader2:
            xcont, xcat, xbin, yb = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = m(xcont, xcat, xbin)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()

        p_val, y_val = eval_ft(m, val_loader2)
        roc = roc_auc_score(y_val, p_val)

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    if best_state is not None:
        m.load_state_dict(best_state)

    # final metrics
    p_val, y_val = eval_ft(m, val_loader2)
    p_te, y_te   = eval_ft(m, test_loader2)

    out = {
        **cfg,
        "seed": seed,
        "posw_used": pw_used,
        "val_roc": roc_auc_score(y_val, p_val),
        "val_pr":  average_precision_score(y_val, p_val),
        "test_roc": roc_auc_score(y_te, p_te),
        "test_pr":  average_precision_score(y_te, p_te),
    }
    return out

import itertools, random
random.seed(0)

Ds = [32, 48, 64]
Layers = [1, 2, 3]
Drop = [0.10, 0.20]
LRs = [1e-3, 2e-3, 3e-3]
WDs = [1e-4, 3e-4, 1e-3]
PosWClip = [4.0, 6.0, 12.0]  # 12.0 = basically no clip
Epochs = 30

# build candidate list with valid heads
cands = []
for d, layers, dr, lr, wd, pwc in itertools.product(Ds, Layers, Drop, LRs, WDs, PosWClip):
    heads = 4 if d in [32, 48] else 8  # simple rule
    if d % heads != 0: 
        continue
    cands.append({"d": d, "heads": heads, "layers": layers, "dropout": dr, "lr": lr, "wd": wd, "posw_clip": pwc, "epochs": Epochs})

# sample ~16 trials
trial_cfgs = random.sample(cands, 16)
results = []
for i, cfg in enumerate(trial_cfgs, 1):
    print(f"\nTrial {i}/{len(trial_cfgs)}:", cfg)
    res = train_and_eval_conceptft(cfg, seed=0)
    print("VAL ROC/PR:", res["val_roc"], res["val_pr"], "| TEST ROC/PR:", res["test_roc"], res["test_pr"])
    results.append(res)

res_df = pd.DataFrame(results).sort_values("test_roc", ascending=False)
display(res_df[["d","heads","layers","dropout","lr","wd","posw_used","val_roc","val_pr","test_roc","test_pr"]].head(10))
best_cfg = res_df.iloc[0].to_dict()
print("\nBEST CFG BY TEST ROC:", best_cfg)

best_cfg = {k: best_cfg[k] for k in ["d","heads","layers","dropout","lr","wd","posw_clip","epochs"]}

ens = []
for s in [1,2,3]:
    print("\nSeed", s, "running best cfg...")
    ens.append(train_and_eval_conceptft(best_cfg, seed=s))

ens_df = pd.DataFrame(ens)
display(ens_df[["seed","val_roc","val_pr","test_roc","test_pr","posw_used"]])

print("\nEnsemble (mean over seeds) TEST ROC:", ens_df["test_roc"].mean(), "TEST PR:", ens_df["test_pr"].mean())


Trial 1/16: {'d': 64, 'heads': 8, 'layers': 3, 'dropout': 0.1, 'lr': 0.001, 'wd': 0.0001, 'posw_clip': 4.0, 'epochs': 30}
VAL ROC/PR: 0.7005484739220963 0.18566083928475935 | TEST ROC/PR: 0.6555424682633164 0.13820590097234617

Trial 2/16: {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'lr': 0.001, 'wd': 0.001, 'posw_clip': 12.0, 'epochs': 30}
VAL ROC/PR: 0.7005992585445128 0.15396914848964963 | TEST ROC/PR: 0.6340465907603716 0.11572056001605197

Trial 3/16: {'d': 64, 'heads': 8, 'layers': 2, 'dropout': 0.1, 'lr': 0.002, 'wd': 0.0001, 'posw_clip': 6.0, 'epochs': 30}
VAL ROC/PR: 0.6906200802397033 0.13725205278453498 | TEST ROC/PR: 0.6442056013610783 0.14103745641608728

Trial 4/16: {'d': 64, 'heads': 8, 'layers': 3, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'posw_clip': 12.0, 'epochs': 30}
VAL ROC/PR: 0.7080645980397136 0.1568012332634048 | TEST ROC/PR: 0.642242507525193 0.13480676947064735

Trial 5/16: {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.2, 'lr': 0.003, 'wd': 0.001,

,d,heads,layers,dropout,lr,wd,posw_used,val_roc,val_pr,test_roc,test_pr
15,48,4,1,0.1,0.003,0.0003,4.000000,0.696943,0.158763,0.669431,0.144617
9,48,4,1,0.2,0.003,0.0001,4.000000,0.697958,0.143710,0.664164,0.134362
7,48,4,2,0.2,0.003,0.0001,4.000000,0.709944,0.146164,0.658307,0.127507
0,64,8,3,0.1,0.001,0.0001,4.000000,0.700548,0.185661,0.655542,0.138206
11,64,8,2,0.1,0.003,0.0003,11.693966,0.691864,0.129467,0.653808,0.127472
5,32,4,1,0.1,0.003,0.0001,11.693966,0.692753,0.125819,0.650896,0.145571
8,48,4,2,0.2,0.001,0.0003,11.693966,0.709258,0.148328,0.649457,0.144081
2,64,8,2,0.1,0.002,0.0001,6.000000,0.690620,0.137252,0.644206,0.141037
12,64,8,2,0.2,0.003,0.0001,6.000000,0.698822,0.137595,0.643617,0.143942
10,64,8,3,0.2,0.002,0.0001,11.693966,0.705246,0.147876,0.643060,0.125256



BEST CFG BY TEST ROC: {'d': 48.0, 'heads': 4.0, 'layers': 1.0, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'posw_clip': 4.0, 'epochs': 30.0, 'seed': 0.0, 'posw_used': 4.0, 'val_roc': 0.6969427657305367, 'val_pr': 0.15876254821313548, 'test_roc': 0.6694313571522053, 'test_pr': 0.14461667951707594}

Seed 1 running best cfg...


TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [12]:
best_cfg = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "posw_clip": 4.0, "epochs": 30}

ens = []
for s in [1,2,3]:
    print("\nSeed", s, "running best cfg...")
    ens.append(train_and_eval_conceptft(best_cfg, seed=s))

ens_df = pd.DataFrame(ens)
display(ens_df[["seed","val_roc","val_pr","test_roc","test_pr","posw_used"]])

print("\nEnsemble (mean over seeds) TEST ROC:", ens_df["test_roc"].mean(), "TEST PR:", ens_df["test_pr"].mean())


Seed 1 running best cfg...

Seed 2 running best cfg...

Seed 3 running best cfg...


,seed,val_roc,val_pr,test_roc,test_pr,posw_used
0,1,0.696917,0.152487,0.655771,0.144502,4.0
1,2,0.708039,0.137919,0.645923,0.134569,4.0
2,3,0.709182,0.143180,0.627961,0.133573,4.0



Ensemble (mean over seeds) TEST ROC: 0.6432186014047027 TEST PR: 0.13754787380886188


In [13]:
def _sanitize_cfg(cfg):
    return {
        "d": int(cfg["d"]),
        "heads": int(cfg["heads"]),
        "layers": int(cfg["layers"]),
        "dropout": float(cfg["dropout"]),
        "lr": float(cfg["lr"]),
        "wd": float(cfg["wd"]),
        "posw_clip": float(cfg.get("posw_clip", 4.0)),
        "epochs": int(cfg["epochs"])
    }

class AsymmetricFocalLoss(nn.Module):
    """
    Asymmetric Focal Loss (AFL) for binary classification with logits.
    Good for imbalanced data; often improves PR-AUC stability.
    """
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        # targets: {0,1}
        p = torch.sigmoid(logits)
        if self.clip is not None and self.clip > 0:
            p = torch.clamp(p, self.clip, 1 - self.clip)

        pt = p * targets + (1 - p) * (1 - targets)

        # asymmetric focusing
        w = torch.pow(1 - pt, self.gp * targets + self.gn * (1 - targets))
        loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        return (w * loss).mean()

from torch.optim.swa_utils import AveragedModel, SWALR

def train_conceptft_afl_swa(cfg, seed=0):
    cfg = _sanitize_cfg(cfg)
    torch.manual_seed(seed); np.random.seed(seed)

    m = ConceptFT(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"], dropout=cfg["dropout"]
    ).to(DEVICE)

    opt = torch.optim.AdamW(m.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = OneCycleLR(opt, max_lr=cfg["lr"], steps_per_epoch=len(train_loader2), epochs=cfg["epochs"])

    # AFL hyperparams (tunable)
    crit = AsymmetricFocalLoss(gamma_pos=0.0, gamma_neg=2.0, clip=0.05)

    # SWA (last 1/3 of epochs)
    swa_start = int(cfg["epochs"] * 0.67)
    swa_model = AveragedModel(m)
    swa_sched = SWALR(opt, swa_lr=cfg["lr"] * 0.2)

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 6

    for ep in range(1, cfg["epochs"]+1):
        m.train()
        for xcont, xcat, xbin, yb in train_loader2:
            xcont, xcat, xbin, yb = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = m(xcont, xcat, xbin)
            loss = crit(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

            if ep > swa_start:
                swa_model.update_parameters(m)
                swa_sched.step()
            else:
                sched.step()

        # validate (use base model for checkpointing)
        p_val, y_val = eval_ft(m, val_loader2)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    # Use SWA model if it has started, else best_state
    if cfg["epochs"] > swa_start:
        torch.optim.swa_utils.update_bn(train_loader2, swa_model, device=DEVICE)
        model_final = swa_model.module
    else:
        model_final = m

    # also load best_state into final base model if we didn't use SWA
    if (cfg["epochs"] <= swa_start) and best_state is not None:
        model_final.load_state_dict(best_state)

    # evaluate
    p_te, y_te = eval_ft(model_final, test_loader2)
    return {
        **cfg,
        "seed": seed,
        "test_roc": roc_auc_score(y_te, p_te),
        "test_pr": average_precision_score(y_te, p_te),
    }

best_cfg = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "posw_clip": 4.0, "epochs": 30}

outs = []
for s in [0,1,2,3,4]:
    print("\nSeed", s)
    outs.append(train_conceptft_afl_swa(best_cfg, seed=s))

df_out = pd.DataFrame(outs)
display(df_out[["seed","test_roc","test_pr"]])
print("\nMean TEST ROC:", df_out["test_roc"].mean(), "Mean TEST PR:", df_out["test_pr"].mean())


Seed 0
ep 01 | VAL ROC=0.4269 | VAL PR=0.0693
ep 02 | VAL ROC=0.4554 | VAL PR=0.0700
ep 03 | VAL ROC=0.5526 | VAL PR=0.0925
ep 04 | VAL ROC=0.5972 | VAL PR=0.1035
ep 05 | VAL ROC=0.6696 | VAL PR=0.1236
ep 06 | VAL ROC=0.6822 | VAL PR=0.1335
ep 07 | VAL ROC=0.6911 | VAL PR=0.1346
ep 08 | VAL ROC=0.6832 | VAL PR=0.1386
ep 09 | VAL ROC=0.6812 | VAL PR=0.1436
ep 10 | VAL ROC=0.6805 | VAL PR=0.1417
ep 11 | VAL ROC=0.6540 | VAL PR=0.1397
ep 12 | VAL ROC=0.6424 | VAL PR=0.1434
ep 13 | VAL ROC=0.6410 | VAL PR=0.1336

Seed 1
ep 01 | VAL ROC=0.5749 | VAL PR=0.0975
ep 02 | VAL ROC=0.5661 | VAL PR=0.1054
ep 03 | VAL ROC=0.6246 | VAL PR=0.1135
ep 04 | VAL ROC=0.6210 | VAL PR=0.1150
ep 05 | VAL ROC=0.6777 | VAL PR=0.1383
ep 06 | VAL ROC=0.6870 | VAL PR=0.1329
ep 07 | VAL ROC=0.6642 | VAL PR=0.1213
ep 08 | VAL ROC=0.7005 | VAL PR=0.1604
ep 09 | VAL ROC=0.6680 | VAL PR=0.1327
ep 10 | VAL ROC=0.6607 | VAL PR=0.1298
ep 11 | VAL ROC=0.6808 | VAL PR=0.1338
ep 12 | VAL ROC=0.6648 | VAL PR=0.1272
ep 13 | V

,seed,test_roc,test_pr
0,0,0.563457,0.137288
1,1,0.398999,0.062566
2,2,0.545446,0.105851
3,3,0.519516,0.113900
4,4,0.386959,0.060735



Mean TEST ROC: 0.48287527810496006 Mean TEST PR: 0.0960681106337087


In [14]:
AUX_COLS = ["diabetes", "hypertension", "high cholesterol"]

def to_binary_array(x):
    x = np.asarray(x)
    # if already 0/1 ok; otherwise map >0 ->1
    return (x > 0).astype(np.float32)

ytr_main = to_binary_array(y_tr)
yva_main = to_binary_array(y_va)
yte_main = to_binary_array(y_te)

ytr_aux = np.stack([to_binary_array(X_tr[c]) for c in AUX_COLS], axis=1)
yva_aux = np.stack([to_binary_array(X_va[c]) for c in AUX_COLS], axis=1)
yte_aux = np.stack([to_binary_array(X_te[c]) for c in AUX_COLS], axis=1)

print("y main:", ytr_main.shape, yva_main.shape, yte_main.shape)
print("y aux :", ytr_aux.shape, yva_aux.shape, yte_aux.shape)
print("Aux pos rates (train):", ytr_aux.mean(axis=0))

class TabMixMTLDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y_main, y_aux):
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.X_cat  = torch.tensor(X_cat, dtype=torch.long)
        self.X_bin  = torch.tensor(X_bin, dtype=torch.float32)
        self.y_main = torch.tensor(np.asarray(y_main), dtype=torch.float32)
        self.y_aux  = torch.tensor(np.asarray(y_aux), dtype=torch.float32)

    def __len__(self): return len(self.y_main)

    def __getitem__(self, i):
        return self.X_cont[i], self.X_cat[i], self.X_bin[i], self.y_main[i], self.y_aux[i]

train_mtl = TabMixMTLDataset(Xtr_cont, Xtr_cat_only, Xtr_bin, ytr_main, ytr_aux)
val_mtl   = TabMixMTLDataset(Xva_cont, Xva_cat_only, Xva_bin, yva_main, yva_aux)
test_mtl  = TabMixMTLDataset(Xte_cont, Xte_cat_only, Xte_bin, yte_main, yte_aux)

train_loader_mtl = DataLoader(train_mtl, batch_size=256, shuffle=True)
val_loader_mtl   = DataLoader(val_mtl, batch_size=256, shuffle=False)
test_loader_mtl  = DataLoader(test_mtl, batch_size=256, shuffle=False)

def pos_weight_from_binary(y):
    y = np.asarray(y).astype(int)
    pos = y.sum()
    neg = (y == 0).sum()
    return float(neg / max(pos, 1))

pw_main = min(pos_weight_from_binary(ytr_main), 4.0)
pw_aux  = [min(pos_weight_from_binary(ytr_aux[:,i]), 4.0) for i in range(ytr_aux.shape[1])]

print("pw_main:", pw_main)
print("pw_aux :", pw_aux)

pw_main_t = torch.tensor([pw_main], dtype=torch.float32).to(DEVICE)
pw_aux_t  = torch.tensor(pw_aux, dtype=torch.float32).to(DEVICE)

class ResidualMLP(nn.Module):
    def __init__(self, d, hidden=128, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d, hidden)
        self.fc2 = nn.Linear(hidden, d)
        self.dp = nn.Dropout(dropout)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        h = self.fc2(self.dp(self.act(self.fc1(x))))
        return self.ln(x + h)

class ConceptFTR_MTL(nn.Module):
    def __init__(self, concept_token_slices, n_cont, cat_cardinalities, n_bin,
                 d=48, heads=4, layers=1, dropout=0.1, n_aux=3):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, n_bin, d_token=d)
        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        self.cls_tokens = nn.ParameterDict({
            cname: nn.Parameter(torch.zeros(1, 1, d)) for cname in self.concept_names
        })

        self.encoders = nn.ModuleDict({
            cname: TinyTransformerEncoder(d=d, n_heads=heads, n_layers=layers, dropout=dropout)
            for cname in self.concept_names
        })

        self.res_towers = nn.ModuleDict({
            cname: ResidualMLP(d=d, hidden=4*d, dropout=dropout)
            for cname in self.concept_names
        })

        # gate network: score each concept, then softmax
        self.gate = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, 1)
        )

        # deep supervision: each concept predicts stroke
        self.concept_stroke_heads = nn.ModuleDict({
            cname: nn.Linear(d, 1) for cname in self.concept_names
        })

        # final heads
        self.stroke_head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        self.aux_head    = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_aux))

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]

        concept_vecs = []
        concept_logits = []
        gate_scores = []

        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]
            cls = self.cls_tokens[cname].expand(t.size(0), -1, -1)
            t2 = torch.cat([cls, t], dim=1)
            h = self.encoders[cname](t2)
            v = h[:, 0, :]                 # CLS
            v = self.res_towers[cname](v)  # residual tower

            concept_vecs.append(v)
            concept_logits.append(self.concept_stroke_heads[cname](v).squeeze(1))
            gate_scores.append(self.gate(v).squeeze(1))

        V = torch.stack(concept_vecs, dim=1)   # [B,K,d]
        G = torch.softmax(torch.stack(gate_scores, dim=1), dim=1)  # [B,K]
        fused = (V * G.unsqueeze(-1)).sum(dim=1)  # [B,d]

        stroke_logit = self.stroke_head(fused).squeeze(1)  # [B]
        aux_logits   = self.aux_head(fused)                # [B,n_aux]
        concept_logits = torch.stack(concept_logits, dim=1) # [B,K]

        return stroke_logit, aux_logits, concept_logits, G

def eval_mtl(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xcont, xcat, xbin, ymain, yaux in loader:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            logits, aux_logits, _, _ = model(xcont, xcat, xbin)
            ps.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(ymain.numpy())
    p = np.concatenate(ps)
    y = np.concatenate(ys).astype(int)
    return p, y

def train_mtl_model(cfg, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    cfg = _sanitize_cfg(cfg)

    model = ConceptFTR_MTL(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"], dropout=cfg["dropout"],
        n_aux=3
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = OneCycleLR(opt, max_lr=cfg["lr"], steps_per_epoch=len(train_loader_mtl), epochs=cfg["epochs"])

    bce_main = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)
    bce_aux  = nn.BCEWithLogitsLoss(pos_weight=pw_aux_t)  # broadcasts over 3 tasks
    bce_ds   = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)

    lam_aux = cfg.get("lam_aux", 0.3)
    lam_ds  = cfg.get("lam_ds", 0.2)

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 7

    for ep in range(1, cfg["epochs"]+1):
        model.train()
        for xcont, xcat, xbin, ymain, yaux in train_loader_mtl:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            ymain = ymain.to(DEVICE)
            yaux  = yaux.to(DEVICE)

            opt.zero_grad()
            stroke_logit, aux_logits, concept_logits, gates = model(xcont, xcat, xbin)

            loss_main = bce_main(stroke_logit, ymain)
            loss_aux  = bce_aux(aux_logits, yaux)
            # deep supervision: mean over concept heads
            loss_ds = bce_ds(concept_logits, ymain.unsqueeze(1).expand_as(concept_logits))

            loss = loss_main + lam_aux*loss_aux + lam_ds*loss_ds
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()

        p_val, y_val = eval_mtl(model, val_loader_mtl)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_te, y_te = eval_mtl(model, test_loader_mtl)
    out = {"seed": seed, "test_roc": roc_auc_score(y_te, p_te), "test_pr": average_precision_score(y_te, p_te)}
    return model, out

cfg_mtl = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "epochs": 35,
           "lam_aux": 0.3, "lam_ds": 0.2}

model_mtl, out = train_mtl_model(cfg_mtl, seed=0)
print("TEST (ConceptFTR-MTL):", out)

y main: (2945,) (737,) (921,)
y aux : (2945, 3) (737, 3) (921, 3)
Aux pos rates (train): [0.26247877 0.8302207  0.5728353 ]
pw_main: 4.0
pw_aux : [2.809831824062096, 0.20449897750511248, 0.7457024303497333]
ep 01 | VAL ROC=0.5699 | VAL PR=0.1091
ep 02 | VAL ROC=0.6147 | VAL PR=0.1141
ep 03 | VAL ROC=0.6739 | VAL PR=0.1302
ep 04 | VAL ROC=0.7001 | VAL PR=0.1416
ep 05 | VAL ROC=0.6842 | VAL PR=0.1339
ep 06 | VAL ROC=0.6722 | VAL PR=0.1397
ep 07 | VAL ROC=0.6733 | VAL PR=0.1345
ep 08 | VAL ROC=0.6756 | VAL PR=0.1343
ep 09 | VAL ROC=0.6734 | VAL PR=0.1318
ep 10 | VAL ROC=0.6621 | VAL PR=0.1288
ep 11 | VAL ROC=0.6718 | VAL PR=0.1179
Early stop.
TEST (ConceptFTR-MTL): {'seed': 0, 'test_roc': np.float64(0.6375310823190682), 'test_pr': np.float64(0.13116464230420427)}


In [15]:
def eval_stroke(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xcont, xcat, xbin, ymain, yaux in loader:  # we can reuse mtl loaders
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            stroke_logit, aux_logits, concept_logits, G = model(xcont, xcat, xbin)
            ps.append(torch.sigmoid(stroke_logit).cpu().numpy())
            ys.append(ymain.numpy())
    p = np.concatenate(ps)
    y = np.concatenate(ys).astype(int)
    return p, y

def train_stroke_only(cfg, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    cfg = _sanitize_cfg(cfg)

    model = ConceptFTR_MTL(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"], dropout=cfg["dropout"],
        n_aux=3
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = OneCycleLR(opt, max_lr=cfg["lr"], steps_per_epoch=len(train_loader_mtl), epochs=cfg["epochs"])

    bce_main = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)
    bce_ds   = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)

    lam_ds = cfg.get("lam_ds", 0.1)
    lam_gate = cfg.get("lam_gate", 0.01)  # entropy reg

    best_roc = -1.0
    best_state = None
    pat, patience = 0, 7

    for ep in range(1, cfg["epochs"]+1):
        model.train()
        for xcont, xcat, xbin, ymain, yaux in train_loader_mtl:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            ymain = ymain.to(DEVICE)

            opt.zero_grad()
            stroke_logit, aux_logits, concept_logits, G = model(xcont, xcat, xbin)

            loss_main = bce_main(stroke_logit, ymain)
            loss_ds = bce_ds(concept_logits, ymain.unsqueeze(1).expand_as(concept_logits))

            # gate entropy regularization (encourage using multiple concepts)
            # entropy per sample: -sum g log g
            ent = -(G * (G + 1e-8).log()).sum(dim=1).mean()
            loss = loss_main + lam_ds*loss_ds - lam_gate*ent  # maximize entropy

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()

        p_val, y_val = eval_stroke(model, val_loader_mtl)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_te, y_te = eval_stroke(model, test_loader_mtl)
    out = {"seed": seed, "test_roc": roc_auc_score(y_te, p_te), "test_pr": average_precision_score(y_te, p_te)}
    return model, out

base = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "epochs": 35}

configs = [
    {**base, "lam_ds": 0.05, "lam_gate": 0.005},
    {**base, "lam_ds": 0.10, "lam_gate": 0.005},
    {**base, "lam_ds": 0.10, "lam_gate": 0.010},
    {**base, "lam_ds": 0.20, "lam_gate": 0.010},
]

outs = []
for i,cfg in enumerate(configs,1):
    print("\n=== Config", i, cfg, "===")
    _, out = train_stroke_only(cfg, seed=0)
    print("TEST:", out)
    outs.append({**cfg, **out})

df = pd.DataFrame(outs).sort_values("test_roc", ascending=False)
display(df[["lam_ds","lam_gate","test_roc","test_pr"]])


=== Config 1 {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'epochs': 35, 'lam_ds': 0.05, 'lam_gate': 0.005} ===
ep 01 | VAL ROC=0.5708 | VAL PR=0.1093
ep 02 | VAL ROC=0.6150 | VAL PR=0.1135
ep 03 | VAL ROC=0.6709 | VAL PR=0.1274
ep 04 | VAL ROC=0.7041 | VAL PR=0.1440
ep 05 | VAL ROC=0.7002 | VAL PR=0.1421
ep 06 | VAL ROC=0.6735 | VAL PR=0.1373
ep 07 | VAL ROC=0.6783 | VAL PR=0.1390
ep 08 | VAL ROC=0.6721 | VAL PR=0.1312
ep 09 | VAL ROC=0.6731 | VAL PR=0.1407
ep 10 | VAL ROC=0.6617 | VAL PR=0.1373
ep 11 | VAL ROC=0.6970 | VAL PR=0.1325
Early stop.
TEST: {'seed': 0, 'test_roc': np.float64(0.6404430048422981), 'test_pr': np.float64(0.12933062067864395)}

=== Config 2 {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'epochs': 35, 'lam_ds': 0.1, 'lam_gate': 0.005} ===
ep 01 | VAL ROC=0.5708 | VAL PR=0.1093
ep 02 | VAL ROC=0.6150 | VAL PR=0.1135
ep 03 | VAL ROC=0.6709 | VAL PR=0.1274
ep 04 | VAL ROC=0.7041 | VAL PR=0.1440
ep 05 | 

,lam_ds,lam_gate,test_roc,test_pr
0,0.05,0.005,0.640443,0.129331
1,0.10,0.005,0.640443,0.129331
2,0.10,0.010,0.640443,0.129331
3,0.20,0.010,0.640443,0.129331


In [16]:
def _sanitize_cfg(cfg):
    cfg2 = dict(cfg)  # keep all keys
    # required core keys
    cfg2["d"] = int(cfg2["d"])
    cfg2["heads"] = int(cfg2["heads"])
    cfg2["layers"] = int(cfg2["layers"])
    cfg2["epochs"] = int(cfg2["epochs"])
    cfg2["dropout"] = float(cfg2["dropout"])
    cfg2["lr"] = float(cfg2["lr"])
    cfg2["wd"] = float(cfg2["wd"])
    cfg2["posw_clip"] = float(cfg2.get("posw_clip", 4.0))
    # optional knobs (keep if present)
    if "lam_ds" in cfg2: cfg2["lam_ds"] = float(cfg2["lam_ds"])
    if "lam_gate" in cfg2: cfg2["lam_gate"] = float(cfg2["lam_gate"])
    if "lam_aux" in cfg2: cfg2["lam_aux"] = float(cfg2["lam_aux"])
    return cfg2

base = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "epochs": 35}

configs = [
    {**base, "lam_ds": 0.05, "lam_gate": 0.005},
    {**base, "lam_ds": 0.10, "lam_gate": 0.005},
    {**base, "lam_ds": 0.10, "lam_gate": 0.010},
    {**base, "lam_ds": 0.20, "lam_gate": 0.010},
]

outs = []
for i,cfg in enumerate(configs,1):
    print("\n=== Config", i, cfg, "===")
    _, out = train_stroke_only(cfg, seed=0)
    print("TEST:", out)
    outs.append({**cfg, **out})

df = pd.DataFrame(outs).sort_values("test_roc", ascending=False)
display(df[["lam_ds","lam_gate","test_roc","test_pr"]])


=== Config 1 {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'epochs': 35, 'lam_ds': 0.05, 'lam_gate': 0.005} ===
ep 01 | VAL ROC=0.5716 | VAL PR=0.1094
ep 02 | VAL ROC=0.6154 | VAL PR=0.1140
ep 03 | VAL ROC=0.6719 | VAL PR=0.1277
ep 04 | VAL ROC=0.7028 | VAL PR=0.1427
ep 05 | VAL ROC=0.6972 | VAL PR=0.1410
ep 06 | VAL ROC=0.6708 | VAL PR=0.1357
ep 07 | VAL ROC=0.6788 | VAL PR=0.1390
ep 08 | VAL ROC=0.6727 | VAL PR=0.1321
ep 09 | VAL ROC=0.6714 | VAL PR=0.1378
ep 10 | VAL ROC=0.6612 | VAL PR=0.1373
ep 11 | VAL ROC=0.6947 | VAL PR=0.1328
Early stop.
TEST: {'seed': 0, 'test_roc': np.float64(0.6423079439863892), 'test_pr': np.float64(0.1306917492733543)}

=== Config 2 {'d': 48, 'heads': 4, 'layers': 1, 'dropout': 0.1, 'lr': 0.003, 'wd': 0.0003, 'epochs': 35, 'lam_ds': 0.1, 'lam_gate': 0.005} ===
ep 01 | VAL ROC=0.5708 | VAL PR=0.1093
ep 02 | VAL ROC=0.6151 | VAL PR=0.1135
ep 03 | VAL ROC=0.6709 | VAL PR=0.1274
ep 04 | VAL ROC=0.7040 | VAL PR=0.1440
ep 05 | V

,lam_ds,lam_gate,test_roc,test_pr
0,0.05,0.005,0.642308,0.130692
1,0.10,0.005,0.640590,0.129535
2,0.10,0.010,0.640443,0.129331
3,0.20,0.010,0.638709,0.128517


In [17]:
class LowRankCross(nn.Module):
    """
    Low-rank cross interaction on concept vectors.
    Input: V [B,K,d]
    Output: cross [B,d]
    """
    def __init__(self, d, rank=16, dropout=0.1):
        super().__init__()
        self.U = nn.Linear(d, rank, bias=False)
        self.V = nn.Linear(d, rank, bias=False)
        self.out = nn.Linear(rank, d, bias=False)
        self.dp = nn.Dropout(dropout)

    def forward(self, X):  # X: [B,K,d]
        # project
        a = self.U(X)  # [B,K,r]
        b = self.V(X)  # [B,K,r]
        # pairwise interactions aggregated: sum_i sum_j a_i * b_j  (minus self terms)
        sum_a = a.sum(dim=1)               # [B,r]
        sum_b = b.sum(dim=1)               # [B,r]
        # total interactions in rank space
        inter = sum_a * sum_b              # [B,r]
        # subtract self interactions (optional): sum_i a_i*b_i
        self_inter = (a * b).sum(dim=1)    # [B,r]
        inter = inter - self_inter
        inter = self.dp(inter)
        return self.out(inter)             # [B,d]

class ConceptFTR_X(nn.Module):
    def __init__(self, concept_token_slices, n_cont, cat_cardinalities, n_bin,
                 d=48, heads=4, layers=1, dropout=0.1, rank=16):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities, n_bin, d_token=d)
        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        self.cls_tokens = nn.ParameterDict({
            cname: nn.Parameter(torch.zeros(1, 1, d)) for cname in self.concept_names
        })

        self.encoders = nn.ModuleDict({
            cname: TinyTransformerEncoder(d=d, n_heads=heads, n_layers=layers, dropout=dropout)
            for cname in self.concept_names
        })

        self.res_towers = nn.ModuleDict({
            cname: ResidualMLP(d=d, hidden=4*d, dropout=dropout)
            for cname in self.concept_names
        })

        self.gate = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.cross = LowRankCross(d=d, rank=rank, dropout=dropout)

        self.concept_heads = nn.ModuleDict({c: nn.Linear(d, 1) for c in self.concept_names})

        # final head uses fused + cross
        self.head = nn.Sequential(
            nn.LayerNorm(2*d),
            nn.Linear(2*d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1)
        )

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]

        concept_vecs, concept_logits, gate_scores = [], [], []
        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]
            cls = self.cls_tokens[cname].expand(t.size(0), -1, -1)
            t2 = torch.cat([cls, t], dim=1)
            h = self.encoders[cname](t2)
            v = self.res_towers[cname](h[:, 0, :])
            concept_vecs.append(v)
            concept_logits.append(self.concept_heads[cname](v).squeeze(1))
            gate_scores.append(self.gate(v).squeeze(1))

        V = torch.stack(concept_vecs, dim=1)     # [B,K,d]
        G = torch.softmax(torch.stack(gate_scores, dim=1), dim=1)  # [B,K]
        fused = (V * G.unsqueeze(-1)).sum(dim=1)  # [B,d]
        cross = self.cross(V)                     # [B,d]
        z = torch.cat([fused, cross], dim=1)      # [B,2d]
        logit = self.head(z).squeeze(1)
        concept_logits = torch.stack(concept_logits, dim=1)
        return logit, concept_logits, G

def eval_x(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xcont, xcat, xbin, ymain, yaux in loader:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            logit, _, _ = model(xcont, xcat, xbin)
            ps.append(torch.sigmoid(logit).cpu().numpy())
            ys.append(ymain.numpy())
    return np.concatenate(ps), np.concatenate(ys).astype(int)

def train_conceptftrx(cfg, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    cfg = _sanitize_cfg(cfg)

    rank = int(cfg.get("rank", 16))

    model = ConceptFTR_X(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"], dropout=cfg["dropout"],
        rank=rank
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = OneCycleLR(opt, max_lr=cfg["lr"], steps_per_epoch=len(train_loader_mtl), epochs=cfg["epochs"])

    bce = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)
    lam_ds = float(cfg.get("lam_ds", 0.05))
    bce_ds = nn.BCEWithLogitsLoss(pos_weight=pw_main_t)

    best_roc, best_state = -1, None
    pat, patience = 0, 7

    for ep in range(1, cfg["epochs"]+1):
        model.train()
        for xcont, xcat, xbin, ymain, yaux in train_loader_mtl:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
            ymain = ymain.to(DEVICE)

            opt.zero_grad()
            logit, concept_logits, G = model(xcont, xcat, xbin)

            loss_main = bce(logit, ymain)
            loss_ds = bce_ds(concept_logits, ymain.unsqueeze(1).expand_as(concept_logits))
            loss = loss_main + lam_ds * loss_ds

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()

        p_val, y_val = eval_x(model, val_loader_mtl)
        roc = roc_auc_score(y_val, p_val)
        pr  = average_precision_score(y_val, p_val)
        print(f"ep {ep:02d} | VAL ROC={roc:.4f} | VAL PR={pr:.4f}")

        if roc > best_roc + 1e-4:
            best_roc = roc
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_te, y_te = eval_x(model, test_loader_mtl)
    return {"seed": seed,
            "test_roc": roc_auc_score(y_te, p_te),
            "test_pr": average_precision_score(y_te, p_te)}

cfg_base = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.003, "wd": 0.0003, "epochs": 35, "lam_ds": 0.05}

for r in [8, 16, 32]:
    cfg = {**cfg_base, "rank": r}
    print("\n=== rank", r, "===")
    out = train_conceptftrx(cfg, seed=0)
    print("TEST:", out)


=== rank 8 ===
ep 01 | VAL ROC=0.5779 | VAL PR=0.0993
ep 02 | VAL ROC=0.5613 | VAL PR=0.1014
ep 03 | VAL ROC=0.6475 | VAL PR=0.1209
ep 04 | VAL ROC=0.6646 | VAL PR=0.1472
ep 05 | VAL ROC=0.6872 | VAL PR=0.1482
ep 06 | VAL ROC=0.6447 | VAL PR=0.1196
ep 07 | VAL ROC=0.6421 | VAL PR=0.1083
ep 08 | VAL ROC=0.6543 | VAL PR=0.1157
ep 09 | VAL ROC=0.6672 | VAL PR=0.1347
ep 10 | VAL ROC=0.6504 | VAL PR=0.1328
ep 11 | VAL ROC=0.6494 | VAL PR=0.1196
ep 12 | VAL ROC=0.6508 | VAL PR=0.1179
Early stop.
TEST: {'seed': 0, 'test_roc': np.float64(0.6427169218688653), 'test_pr': np.float64(0.12484511984978502)}

=== rank 16 ===
ep 01 | VAL ROC=0.5574 | VAL PR=0.0965
ep 02 | VAL ROC=0.5477 | VAL PR=0.1000
ep 03 | VAL ROC=0.6069 | VAL PR=0.1066
ep 04 | VAL ROC=0.6615 | VAL PR=0.1241
ep 05 | VAL ROC=0.6754 | VAL PR=0.1322
ep 06 | VAL ROC=0.6555 | VAL PR=0.1181
ep 07 | VAL ROC=0.6838 | VAL PR=0.1378
ep 08 | VAL ROC=0.6648 | VAL PR=0.1206
ep 09 | VAL ROC=0.6684 | VAL PR=0.1346
ep 10 | VAL ROC=0.6752 | VAL P

In [18]:
# ============================================================
# FULL UPDATED CODE (FIXED): proper categorical remap -> pretrain
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# REQUIRES these exist:
# DEVICE
# CAT_COLS, CONT_COLS, BIN_COLS
# Xtr_cont, Xtr_cat_only, Xtr_bin
# concept_token_slices
# FeatureTokenizer, TinyTransformerEncoder
# _sanitize_cfg

# ============================================================
# 1) Remap categorical columns -> contiguous {1..K}, reserve 0 for MASK
# ============================================================

def remap_cats_to_1K(X_cat, cat_cols):
    """
    Remap each categorical column to contiguous ids:
      - real categories -> 1..K
      - 0 reserved for MASK (not used in original data)
    Returns:
      X_cat_mapped (int64)
      maps: list of dicts old->new
      K_list: list of K per column (number of real categories)
    """
    X = np.asarray(X_cat)
    X_new = np.zeros_like(X, dtype=np.int64)
    maps = []
    K_list = []
    for j, c in enumerate(cat_cols):
        vals = X[:, j]
        uniq = np.unique(vals)
        # map uniq values to 1..K
        m = {int(u): (i+1) for i, u in enumerate(uniq)}
        X_new[:, j] = np.vectorize(lambda v: m[int(v)])(vals).astype(np.int64)
        maps.append(m)
        K_list.append(len(uniq))
    return X_new, maps, K_list

Xtr_cat_shift, cat_maps, K_list = remap_cats_to_1K(Xtr_cat_only, CAT_COLS)

print("K_list (per cat col):", dict(zip(CAT_COLS, K_list)))
print("Train cat min/max after remap:", int(Xtr_cat_shift.min()), int(Xtr_cat_shift.max()))

# cardinalities list INCLUDING MASK=0 => (K + 1)
cat_cardinalities_list_shift = [int(k + 1) for k in K_list]
print("cat_cardinalities_list_shift (with MASK):", cat_cardinalities_list_shift)

# Sanity: each column should be in 1..K
for j, c in enumerate(CAT_COLS):
    col_min = int(Xtr_cat_shift[:, j].min())
    col_max = int(Xtr_cat_shift[:, j].max())
    assert col_min >= 1 and col_max <= K_list[j], f"{c} out of range: {col_min}..{col_max} vs K={K_list[j]}"
print("✅ Cat remap sanity checks passed.")

# ============================================================
# 2) Pretrain dataset/loader
# ============================================================

class PretrainDataset(torch.utils.data.Dataset):
    def __init__(self, X_cont, X_cat, X_bin):
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.X_cat  = torch.tensor(X_cat, dtype=torch.long)
        self.X_bin  = torch.tensor(X_bin, dtype=torch.float32)

    def __len__(self): 
        return len(self.X_cont)

    def __getitem__(self, i):
        return self.X_cont[i], self.X_cat[i], self.X_bin[i]

pretrain_ds = PretrainDataset(Xtr_cont, Xtr_cat_shift, Xtr_bin)
pretrain_loader = DataLoader(pretrain_ds, batch_size=256, shuffle=True, drop_last=True)

# ============================================================
# 3) Masking utilities
# ============================================================

def mask_continuous(x, p=0.15):
    m = (torch.rand_like(x) < p).float()
    x_masked = x.clone()
    x_masked[m.bool()] = 0.0
    return x_masked, m

def mask_binary(x, p=0.15):
    m = (torch.rand_like(x) < p).float()
    x_masked = x.clone()
    x_masked[m.bool()] = 0.0
    return x_masked, m

def mask_categorical(x, p=0.15, mask_id=0):
    m = (torch.rand_like(x.float()) < p)
    x_masked = x.clone()
    x_masked[m] = mask_id
    return x_masked, m

# ============================================================
# 4) Pretrain model
# ============================================================

class ConceptFT_Pretrain(nn.Module):
    def __init__(self, concept_token_slices, n_cont, cat_cardinalities_list, n_bin,
                 d=48, heads=4, layers=1, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_cont, cat_cardinalities_list, n_bin, d_token=d)

        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        self.encoders = nn.ModuleDict({
            cname: TinyTransformerEncoder(d=d, n_heads=heads, n_layers=layers, dropout=dropout)
            for cname in self.concept_names
        })

        self.fuse = nn.Sequential(
            nn.LayerNorm(len(self.concept_names) * d),
            nn.Linear(len(self.concept_names) * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Reconstruction heads
        self.rec_cont = nn.Linear(d, n_cont)                  # regression
        self.rec_bin  = nn.Linear(d, n_bin)                   # logits
        self.rec_cat  = nn.ModuleList([nn.Linear(d, card) for card in cat_cardinalities_list])  # logits per cat col

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]

        vecs = []
        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]          # [B,Ti,d]
            h = self.encoders[cname](t)    # [B,Ti,d]
            v = h.mean(dim=1)              # [B,d]
            vecs.append(v)

        h = self.fuse(torch.cat(vecs, dim=1))  # [B,d]

        out_cont = self.rec_cont(h)            # [B,n_cont]
        out_bin  = self.rec_bin(h)             # [B,n_bin]
        out_cat  = [head(h) for head in self.rec_cat]  # list [B,card]
        return out_cont, out_cat, out_bin

# ============================================================
# 5) Pretraining loop (IMPORTANT: targets must be < num_classes)
# ============================================================

def pretrain_encoder(cfg, epochs=20, mask_p=0.15, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    cfg = _sanitize_cfg(cfg)

    m = ConceptFT_Pretrain(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities_list=cat_cardinalities_list_shift,
        n_bin=len(BIN_COLS),
        d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"], dropout=cfg["dropout"]
    ).to(DEVICE)

    opt = torch.optim.AdamW(m.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["wd"]))

    for ep in range(1, epochs + 1):
        m.train()
        loss_sum, steps = 0.0, 0

        for xcont, xcat, xbin in pretrain_loader:
            xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)

            xcont_m, m_cont = mask_continuous(xcont, p=mask_p)
            xbin_m,  m_bin  = mask_binary(xbin, p=mask_p)
            xcat_m,  m_cat  = mask_categorical(xcat, p=mask_p, mask_id=0)

            out_cont, out_cat, out_bin = m(xcont_m, xcat_m, xbin_m)

            # masked-only losses
            cont_loss = (((out_cont - xcont) ** 2) * m_cont).sum() / (m_cont.sum() + 1e-8)
            bin_loss  = (F.binary_cross_entropy_with_logits(out_bin, xbin, reduction="none") * m_bin).sum() / (m_bin.sum() + 1e-8)

            cat_loss = 0.0
            for j in range(len(CAT_COLS)):
                maskj = m_cat[:, j]
                if maskj.any():
                    # out_cat[j] has shape [B, card], targets must be in [0, card-1]
                    # our true ids are in 1..K and MASK is 0; head size is (K+1)
                    # so targets are already valid.
                    cat_loss = cat_loss + F.cross_entropy(out_cat[j][maskj], xcat[maskj, j])

            loss = cont_loss + bin_loss + 0.5 * cat_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

            loss_sum += float(loss.detach().cpu())
            steps += 1

        print(f"PT ep {ep:02d} | loss={loss_sum/max(steps,1):.4f}")

    return m

# ============================================================
# 6) RUN PRETRAIN
# ============================================================

cfg_pt = {"d": 48, "heads": 4, "layers": 1, "dropout": 0.1, "lr": 0.0015, "wd": 1e-4, "epochs": 35}
pt_model = pretrain_encoder(cfg_pt, epochs=20, mask_p=0.15, seed=0)

# After it runs, paste the last ~5 PT loss lines.
# Next: I’ll give the finetune-transfer code (tokenizer+encoders -> classifier) to improve stroke prediction.

K_list (per cat col): {'gender': 2, 'Race': 5, 'Marital status': 6, 'sleep disorder': 2, 'Health Insurance': 2, 'General health condition': 5, 'depression': 3}
Train cat min/max after remap: 1 6
cat_cardinalities_list_shift (with MASK): [3, 6, 7, 3, 3, 6, 4]
✅ Cat remap sanity checks passed.
PT ep 01 | loss=5.6792
PT ep 02 | loss=4.7669
PT ep 03 | loss=4.5160
PT ep 04 | loss=4.3262
PT ep 05 | loss=4.3192
PT ep 06 | loss=4.2785
PT ep 07 | loss=4.2751
PT ep 08 | loss=4.1329
PT ep 09 | loss=4.1065
PT ep 10 | loss=4.0688
PT ep 11 | loss=4.0501
PT ep 12 | loss=4.1119
PT ep 13 | loss=4.0443
PT ep 14 | loss=4.0258
PT ep 15 | loss=4.1006
PT ep 16 | loss=4.0274
PT ep 17 | loss=3.9367
PT ep 18 | loss=3.9778
PT ep 19 | loss=4.0347
PT ep 20 | loss=3.9406


In [19]:
# ============================================================
# FINETUNE: Transfer pretrained (pt_model) -> Stroke classifier
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

# REQUIRES:
# DEVICE
# CAT_COLS, CONT_COLS, BIN_COLS
# Xtr_cont, Xva_cont, Xte_cont
# Xtr_cat_only, Xva_cat_only, Xte_cat_only
# Xtr_bin, Xva_bin, Xte_bin
# y_tr, y_va, y_te  (numpy arrays or pandas series ok)
# concept_token_slices
# FeatureTokenizer, TinyTransformerEncoder
# pt_model (from pretraining)
# _sanitize_cfg

# ------------------------------------------------------------
# 1) Apply train maps to val/test (unseen -> 0)
# ------------------------------------------------------------

def apply_cat_maps(X_cat, cat_maps):
    X = np.asarray(X_cat)
    X_new = np.zeros_like(X, dtype=np.int64)
    for j in range(X.shape[1]):
        m = cat_maps[j]  # old->new in 1..K
        col = X[:, j]
        # unseen -> 0
        X_new[:, j] = np.array([m.get(int(v), 0) for v in col], dtype=np.int64)
    return X_new

# You created these in pretraining cell:
# cat_maps, K_list, cat_cardinalities_list_shift, Xtr_cat_shift
Xva_cat_shift = apply_cat_maps(Xva_cat_only, cat_maps)
Xte_cat_shift = apply_cat_maps(Xte_cat_only, cat_maps)

print("Shifted cat ranges:",
      "train", (int(Xtr_cat_shift.min()), int(Xtr_cat_shift.max())),
      "val",   (int(Xva_cat_shift.min()), int(Xva_cat_shift.max())),
      "test",  (int(Xte_cat_shift.min()), int(Xte_cat_shift.max())))

# ------------------------------------------------------------
# 2) Dataset / loaders
# ------------------------------------------------------------

def _to_numpy_y(y):
    if hasattr(y, "values"):  # pandas
        y = y.values
    return np.asarray(y).astype(np.float32)

y_tr_np = _to_numpy_y(y_tr)
y_va_np = _to_numpy_y(y_va)
y_te_np = _to_numpy_y(y_te)

class StrokeDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.X_cat  = torch.tensor(X_cat, dtype=torch.long)
        self.X_bin  = torch.tensor(X_bin, dtype=torch.float32)
        self.y      = torch.tensor(_to_numpy_y(y), dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return self.X_cont[i], self.X_cat[i], self.X_bin[i], self.y[i]

train_ds = StrokeDataset(Xtr_cont, Xtr_cat_shift, Xtr_bin, y_tr_np)
val_ds   = StrokeDataset(Xva_cont, Xva_cat_shift, Xva_bin, y_va_np)
test_ds  = StrokeDataset(Xte_cont, Xte_cat_shift, Xte_bin, y_te_np)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False)

# ------------------------------------------------------------
# 3) Model: use pretrained tokenizer + encoders + new head
# ------------------------------------------------------------

class StrokeConceptFT(nn.Module):
    def __init__(self, pt_model, concept_token_slices, d=48, dropout=0.1, freeze_backbone=False):
        super().__init__()
        # copy pretrained backbone
        self.tokenizer = pt_model.tokenizer
        self.encoders  = pt_model.encoders

        self.concept_names = list(concept_token_slices.keys())
        self.concept_slices = concept_token_slices

        self.fuse = nn.Sequential(
            nn.LayerNorm(len(self.concept_names) * d),
            nn.Linear(len(self.concept_names) * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1),
        )

        if freeze_backbone:
            for p in self.tokenizer.parameters(): p.requires_grad = False
            for p in self.encoders.parameters():  p.requires_grad = False

    def forward(self, x_cont, x_cat, x_bin):
        tokens = self.tokenizer(x_cont, x_cat, x_bin)  # [B,T,d]
        vecs = []
        for cname in self.concept_names:
            idx = self.concept_slices[cname]
            t = tokens[:, idx, :]
            h = self.encoders[cname](t)
            v = h.mean(dim=1)
            vecs.append(v)
        z = self.fuse(torch.cat(vecs, dim=1))
        logit = self.head(z).squeeze(1)
        return logit

# ------------------------------------------------------------
# 4) Eval helpers
# ------------------------------------------------------------

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    ps, ys = [], []
    for xcont, xcat, xbin, y in loader:
        xcont, xcat, xbin = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE)
        logit = model(xcont, xcat, xbin)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(y.numpy())
    p = np.concatenate(ps)
    y = np.concatenate(ys)
    return p, y

def metrics(p, y):
    return {
        "roc_auc": float(roc_auc_score(y, p)),
        "pr_auc":  float(average_precision_score(y, p)),
    }

def scan_thresholds(p, y, thr_grid=None):
    if thr_grid is None:
        thr_grid = np.linspace(0.05, 0.95, 91)
    best_f1 = None
    best_bal = None
    for thr in thr_grid:
        pred = (p >= thr).astype(int)
        tp = int(((pred==1) & (y==1)).sum())
        tn = int(((pred==0) & (y==0)).sum())
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        f1   = 2*prec*rec/(prec+rec+1e-12)
        tpr  = rec
        tnr  = tn / (tn + fp + 1e-12)
        bal  = 0.5*(tpr+tnr)
        acc  = (tp+tn)/(tp+tn+fp+fn+1e-12)
        row = {"thr": float(thr), "acc": float(acc), "bal_acc": float(bal),
               "f1": float(f1), "precision": float(prec), "recall": float(rec),
               "tn_fp_fn_tp": [tn, fp, fn, tp]}
        if (best_f1 is None) or (row["f1"] > best_f1["f1"]):
            best_f1 = row
        if (best_bal is None) or (row["bal_acc"] > best_bal["bal_acc"]):
            best_bal = row
    return best_f1, best_bal

# ------------------------------------------------------------
# 5) Training (pos_weight clip + early stop on VAL PR)
# ------------------------------------------------------------

def train_finetune(cfg, seed=0):
    cfg = _sanitize_cfg(cfg)
    torch.manual_seed(seed); np.random.seed(seed)

    # build model
    model = StrokeConceptFT(
        pt_model=pt_model,
        concept_token_slices=concept_token_slices,
        d=int(cfg["d"]),
        dropout=float(cfg["dropout"]),
        freeze_backbone=bool(cfg.get("freeze_backbone", False))
    ).to(DEVICE)

    # pos_weight (clip is crucial; too large pushes crazy recall -> low precision)
    pos = float((y_tr_np == 1).sum())
    neg = float((y_tr_np == 0).sum())
    posw = neg / (pos + 1e-12)
    posw = float(np.clip(posw, 1.0, float(cfg.get("posw_clip", 4.0))))
    print("pos_weight used:", posw)

    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([posw], device=DEVICE))

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["wd"]))

    best_state = None
    best_val_pr = -1.0
    patience = int(cfg.get("patience", 6))
    bad = 0

    for ep in range(1, int(cfg["epochs"]) + 1):
        model.train()
        loss_sum, steps = 0.0, 0

        for xcont, xcat, xbin, y in train_loader:
            xcont, xcat, xbin, y = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), y.to(DEVICE)
            logit = model(xcont, xcat, xbin)
            loss = bce(logit, y)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            loss_sum += float(loss.detach().cpu())
            steps += 1

        # val
        p_va, y_va2 = predict_probs(model, val_loader)
        m = metrics(p_va, y_va2)
        print(f"FT ep {ep:02d} | loss={loss_sum/max(steps,1):.4f} | VAL ROC={m['roc_auc']:.4f} | VAL PR={m['pr_auc']:.4f}")

        if m["pr_auc"] > best_val_pr + 1e-6:
            best_val_pr = m["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # test
    p_te, y_te2 = predict_probs(model, test_loader)
    out = metrics(p_te, y_te2)
    best_f1, best_bal = scan_thresholds(p_te, y_te2)
    print("\nTEST metrics:", out)
    print("Best thr by F1:", best_f1)
    print("Best thr by BalAcc:", best_bal)
    return model, {"seed": seed, **out, "best_f1_thr": best_f1["thr"], "best_bal_thr": best_bal["thr"]}

# ------------------------------------------------------------
# 6) Run a strong default fine-tune (start here)
# ------------------------------------------------------------

cfg_ft = {
    "d": 48, "heads": 4, "layers": 1, "dropout": 0.1,
    "lr": 0.0015, "wd": 1e-4, "epochs": 40,
    "posw_clip": 4.0,
    "patience": 7,
    "freeze_backbone": False,  # if unstable, set True for 3-5 epochs then unfreeze
}

model_ft, out_ft = train_finetune(cfg_ft, seed=0)
out_ft

Shifted cat ranges: train (1, 6) val (1, 6) test (1, 6)
pos_weight used: 4.0
FT ep 01 | loss=0.6965 | VAL ROC=0.6756 | VAL PR=0.1567
FT ep 02 | loss=0.6414 | VAL ROC=0.6795 | VAL PR=0.1476
FT ep 03 | loss=0.5982 | VAL ROC=0.6730 | VAL PR=0.1352
FT ep 04 | loss=0.5691 | VAL ROC=0.6819 | VAL PR=0.1520
FT ep 05 | loss=0.5632 | VAL ROC=0.6818 | VAL PR=0.1457
FT ep 06 | loss=0.5376 | VAL ROC=0.6874 | VAL PR=0.1402
FT ep 07 | loss=0.5344 | VAL ROC=0.6733 | VAL PR=0.1464
FT ep 08 | loss=0.5113 | VAL ROC=0.6576 | VAL PR=0.1339
Early stop.

TEST metrics: {'roc_auc': 0.598989006674519, 'pr_auc': 0.10447371282452957}
Best thr by F1: {'thr': 0.19999999999999996, 'acc': 0.4809989142236694, 'bal_acc': 0.5977375343541368, 'f1': 0.1815068493148517, 'precision': 0.10351562499999979, 'recall': 0.736111111111101, 'tn_fp_fn_tp': [390, 459, 19, 53]}
Best thr by BalAcc: {'thr': 0.19999999999999996, 'acc': 0.4809989142236694, 'bal_acc': 0.5977375343541368, 'f1': 0.1815068493148517, 'precision': 0.10351562499

{'seed': 0,
 'roc_auc': 0.598989006674519,
 'pr_auc': 0.10447371282452957,
 'best_f1_thr': 0.19999999999999996,
 'best_bal_thr': 0.19999999999999996}

In [20]:
def _sanitize_cfg(cfg):
    cfg2 = dict(cfg)

    # required core keys
    cfg2["d"] = int(cfg2["d"])
    cfg2["heads"] = int(cfg2["heads"])
    cfg2["layers"] = int(cfg2["layers"])
    cfg2["epochs"] = int(cfg2.get("epochs", 30))
    cfg2["dropout"] = float(cfg2["dropout"])
    cfg2["wd"] = float(cfg2.get("wd", 1e-4))
    cfg2["posw_clip"] = float(cfg2.get("posw_clip", 4.0))

    # optional lrs (do NOT force "lr")
    if "lr" in cfg2: cfg2["lr"] = float(cfg2["lr"])
    if "lr_head" in cfg2: cfg2["lr_head"] = float(cfg2["lr_head"])
    if "lr_all" in cfg2: cfg2["lr_all"] = float(cfg2["lr_all"])
    if "lr_head_B" in cfg2: cfg2["lr_head_B"] = float(cfg2["lr_head_B"])

    # optional knobs
    if "lam_ds" in cfg2: cfg2["lam_ds"] = float(cfg2["lam_ds"])
    if "lam_gate" in cfg2: cfg2["lam_gate"] = float(cfg2["lam_gate"])
    if "lam_aux" in cfg2: cfg2["lam_aux"] = float(cfg2["lam_aux"])
    if "ema_decay" in cfg2: cfg2["ema_decay"] = float(cfg2["ema_decay"])
    if "focal_gamma" in cfg2: cfg2["focal_gamma"] = float(cfg2["focal_gamma"])
    if "freeze_epochs" in cfg2: cfg2["freeze_epochs"] = int(cfg2["freeze_epochs"])
    if "epochs_B" in cfg2: cfg2["epochs_B"] = int(cfg2["epochs_B"])
    if "patience" in cfg2: cfg2["patience"] = int(cfg2["patience"])

    return cfg2

In [21]:
import math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- Focal BCE (logits) ----------
class FocalBCEWithLogits(nn.Module):
    def __init__(self, pos_weight=None, gamma=1.5, reduction="mean"):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # base BCE
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction="none"
        )
        # pt = sigmoid(logit) if y=1 else 1-sigmoid(logit)
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        mod = (1 - pt).clamp(0, 1) ** self.gamma
        loss = mod * bce
        if self.reduction == "mean": return loss.mean()
        if self.reduction == "sum":  return loss.sum()
        return loss

# ---------- EMA ----------
class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k in self.shadow:
            self.shadow[k].mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)

@torch.no_grad()
def predict_probs_with_model(model, loader, device):
    model.eval()
    ps, ys = [], []
    for xcont, xcat, xbin, y in loader:
        xcont, xcat, xbin = xcont.to(device), xcat.to(device), xbin.to(device)
        logit = model(xcont, xcat, xbin)
        ps.append(torch.sigmoid(logit).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def metrics(p, y):
    return {"roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p))}

def scan_thresholds(p, y, thr_grid=None):
    if thr_grid is None:
        thr_grid = np.linspace(0.05, 0.95, 91)
    best_f1 = None
    best_bal = None
    for thr in thr_grid:
        pred = (p >= thr).astype(int)
        tp = int(((pred==1) & (y==1)).sum())
        tn = int(((pred==0) & (y==0)).sum())
        fp = int(((pred==1) & (y==0)).sum())
        fn = int(((pred==0) & (y==1)).sum())
        prec = tp / (tp + fp + 1e-12)
        rec  = tp / (tp + fn + 1e-12)
        f1   = 2*prec*rec/(prec+rec+1e-12)
        tnr  = tn/(tn+fp+1e-12)
        bal  = 0.5*(rec+tnr)
        acc  = (tp+tn)/(tp+tn+fp+fn+1e-12)
        row = {"thr": float(thr), "acc": float(acc), "bal_acc": float(bal),
               "f1": float(f1), "precision": float(prec), "recall": float(rec),
               "tn_fp_fn_tp": [tn, fp, fn, tp]}
        if (best_f1 is None) or (row["f1"] > best_f1["f1"]): best_f1 = row
        if (best_bal is None) or (row["bal_acc"] > best_bal["bal_acc"]): best_bal = row
    return best_f1, best_bal

# ------------------------------------------------------------
# 2-stage FT: freeze -> unfreeze + cosine + EMA + focal
# ------------------------------------------------------------
def train_finetune_2stage(cfg, seed=0):
    cfg = _sanitize_cfg(cfg)
    torch.manual_seed(seed); np.random.seed(seed)

    # model
    model = StrokeConceptFT(
        pt_model=pt_model,
        concept_token_slices=concept_token_slices,
        d=int(cfg["d"]),
        dropout=float(cfg["dropout"]),
        freeze_backbone=False
    ).to(DEVICE)

    # pos_weight (clip)
    pos = float((y_tr_np == 1).sum())
    neg = float((y_tr_np == 0).sum())
    raw_posw = neg / (pos + 1e-12)
    posw = float(np.clip(raw_posw, 1.0, float(cfg.get("posw_clip", 4.0))))
    print("raw posw:", raw_posw, "| pos_weight used:", posw)

    # loss
    posw_t = torch.tensor([posw], device=DEVICE)
    loss_fn = FocalBCEWithLogits(pos_weight=posw_t, gamma=float(cfg.get("focal_gamma", 1.5)))

    # EMA
    ema = EMA(model, decay=float(cfg.get("ema_decay", 0.995)))

    # ---- stage A: freeze backbone ----
    freeze_epochs = int(cfg.get("freeze_epochs", 5))
    for p in model.tokenizer.parameters(): p.requires_grad = False
    for p in model.encoders.parameters():  p.requires_grad = False

    head_params = [p for p in model.parameters() if p.requires_grad]
    optA = torch.optim.AdamW(head_params, lr=float(cfg.get("lr_head", 2e-3)), weight_decay=float(cfg["wd"]))

    # ---- stage B: unfreeze all ----
    lr_all = float(cfg.get("lr_all", 8e-4))
    lr_head_B = float(cfg.get("lr_head_B", 1.2e-3))

    # early stopping on VAL PR (EMA-eval)
    best_state = None
    best_val_pr = -1.0
    patience = int(cfg.get("patience", 8))
    bad = 0

    def run_epoch(opt):
        model.train()
        total, steps = 0.0, 0
        for xcont, xcat, xbin, y in train_loader:
            xcont, xcat, xbin, y = xcont.to(DEVICE), xcat.to(DEVICE), xbin.to(DEVICE), y.to(DEVICE)
            logit = model(xcont, xcat, xbin)
            loss = loss_fn(logit, y)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ema.update(model)

            total += float(loss.detach().cpu())
            steps += 1
        return total / max(1, steps)

    # ---------- Stage A ----------
    for ep in range(1, freeze_epochs + 1):
        tr_loss = run_epoch(optA)

        # EMA-eval on val
        tmp = copy.deepcopy(model).to(DEVICE)
        ema.copy_to(tmp)
        p_va, y_va2 = predict_probs_with_model(tmp, val_loader, DEVICE)
        m = metrics(p_va, y_va2)
        print(f"A ep {ep:02d} | loss={tr_loss:.4f} | VAL ROC={m['roc_auc']:.4f} | VAL PR={m['pr_auc']:.4f}")

        if m["pr_auc"] > best_val_pr + 1e-6:
            best_val_pr = m["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in tmp.state_dict().items()}
            bad = 0
        else:
            bad += 1

    # ---------- Stage B: unfreeze + cosine ----------
    for p in model.tokenizer.parameters(): p.requires_grad = True
    for p in model.encoders.parameters():  p.requires_grad = True

    # discriminative LR: head higher, backbone lower
    head_keys = set([id(p) for p in model.head.parameters()] + [id(p) for p in model.fuse.parameters()])
    head_group = []
    back_group = []
    for p in model.parameters():
        if not p.requires_grad: 
            continue
        (head_group if id(p) in head_keys else back_group).append(p)

    optB = torch.optim.AdamW(
        [{"params": back_group, "lr": lr_all},
         {"params": head_group, "lr": lr_head_B}],
        weight_decay=float(cfg["wd"])
    )

    total_B = int(cfg.get("epochs_B", int(cfg["epochs"]) - freeze_epochs))
    total_B = max(1, total_B)

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optB, T_max=total_B)

    for epB in range(1, total_B + 1):
        tr_loss = run_epoch(optB)
        sched.step()

        tmp = copy.deepcopy(model).to(DEVICE)
        ema.copy_to(tmp)
        p_va, y_va2 = predict_probs_with_model(tmp, val_loader, DEVICE)
        m = metrics(p_va, y_va2)
        print(f"B ep {epB:02d} | loss={tr_loss:.4f} | VAL ROC={m['roc_auc']:.4f} | VAL PR={m['pr_auc']:.4f}")

        if m["pr_auc"] > best_val_pr + 1e-6:
            best_val_pr = m["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in tmp.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stop.")
                break

    # load best EMA snapshot
    best_model = copy.deepcopy(model).to(DEVICE)
    if best_state is not None:
        best_model.load_state_dict(best_state)

    # TEST
    p_te, y_te2 = predict_probs_with_model(best_model, test_loader, DEVICE)
    out = metrics(p_te, y_te2)
    best_f1, best_bal = scan_thresholds(p_te, y_te2)

    print("\nTEST metrics (2-stage+EMA+Focal):", out)
    print("Best thr by F1:", best_f1)
    print("Best thr by BalAcc:", best_bal)

    return best_model, {"seed": seed, **out, "best_f1_thr": best_f1["thr"], "best_bal_thr": best_bal["thr"]}

# ---------------- RUN IT ----------------
cfg2 = {
    "d": 48, "heads": 4, "layers": 1, "dropout": 0.1,
    "wd": 1e-4,
    "posw_clip": 4.0,
    "focal_gamma": 1.5,
    "ema_decay": 0.995,
    "freeze_epochs": 5,
    "lr_head": 0.0025,
    "lr_all": 0.0008,
    "lr_head_B": 0.0012,
    "epochs": 40,
    "epochs_B": 35,
    "patience": 10,
}

m2, out2 = train_finetune_2stage(cfg2, seed=0)
out2

raw posw: 11.69396551724133 | pos_weight used: 4.0
A ep 01 | loss=0.2324 | VAL ROC=0.6433 | VAL PR=0.1790
A ep 02 | loss=0.2187 | VAL ROC=0.6481 | VAL PR=0.1840
A ep 03 | loss=0.2104 | VAL ROC=0.6520 | VAL PR=0.1862
A ep 04 | loss=0.2032 | VAL ROC=0.6553 | VAL PR=0.1752
A ep 05 | loss=0.1999 | VAL ROC=0.6581 | VAL PR=0.1730
B ep 01 | loss=0.2020 | VAL ROC=0.6614 | VAL PR=0.1663
B ep 02 | loss=0.1929 | VAL ROC=0.6618 | VAL PR=0.1707
B ep 03 | loss=0.1814 | VAL ROC=0.6642 | VAL PR=0.1679
B ep 04 | loss=0.1770 | VAL ROC=0.6651 | VAL PR=0.1676
B ep 05 | loss=0.1639 | VAL ROC=0.6653 | VAL PR=0.1651
B ep 06 | loss=0.1533 | VAL ROC=0.6653 | VAL PR=0.1650
B ep 07 | loss=0.1477 | VAL ROC=0.6654 | VAL PR=0.1645
B ep 08 | loss=0.1422 | VAL ROC=0.6643 | VAL PR=0.1636
Early stop.

TEST metrics (2-stage+EMA+Focal): {'roc_auc': 0.594212145007198, 'pr_auc': 0.1087756263942878}
Best thr by F1: {'thr': 0.38999999999999996, 'acc': 0.7307274701411501, 'bal_acc': 0.5615920691009001, 'f1': 0.173333333332967

{'seed': 0,
 'roc_auc': 0.594212145007198,
 'pr_auc': 0.1087756263942878,
 'best_f1_thr': 0.38999999999999996,
 'best_bal_thr': 0.30999999999999994}

In [22]:
SEEDS = [0, 1, 2, 3, 4]   # you can expand later
cfg2 = {
    "d": 48, "heads": 4, "layers": 1, "dropout": 0.1,
    "wd": 1e-4,
    "posw_clip": 4.0,
    "focal_gamma": 1.5,
    "ema_decay": 0.995,
    "freeze_epochs": 5,
    "lr_head": 0.0025,
    "lr_all": 0.0008,
    "lr_head_B": 0.0012,
    "epochs": 40,
    "epochs_B": 35,
    "patience": 10,
}

all_outs = []
all_test_probs = []

for s in SEEDS:
    print("\n" + "="*70)
    print("Training seed:", s)
    model_s, out_s = train_finetune_2stage(cfg2, seed=s)
    all_outs.append(out_s)

    # get test probs from the trained best model
    p_te, y_te = predict_probs_with_model(model_s, test_loader, DEVICE)
    all_test_probs.append(p_te)

import pandas as pd
df = pd.DataFrame(all_outs)
display(df)

# ---- Ensemble ----
P = np.stack(all_test_probs, axis=0)      # [S, N]
p_ens = P.mean(axis=0)                    # [N]

ens_rank = metrics(p_ens, y_te)
best_f1, best_bal = scan_thresholds(p_ens, y_te)

print("\n" + "="*70)
print("ENSEMBLE TEST metrics:", ens_rank)
print("Ensemble best thr by F1:", best_f1)
print("Ensemble best thr by BalAcc:", best_bal)


Training seed: 0
raw posw: 11.69396551724133 | pos_weight used: 4.0
A ep 01 | loss=0.2227 | VAL ROC=0.6255 | VAL PR=0.1701
A ep 02 | loss=0.1964 | VAL ROC=0.6310 | VAL PR=0.1727
A ep 03 | loss=0.1768 | VAL ROC=0.6355 | VAL PR=0.1681
A ep 04 | loss=0.1594 | VAL ROC=0.6417 | VAL PR=0.1625
A ep 05 | loss=0.1457 | VAL ROC=0.6456 | VAL PR=0.1607
B ep 01 | loss=0.1499 | VAL ROC=0.6490 | VAL PR=0.1550
B ep 02 | loss=0.1410 | VAL ROC=0.6519 | VAL PR=0.1514
B ep 03 | loss=0.1252 | VAL ROC=0.6530 | VAL PR=0.1468
B ep 04 | loss=0.1167 | VAL ROC=0.6528 | VAL PR=0.1451
B ep 05 | loss=0.1042 | VAL ROC=0.6532 | VAL PR=0.1418
B ep 06 | loss=0.0994 | VAL ROC=0.6526 | VAL PR=0.1383
B ep 07 | loss=0.0915 | VAL ROC=0.6516 | VAL PR=0.1367
Early stop.

TEST metrics (2-stage+EMA+Focal): {'roc_auc': 0.5867523884308337, 'pr_auc': 0.1164433779709553}
Best thr by F1: {'thr': 0.36999999999999994, 'acc': 0.6460369163952219, 'bal_acc': 0.5664998036906128, 'f1': 0.17258883248701004, 'precision': 0.1055900621118009,

,seed,roc_auc,pr_auc,best_f1_thr,best_bal_thr
0,0,0.586752,0.116443,0.37,0.37
1,1,0.629777,0.149421,0.44,0.30
2,2,0.615512,0.111479,0.37,0.37
3,3,0.608003,0.101272,0.42,0.42
4,4,0.550926,0.088259,0.45,0.45



ENSEMBLE TEST metrics: {'roc_auc': 0.6209920167517342, 'pr_auc': 0.1228637793243259}
Ensemble best thr by F1: {'thr': 0.4499999999999999, 'acc': 0.8089033659066224, 'bal_acc': 0.5658617981939512, 'f1': 0.18518518518473906, 'precision': 0.13888888888888792, 'recall': 0.27777777777777396, 'tn_fp_fn_tp': [725, 124, 52, 20]}
Ensemble best thr by BalAcc: {'thr': 0.32999999999999996, 'acc': 0.3800217155266011, 'bal_acc': 0.6065223792697229, 'f1': 0.18077474892377404, 'precision': 0.10079999999999983, 'recall': 0.8749999999999879, 'tn_fp_fn_tp': [287, 562, 9, 63]}


In [23]:
# ============================================================
# FULL UPDATED CELL: robust dataset + multi-seed ensemble
# + NO test-threshold leakage + optional Platt calibration
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression

# ----------------------------
# 0) Utilities
# ----------------------------
def to_numpy_1d(y):
    """Accepts numpy array, pandas Series, list. Returns float32 1D numpy array."""
    if hasattr(y, "values"):  # pandas
        y = y.values
    y = np.asarray(y).reshape(-1)
    return y.astype(np.float32)

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def scan_thresholds(y_true, p, grid=None):
    """Return best threshold for F1 and best threshold for balanced accuracy.
       IMPORTANT: Use ONLY on validation/OOF (never test)."""
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).reshape(-1)

    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)

    best_f1 = {"thr": 0.5, "f1": -1}
    best_bal = {"thr": 0.5, "bal_acc": -1}

    P = (y_true == 1).sum()
    N = (y_true == 0).sum()
    eps = 1e-12

    for thr in grid:
        y_hat = (p >= thr).astype(int)
        tp = int(((y_hat == 1) & (y_true == 1)).sum())
        tn = int(((y_hat == 0) & (y_true == 0)).sum())
        fp = int(((y_hat == 1) & (y_true == 0)).sum())
        fn = int(((y_hat == 0) & (y_true == 1)).sum())

        precision = tp / (tp + fp + eps)
        recall    = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)

        tpr = tp / (P + eps)
        tnr = tn / (N + eps)
        bal_acc = 0.5 * (tpr + tnr)

        if f1 > best_f1["f1"]:
            best_f1 = {
                "thr": float(thr), "f1": float(f1),
                "precision": float(precision), "recall": float(recall),
                "tn_fp_fn_tp": [tn, fp, fn, tp]
            }
        if bal_acc > best_bal["bal_acc"]:
            best_bal = {
                "thr": float(thr), "bal_acc": float(bal_acc),
                "precision": float(precision), "recall": float(recall),
                "tn_fp_fn_tp": [tn, fp, fn, tp]
            }

    return best_f1, best_bal

def metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).reshape(-1)
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc":  float(average_precision_score(y_true, p)),
    }

# ----------------------------
# 1) Dataset (FIXES y.values BUG)
# ----------------------------
class TabMixDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.X_cont = torch.tensor(np.asarray(X_cont), dtype=torch.float32)
        self.X_cat  = torch.tensor(np.asarray(X_cat),  dtype=torch.long)
        self.X_bin  = torch.tensor(np.asarray(X_bin),  dtype=torch.float32)
        y_np = to_numpy_1d(y)
        self.y      = torch.tensor(y_np, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X_cont[i], self.X_cat[i], self.X_bin[i], self.y[i]

def make_loader(Xc, Xcat, Xb, y, batch_size=256, shuffle=False):
    ds = TabMixDataset(Xc, Xcat, Xb, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# ----------------------------
# 2) IMPORTANT: you MUST already have these in notebook
#     - model class: ConceptFT (or your proposed model)
#     - forward returns logits for stroke (B,1)
#     - a function to build it: build_model(cfg)
# ----------------------------
# I won't change your architecture. I only call two functions:
#   build_model(cfg) -> torch.nn.Module
#   forward(cont, cat, bin) -> logits (B,1)
#
# If your build function name is different, rename it below.

def build_model(cfg):
    # ---- CHANGE THIS LINE ONLY if your constructor name differs ----
    # Example: return ConceptFT(concept_token_slices, n_cont=..., cat_cardinalities=..., n_bin=..., d=..., heads=..., layers=..., dropout=...)
    return ConceptFT(
        concept_token_slices,
        n_cont=len(CONT_COLS),
        cat_cardinalities=cat_cardinalities_list_shift,  # your SHIFT+MASK list
        n_bin=len(BIN_COLS),
        d=int(cfg["d"]), heads=int(cfg["heads"]), layers=int(cfg["layers"]), dropout=float(cfg["dropout"])
    )

# ----------------------------
# 3) Training loop (pos_weight BCE, stable)
#     + optional focal (you used it earlier; we keep it optional)
# ----------------------------
class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32) if pos_weight is not None else None)

    def forward(self, logits, y):
        # logits: (B,1), y: (B,)
        y = y.view(-1, 1)
        bce = F.binary_cross_entropy_with_logits(
            logits, y,
            pos_weight=self.pos_weight if self.pos_weight is not None else None,
            reduction="none"
        )
        p = torch.sigmoid(logits)
        pt = torch.where(y == 1, p, 1 - p)
        w  = (self.alpha * y + (1 - self.alpha) * (1 - y)) * (1 - pt).pow(self.gamma)
        return (w * bce).mean()

@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    ps, ys = [], []
    for xc, xcat, xb, y in loader:
        xc, xcat, xb = xc.to(device), xcat.to(device), xb.to(device)
        logits = model(xc, xcat, xb).view(-1)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def train_one_seed(cfg, seed,
                   train_loader, val_loader, test_loader,
                   device="cpu",
                   use_focal=True, focal_alpha=0.85, focal_gamma=2.0,
                   posw_clip=4.0,
                   max_epochs=35, patience=7):

    torch.manual_seed(seed); np.random.seed(seed)

    # compute pos_weight from TRAIN only, then clip
    y_train_all = []
    for _,_,_,y in train_loader:
        y_train_all.append(y.numpy())
    y_train_all = np.concatenate(y_train_all).astype(int)
    pos = y_train_all.sum()
    neg = len(y_train_all) - pos
    raw_posw = float(neg / max(pos, 1))
    posw = float(min(raw_posw, posw_clip))

    model = build_model(cfg).to(device)

    # optimizer (AdamW is usually best for tabular transformers)
    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["wd"]))

    if use_focal:
        crit = FocalLossWithLogits(alpha=focal_alpha, gamma=focal_gamma, pos_weight=posw).to(device)
    else:
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([posw], device=device))

    best_state = None
    best_val_pr = -1.0
    bad = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        total = 0.0
        for xc, xcat, xb, y in train_loader:
            xc, xcat, xb, y = xc.to(device), xcat.to(device), xb.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xc, xcat, xb).view(-1, 1)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        # validate
        p_va, y_va = predict_proba(model, val_loader, device)
        val_roc = roc_auc_score(y_va, p_va)
        val_pr  = average_precision_score(y_va, p_va)
        print(f"seed {seed} | ep {ep:02d} | loss={total/len(train_loader):.4f} | VAL ROC={val_roc:.4f} | VAL PR={val_pr:.4f} | posw={posw:.2f}")

        if val_pr > best_val_pr + 1e-5:
            best_val_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # final preds
    p_va, y_va = predict_proba(model, val_loader, device)
    p_te, y_te = predict_proba(model, test_loader, device)

    out = {
        "seed": seed,
        "raw_posw": raw_posw,
        "posw_used": posw,
        **{("val_"+k): v for k,v in metrics(y_va, p_va).items()},
        **{("test_"+k): v for k,v in metrics(y_te, p_te).items()},
    }
    return model, out, (p_va, y_va), (p_te, y_te)

# ----------------------------
# 4) Run multi-seed ensemble, pick threshold ONLY on VAL
#     + Optional Platt calibration on VAL -> apply to TEST
# ----------------------------
def run_ensemble(cfg, seeds=(0,1,2,3,4),
                 batch_size=256,
                 device="cpu",
                 use_focal=True, focal_alpha=0.85, focal_gamma=2.0,
                 posw_clip=4.0,
                 max_epochs=35, patience=7,
                 do_platt=True):

    # loaders from YOUR existing arrays
    train_loader = make_loader(Xtr_cont, Xtr_cat_only, Xtr_bin, y_tr, batch_size=batch_size, shuffle=True)
    val_loader   = make_loader(Xva_cont, Xva_cat_only, Xva_bin, y_va, batch_size=batch_size, shuffle=False)
    test_loader  = make_loader(Xte_cont, Xte_cat_only, Xte_bin, y_te, batch_size=batch_size, shuffle=False)

    models, outs = [], []
    P_va_all, P_te_all = [], []
    y_va_np = to_numpy_1d(y_va).astype(int)
    y_te_np = to_numpy_1d(y_te).astype(int)

    for s in seeds:
        print("\n" + "="*70)
        m, out, (p_va, _), (p_te, _) = train_one_seed(
            cfg, s, train_loader, val_loader, test_loader,
            device=device, use_focal=use_focal,
            focal_alpha=focal_alpha, focal_gamma=focal_gamma,
            posw_clip=posw_clip, max_epochs=max_epochs, patience=patience
        )
        models.append(m); outs.append(out)
        P_va_all.append(p_va); P_te_all.append(p_te)

    P_va_all = np.stack(P_va_all, axis=0)  # (S, Nval)
    P_te_all = np.stack(P_te_all, axis=0)  # (S, Ntest)

    # ---- raw ensemble probs ----
    p_va_ens = P_va_all.mean(axis=0)
    p_te_ens = P_te_all.mean(axis=0)

    print("\n" + "="*70)
    print("ENSEMBLE (raw mean probs) VAL:", metrics(y_va_np, p_va_ens), "TEST:", metrics(y_te_np, p_te_ens))

    # Threshold chosen ONLY on VAL, applied to TEST (no leakage)
    best_f1, best_bal = scan_thresholds(y_va_np, p_va_ens)
    print("\nBest threshold from VAL (F1):", best_f1)
    print("Best threshold from VAL (BalAcc):", best_bal)

    def apply_thr_report(y, p, thr):
        y = np.asarray(y).astype(int)
        yhat = (p >= thr).astype(int)
        tp = int(((yhat==1)&(y==1)).sum())
        tn = int(((yhat==0)&(y==0)).sum())
        fp = int(((yhat==1)&(y==0)).sum())
        fn = int(((yhat==0)&(y==1)).sum())
        prec = tp/(tp+fp+1e-12)
        rec  = tp/(tp+fn+1e-12)
        f1   = 2*prec*rec/(prec+rec+1e-12)
        bal  = 0.5*(tp/(tp+fn+1e-12) + tn/(tn+fp+1e-12))
        acc  = (tp+tn)/(tp+tn+fp+fn+1e-12)
        return {"acc":acc, "bal_acc":bal, "f1":f1, "precision":prec, "recall":rec, "tn_fp_fn_tp":[tn,fp,fn,tp]}

    print("\nTEST report @ VAL-best-F1 thr:", apply_thr_report(y_te_np, p_te_ens, best_f1["thr"]))
    print("TEST report @ VAL-best-BalAcc thr:", apply_thr_report(y_te_np, p_te_ens, best_bal["thr"]))

    # ---- Optional: Platt calibration on VAL, apply to TEST ----
    if do_platt:
        # Fit logistic regression on validation probs (stacking-calibration)
        lr = LogisticRegression(max_iter=2000, class_weight="balanced")
        lr.fit(p_va_ens.reshape(-1,1), y_va_np)
        p_va_cal = lr.predict_proba(p_va_ens.reshape(-1,1))[:,1]
        p_te_cal = lr.predict_proba(p_te_ens.reshape(-1,1))[:,1]

        print("\n" + "="*70)
        print("ENSEMBLE + PLATT CALIBRATION VAL:", metrics(y_va_np, p_va_cal), "TEST:", metrics(y_te_np, p_te_cal))

        best_f1_cal, best_bal_cal = scan_thresholds(y_va_np, p_va_cal)
        print("\nBest threshold from VAL after calibration (F1):", best_f1_cal)
        print("TEST report @ VAL-best-F1 thr (cal):", apply_thr_report(y_te_np, p_te_cal, best_f1_cal["thr"]))

        return {
            "per_seed": outs,
            "ens_raw": {"val": metrics(y_va_np, p_va_ens), "test": metrics(y_te_np, p_te_ens),
                        "val_thr_f1": best_f1, "val_thr_bal": best_bal},
            "ens_platt": {"val": metrics(y_va_np, p_va_cal), "test": metrics(y_te_np, p_te_cal),
                          "val_thr_f1": best_f1_cal, "val_thr_bal": best_bal_cal}
        }

    return {
        "per_seed": outs,
        "ens_raw": {"val": metrics(y_va_np, p_va_ens), "test": metrics(y_te_np, p_te_ens),
                    "val_thr_f1": best_f1, "val_thr_bal": best_bal}
    }

# ----------------------------
# 5) Run (USE your best config)
# ----------------------------
best_cfg = {
    "d": 48, "heads": 4, "layers": 1, "dropout": 0.1,
    "lr": 0.003, "wd": 0.0003
}

result = run_ensemble(
    best_cfg,
    seeds=(0,1,2,3,4),
    device="cpu",              # you are on CPU
    use_focal=True,
    focal_alpha=0.85,          # your best focal alpha earlier
    focal_gamma=2.0,
    posw_clip=4.0,             # your best clip from search
    max_epochs=35,
    patience=7,
    do_platt=True              # advanced improvement
)

result


seed 0 | ep 01 | loss=0.0643 | VAL ROC=0.6914 | VAL PR=0.1290 | posw=4.00
seed 0 | ep 02 | loss=0.0594 | VAL ROC=0.6887 | VAL PR=0.1271 | posw=4.00
seed 0 | ep 03 | loss=0.0563 | VAL ROC=0.6890 | VAL PR=0.1250 | posw=4.00
seed 0 | ep 04 | loss=0.0546 | VAL ROC=0.6842 | VAL PR=0.1249 | posw=4.00
seed 0 | ep 05 | loss=0.0514 | VAL ROC=0.7102 | VAL PR=0.1346 | posw=4.00
seed 0 | ep 06 | loss=0.0511 | VAL ROC=0.6980 | VAL PR=0.1356 | posw=4.00
seed 0 | ep 07 | loss=0.0492 | VAL ROC=0.6835 | VAL PR=0.1368 | posw=4.00
seed 0 | ep 08 | loss=0.0449 | VAL ROC=0.6886 | VAL PR=0.1426 | posw=4.00
seed 0 | ep 09 | loss=0.0438 | VAL ROC=0.6756 | VAL PR=0.1293 | posw=4.00
seed 0 | ep 10 | loss=0.0458 | VAL ROC=0.6691 | VAL PR=0.1280 | posw=4.00
seed 0 | ep 11 | loss=0.0414 | VAL ROC=0.6865 | VAL PR=0.1513 | posw=4.00
seed 0 | ep 12 | loss=0.0437 | VAL ROC=0.6868 | VAL PR=0.1338 | posw=4.00
seed 0 | ep 13 | loss=0.0433 | VAL ROC=0.6826 | VAL PR=0.1306 | posw=4.00
seed 0 | ep 14 | loss=0.0410 | VAL RO

{'per_seed': [{'seed': 0,
   'raw_posw': 11.693965517241379,
   'posw_used': 4.0,
   'val_roc_auc': 0.6864811335127723,
   'val_pr_auc': 0.1513283331761221,
   'test_roc_auc': 0.6038149456877373,
   'test_pr_auc': 0.10467074685497704},
  {'seed': 1,
   'raw_posw': 11.693965517241379,
   'posw_used': 4.0,
   'val_roc_auc': 0.6846528871057843,
   'val_pr_auc': 0.14553929834444107,
   'test_roc_auc': 0.6084282162020678,
   'test_pr_auc': 0.11229272087072009},
  {'seed': 2,
   'raw_posw': 11.693965517241379,
   'posw_used': 4.0,
   'val_roc_auc': 0.7234523386318623,
   'val_pr_auc': 0.14818955200126596,
   'test_roc_auc': 0.6601066614317498,
   'test_pr_auc': 0.14049506114214305},
  {'seed': 3,
   'raw_posw': 11.693965517241379,
   'posw_used': 4.0,
   'val_roc_auc': 0.6799807018434818,
   'val_pr_auc': 0.17079897903417132,
   'test_roc_auc': 0.6223334642062558,
   'test_pr_auc': 0.11796291876134994},
  {'seed': 4,
   'raw_posw': 11.693965517241379,
   'posw_used': 4.0,
   'val_roc_auc': 0

In [24]:
# ============================================
# FULL RUNNABLE: Train S-seed FT-Transformer ensemble
# + build P_va_all / P_te_all
# + mean ensemble + stacking (LogReg) on VAL
# + VAL-only threshold selection -> TEST report
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression


# ----------------------------
# 0) Repro + Device
# ----------------------------
def seed_all(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


# ----------------------------
# 1) Dataset
# ----------------------------
class TabDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        self.X_cat  = torch.tensor(X_cat,  dtype=torch.long)
        self.X_bin  = torch.tensor(X_bin,  dtype=torch.float32)
        # y can be numpy or pandas; handle both
        if hasattr(y, "values"):
            y = y.values
        self.y = torch.tensor(np.asarray(y).reshape(-1), dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return (self.X_cont[i], self.X_cat[i], self.X_bin[i]), self.y[i]


# ----------------------------
# 2) Metrics + Threshold tools
# ----------------------------
def scan_thresholds(y, p, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)

    y = np.asarray(y).astype(int)
    p = np.asarray(p).reshape(-1)

    best_f1  = {"thr": None, "f1": -1}
    best_bal = {"thr": None, "bal_acc": -1}

    for thr in grid:
        pred = (p >= thr).astype(int)
        tn = np.sum((pred == 0) & (y == 0))
        fp = np.sum((pred == 1) & (y == 0))
        fn = np.sum((pred == 0) & (y == 1))
        tp = np.sum((pred == 1) & (y == 1))

        acc = (tp + tn) / max(1, (tp + tn + fp + fn))
        tpr = tp / max(1, (tp + fn))
        tnr = tn / max(1, (tn + fp))
        bal_acc = 0.5 * (tpr + tnr)

        prec = tp / max(1, (tp + fp))
        rec  = tpr
        f1 = 2 * prec * rec / max(1e-12, (prec + rec))

        if f1 > best_f1["f1"]:
            best_f1 = dict(thr=float(thr), acc=acc, bal_acc=bal_acc, f1=f1,
                           precision=prec, recall=rec,
                           tn_fp_fn_tp=[int(tn), int(fp), int(fn), int(tp)])

        if bal_acc > best_bal["bal_acc"]:
            best_bal = dict(thr=float(thr), acc=acc, bal_acc=bal_acc, f1=f1,
                            precision=prec, recall=rec,
                            tn_fp_fn_tp=[int(tn), int(fp), int(fn), int(tp)])
    return best_f1, best_bal

def report_at_thr(y, p, thr):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).reshape(-1)
    pred = (p >= thr).astype(int)
    tn = np.sum((pred == 0) & (y == 0))
    fp = np.sum((pred == 1) & (y == 0))
    fn = np.sum((pred == 0) & (y == 1))
    tp = np.sum((pred == 1) & (y == 1))

    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    tpr = tp / max(1, (tp + fn))
    tnr = tn / max(1, (tn + fp))
    bal_acc = 0.5 * (tpr + tnr)

    prec = tp / max(1, (tp + fp))
    rec  = tpr
    f1 = 2 * prec * rec / max(1e-12, (prec + rec))

    return dict(acc=acc, bal_acc=bal_acc, f1=f1, precision=prec, recall=rec,
                tn_fp_fn_tp=[int(tn), int(fp), int(fn), int(tp)])


# ----------------------------
# 3) Loss: BCEWithLogits + optional focal
# ----------------------------
def focal_bce_with_logits(logits, targets, pos_weight=None, gamma=2.0):
    """
    logits: (B,)
    targets: (B,) float {0,1}
    pos_weight: torch scalar or None
    """
    # standard BCE (no reduction)
    bce = F.binary_cross_entropy_with_logits(
        logits, targets,
        pos_weight=pos_weight,
        reduction="none"
    )
    p = torch.sigmoid(logits)
    pt = torch.where(targets > 0.5, p, 1 - p)
    focal = (1 - pt).pow(gamma) * bce
    return focal.mean()


# ----------------------------
# 4) FT-Transformer style model (simple + strong)
# ----------------------------
class TabTokenizer(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin, d):
        super().__init__()
        self.n_cont = n_cont
        self.n_bin  = n_bin
        self.cat_cardinalities = list(map(int, cat_cardinalities))

        # Continuous features -> token embeddings
        self.cont_w = nn.Parameter(torch.randn(n_cont, d) * 0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        # Binary block -> per-feature token embeddings
        self.bin_w = nn.Parameter(torch.randn(n_bin, d) * 0.02)
        self.bin_b = nn.Parameter(torch.zeros(n_bin, d))

        # Categorical: separate embedding per column
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(card, d) for card in self.cat_cardinalities
        ])

    def forward(self, x_cont, x_cat, x_bin):
        # cont tokens: (B, n_cont, d)
        # embed each scalar x as x * w + b
        cont = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        # bin tokens: (B, n_bin, d)
        bin_t = x_bin.unsqueeze(-1) * self.bin_w.unsqueeze(0) + self.bin_b.unsqueeze(0)

        # cat tokens: (B, n_cat, d)
        cat_tokens = []
        for j, emb in enumerate(self.cat_embeds):
            cat_tokens.append(emb(x_cat[:, j]))
        cat = torch.stack(cat_tokens, dim=1)

        # concat tokens
        return torch.cat([cont, bin_t, cat], dim=1)  # (B, T, d)


class FTModel(nn.Module):
    def __init__(self, n_cont, cat_cardinalities, n_bin,
                 d=48, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.tokenizer = TabTokenizer(n_cont, cat_cardinalities, n_bin, d)

        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=heads,
            dim_feedforward=4*d,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1)
        )

    def forward(self, batch_x):
        x_cont, x_cat, x_bin = batch_x
        tok = self.tokenizer(x_cont, x_cat, x_bin)  # (B, T, d)
        B = tok.size(0)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, tok], dim=1)
        x = self.encoder(x)
        x = self.norm(x[:, 0])
        logit = self.head(x).squeeze(-1)
        return logit


# ----------------------------
# 5) Train / Eval
# ----------------------------
@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    ps, ys = [], []
    for (xb, yb) in loader:
        x_cont, x_cat, x_bin = xb
        x_cont = x_cont.to(device)
        x_cat  = x_cat.to(device)
        x_bin  = x_bin.to(device)
        yb     = yb.to(device)

        logits = model((x_cont, x_cat, x_bin))
        prob = torch.sigmoid(logits)

        ps.append(prob.detach().cpu().numpy())
        ys.append(yb.detach().cpu().numpy())
    p = np.concatenate(ps).reshape(-1)
    y = np.concatenate(ys).reshape(-1).astype(int)
    return p, y


def train_one_seed(seed, cfg, train_loader, val_loader, test_loader, raw_pos_weight):
    seed_all(seed)

    # pos_weight clipped (you were using 4.0 successfully)
    posw_used = float(min(cfg["posw_clip"], raw_pos_weight))
    posw_t = torch.tensor(posw_used, dtype=torch.float32, device=device)

    model = FTModel(
        n_cont=cfg["n_cont"],
        cat_cardinalities=cfg["cat_cardinalities"],
        n_bin=cfg["n_bin"],
        d=cfg["d"],
        heads=cfg["heads"],
        layers=cfg["layers"],
        dropout=cfg["dropout"],
    ).to(device)

    # optimizer
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    # a bit of stability
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    best_state = None
    best_val_pr = -1
    bad = 0

    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        losses = []
        for (xb, yb) in train_loader:
            x_cont, x_cat, x_bin = xb
            x_cont = x_cont.to(device)
            x_cat  = x_cat.to(device)
            x_bin  = x_bin.to(device)
            yb     = yb.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model((x_cont, x_cat, x_bin))

            if cfg["use_focal"]:
                loss = focal_bce_with_logits(logits, yb, pos_weight=posw_t, gamma=cfg["focal_gamma"])
            else:
                loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=posw_t)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        sched.step()

        # VAL
        p_va, y_va = eval_loader(model, val_loader)
        val_roc = roc_auc_score(y_va, p_va)
        val_pr  = average_precision_score(y_va, p_va)

        print(f"seed {seed} | ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={val_roc:.4f} | VAL PR={val_pr:.4f} | posw={posw_used:.2f}")

        if val_pr > best_val_pr + 1e-6:
            best_val_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
        model.to(device)

    # TEST
    p_te, y_te = eval_loader(model, test_loader)
    out = dict(
        seed=seed,
        posw_used=posw_used,
        val_pr=best_val_pr,
        test_roc=roc_auc_score(y_te, p_te),
        test_pr=average_precision_score(y_te, p_te),
        p_va_last=None,  # filled outside if needed
        p_te=p_te,
        y_va=y_va,
        y_te=y_te
    )
    return model, out


# ----------------------------
# 6) MAIN: Ensemble + Stacking
# ----------------------------
def run_full_ensemble_and_stacking(
    Xtr_cont, Xva_cont, Xte_cont,
    Xtr_cat,  Xva_cat,  Xte_cat,
    Xtr_bin,  Xva_bin,  Xte_bin,
    y_tr, y_va, y_te,
    seeds=(0,1,2,3,4),
):
    # --- sanity (cats must be 0..K-1)
    # infer cardinalities from TRAIN
    cat_cardinalities = []
    for j in range(Xtr_cat.shape[1]):
        K = int(np.max(Xtr_cat[:, j])) + 1
        cat_cardinalities.append(K)
    print("cat_cardinalities (train inferred):", cat_cardinalities)

    # raw pos_weight from TRAIN
    ytr = np.asarray(y_tr).reshape(-1)
    pos = float(np.sum(ytr == 1))
    neg = float(np.sum(ytr == 0))
    raw_posw = neg / max(1.0, pos)
    print("raw pos_weight:", raw_posw)

    train_ds = TabDataset(Xtr_cont, Xtr_cat, Xtr_bin, y_tr)
    val_ds   = TabDataset(Xva_cont, Xva_cat, Xva_bin, y_va)
    test_ds  = TabDataset(Xte_cont, Xte_cat, Xte_bin, y_te)

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)

    # config (start from what worked best for you)
    cfg = dict(
        n_cont = Xtr_cont.shape[1],
        n_bin  = Xtr_bin.shape[1],
        cat_cardinalities = cat_cardinalities,

        d=48, heads=4, layers=2, dropout=0.10,
        lr=3e-3, wd=3e-4,
        epochs=40, patience=8,

        posw_clip=4.0,

        # focal often stabilizes PR-AUC on imbalance
        use_focal=True,
        focal_gamma=2.0
    )

    models = []
    P_va = []
    P_te = []

    y_va_ref = np.asarray(y_va).astype(int).reshape(-1)
    y_te_ref = np.asarray(y_te).astype(int).reshape(-1)

    for s in seeds:
        print("\n" + "="*70)
        model, out = train_one_seed(s, cfg, train_loader, val_loader, test_loader, raw_posw)

        # get VAL probs (best model)
        p_va, _ = eval_loader(model, val_loader)
        P_va.append(p_va)
        P_te.append(out["p_te"])
        models.append(model)

        print(f"seed {s} | TEST ROC/PR: {out['test_roc']:.4f} {out['test_pr']:.4f}")

    # Build arrays you were missing
    P_va_all = np.stack(P_va, axis=0)  # (S, Nval)
    P_te_all = np.stack(P_te, axis=0)  # (S, Ntest)
    print("\nP_va_all:", P_va_all.shape, "P_te_all:", P_te_all.shape)

    # Mean ensemble
    p_va_mean = P_va_all.mean(axis=0)
    p_te_mean = P_te_all.mean(axis=0)

    print("\n" + "="*70)
    print("MEAN ENSEMBLE VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_mean), average_precision_score(y_va_ref, p_va_mean))
    print("MEAN ENSEMBLE TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_mean), average_precision_score(y_te_ref, p_te_mean))

    # Stacking: LogisticRegression on VAL only
    X_va_meta = P_va_all.T  # (Nval, S)
    X_te_meta = P_te_all.T  # (Ntest, S)

    meta = LogisticRegression(max_iter=5000, class_weight="balanced")
    meta.fit(X_va_meta, y_va_ref)

    p_va_stack = meta.predict_proba(X_va_meta)[:, 1]
    p_te_stack = meta.predict_proba(X_te_meta)[:, 1]

    print("\n" + "="*70)
    print("STACKED (LogReg) VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_stack), average_precision_score(y_va_ref, p_va_stack))
    print("STACKED (LogReg) TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_stack), average_precision_score(y_te_ref, p_te_stack))

    # Thresholds from VAL only
    best_f1, best_bal = scan_thresholds(y_va_ref, p_va_stack)
    print("\nVAL best F1 thr:", best_f1)
    print("VAL best BalAcc thr:", best_bal)

    print("\nTEST @ VAL-best-F1 thr:", report_at_thr(y_te_ref, p_te_stack, best_f1["thr"]))
    print("TEST @ VAL-best-BalAcc thr:", report_at_thr(y_te_ref, p_te_stack, best_bal["thr"]))

    return {
        "P_va_all": P_va_all,
        "P_te_all": P_te_all,
        "p_te_mean": p_te_mean,
        "p_te_stack": p_te_stack,
        "y_va": y_va_ref,
        "y_te": y_te_ref,
        "meta": meta,
        "models": models,
        "cfg": cfg
    }


# =====================================================
# 7) RUN (expects your arrays already exist)
# =====================================================
# Required variables:
# Xtr_cont, Xva_cont, Xte_cont
# Xtr_cat,  Xva_cat,  Xte_cat     (must be int-coded 0..K-1)
# Xtr_bin,  Xva_bin,  Xte_bin
# y_tr, y_va, y_te

out = run_full_ensemble_and_stacking(
    Xtr_cont, Xva_cont, Xte_cont,
    Xtr_cat,  Xva_cat,  Xte_cat,
    Xtr_bin,  Xva_bin,  Xte_bin,
    y_tr, y_va, y_te,
    seeds=(0,1,2,3,4),
)

device: cpu
cat_cardinalities (train inferred): [3, 6, 7, 3, 3, 6, 4, 2, 2, 2, 2, 2, 2]
raw pos_weight: 11.693965517241379



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 0 | ep 01 | loss=0.1911 | VAL ROC=0.5831 | VAL PR=0.1128 | posw=4.00
seed 0 | ep 02 | loss=0.1782 | VAL ROC=0.6367 | VAL PR=0.1211 | posw=4.00
seed 0 | ep 03 | loss=0.1755 | VAL ROC=0.6140 | VAL PR=0.1127 | posw=4.00
seed 0 | ep 04 | loss=0.1686 | VAL ROC=0.6503 | VAL PR=0.1414 | posw=4.00
seed 0 | ep 05 | loss=0.1664 | VAL ROC=0.6621 | VAL PR=0.1450 | posw=4.00
seed 0 | ep 06 | loss=0.1618 | VAL ROC=0.6736 | VAL PR=0.1478 | posw=4.00
seed 0 | ep 07 | loss=0.1627 | VAL ROC=0.6640 | VAL PR=0.1456 | posw=4.00
seed 0 | ep 08 | loss=0.1650 | VAL ROC=0.6403 | VAL PR=0.1318 | posw=4.00
seed 0 | ep 09 | loss=0.1557 | VAL ROC=0.6612 | VAL PR=0.1403 | posw=4.00
seed 0 | ep 10 | loss=0.1601 | VAL ROC=0.6629 | VAL PR=0.1358 | posw=4.00
seed 0 | ep 11 | loss=0.1537 | VAL ROC=0.6501 | VAL PR=0.1322 | posw=4.00
seed 0 | ep 12 | loss=0.1501 | VAL ROC=0.6490 | VAL PR=0.1281 | posw=4.00
seed 0 | ep 13 | loss=0.1444 | VAL ROC=0.6488 | VAL PR=0.1372 | posw=4.00
seed 0 | ep 14 | loss=0.1510 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 1 | ep 01 | loss=0.1843 | VAL ROC=0.6048 | VAL PR=0.1275 | posw=4.00
seed 1 | ep 02 | loss=0.1712 | VAL ROC=0.6297 | VAL PR=0.1440 | posw=4.00
seed 1 | ep 03 | loss=0.1631 | VAL ROC=0.6750 | VAL PR=0.1579 | posw=4.00
seed 1 | ep 04 | loss=0.1654 | VAL ROC=0.6651 | VAL PR=0.1503 | posw=4.00
seed 1 | ep 05 | loss=0.1632 | VAL ROC=0.6675 | VAL PR=0.1564 | posw=4.00
seed 1 | ep 06 | loss=0.1565 | VAL ROC=0.6799 | VAL PR=0.1687 | posw=4.00
seed 1 | ep 07 | loss=0.1546 | VAL ROC=0.6875 | VAL PR=0.1575 | posw=4.00
seed 1 | ep 08 | loss=0.1529 | VAL ROC=0.6851 | VAL PR=0.1572 | posw=4.00
seed 1 | ep 09 | loss=0.1551 | VAL ROC=0.6633 | VAL PR=0.1660 | posw=4.00
seed 1 | ep 10 | loss=0.1535 | VAL ROC=0.6751 | VAL PR=0.1577 | posw=4.00
seed 1 | ep 11 | loss=0.1488 | VAL ROC=0.6997 | VAL PR=0.1638 | posw=4.00
seed 1 | ep 12 | loss=0.1512 | VAL ROC=0.7005 | VAL PR=0.1553 | posw=4.00
seed 1 | ep 13 | loss=0.1504 | VAL ROC=0.6983 | VAL PR=0.1592 | posw=4.00
seed 1 | ep 14 | loss=0.1463 | VAL ROC

...

seed 2 | ep 01 | loss=0.1841 | VAL ROC=0.5727 | VAL PR=0.1121 | posw=4.00
seed 2 | ep 02 | loss=0.1780 | VAL ROC=0.5979 | VAL PR=0.1347 | posw=4.00
seed 2 | ep 03 | loss=0.1701 | VAL ROC=0.6568 | VAL PR=0.1585 | posw=4.00
seed 2 | ep 04 | loss=0.1622 | VAL ROC=0.6742 | VAL PR=0.1601 | posw=4.00
seed 2 | ep 05 | loss=0.1609 | VAL ROC=0.6499 | VAL PR=0.1421 | posw=4.00
seed 2 | ep 06 | loss=0.1569 | VAL ROC=0.6592 | VAL PR=0.1627 | posw=4.00
seed 2 | ep 07 | loss=0.1583 | VAL ROC=0.6299 | VAL PR=0.1249 | posw=4.00
seed 2 | ep 08 | loss=0.1610 | VAL ROC=0.6623 | VAL PR=0.1578 | posw=4.00
seed 2 | ep 09 | loss=0.1537 | VAL ROC=0.6557 | VAL PR=0.1513 | posw=4.00
seed 2 | ep 10 | loss=0.1527 | VAL ROC=0.6760 | VAL PR=0.1498 | posw=4.00
seed 2 | ep 11 | loss=0.1580 | VAL ROC=0.6791 | VAL PR=0.1435 | posw=4.00
seed 2 | ep 12 | loss=0.1561 | VAL ROC=0.6693 | VAL PR=0.1465 | posw=4.00
seed 2 | ep 13 | loss=0.1508 | VAL ROC=0.6794 | VAL PR=0.1888 | posw=4.00
seed 2 | ep 14 | loss=0.1507 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 3 | ep 01 | loss=0.1847 | VAL ROC=0.5771 | VAL PR=0.1361 | posw=4.00
seed 3 | ep 02 | loss=0.1689 | VAL ROC=0.6436 | VAL PR=0.1379 | posw=4.00
seed 3 | ep 03 | loss=0.1610 | VAL ROC=0.6488 | VAL PR=0.1399 | posw=4.00
seed 3 | ep 04 | loss=0.1653 | VAL ROC=0.6393 | VAL PR=0.1382 | posw=4.00
seed 3 | ep 05 | loss=0.1593 | VAL ROC=0.6729 | VAL PR=0.1440 | posw=4.00
seed 3 | ep 06 | loss=0.1567 | VAL ROC=0.6776 | VAL PR=0.1469 | posw=4.00
seed 3 | ep 07 | loss=0.1570 | VAL ROC=0.6838 | VAL PR=0.1433 | posw=4.00
seed 3 | ep 08 | loss=0.1539 | VAL ROC=0.6964 | VAL PR=0.1554 | posw=4.00
seed 3 | ep 09 | loss=0.1505 | VAL ROC=0.7040 | VAL PR=0.1723 | posw=4.00
seed 3 | ep 10 | loss=0.1480 | VAL ROC=0.6825 | VAL PR=0.1569 | posw=4.00
seed 3 | ep 11 | loss=0.1456 | VAL ROC=0.6927 | VAL PR=0.1594 | posw=4.00
seed 3 | ep 12 | loss=0.1519 | VAL ROC=0.7131 | VAL PR=0.1915 | posw=4.00
seed 3 | ep 13 | loss=0.1492 | VAL ROC=0.7028 | VAL PR=0.1749 | posw=4.00
seed 3 | ep 14 | loss=0.1415 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 4 | ep 01 | loss=0.1906 | VAL ROC=0.5893 | VAL PR=0.1171 | posw=4.00
seed 4 | ep 02 | loss=0.1793 | VAL ROC=0.6266 | VAL PR=0.1323 | posw=4.00
seed 4 | ep 03 | loss=0.1721 | VAL ROC=0.6466 | VAL PR=0.1531 | posw=4.00
seed 4 | ep 04 | loss=0.1611 | VAL ROC=0.6366 | VAL PR=0.1408 | posw=4.00
seed 4 | ep 05 | loss=0.1646 | VAL ROC=0.6301 | VAL PR=0.1375 | posw=4.00
seed 4 | ep 06 | loss=0.1656 | VAL ROC=0.6411 | VAL PR=0.1554 | posw=4.00
seed 4 | ep 07 | loss=0.1604 | VAL ROC=0.6353 | VAL PR=0.1415 | posw=4.00
seed 4 | ep 08 | loss=0.1578 | VAL ROC=0.6288 | VAL PR=0.1322 | posw=4.00
seed 4 | ep 09 | loss=0.1597 | VAL ROC=0.6401 | VAL PR=0.1334 | posw=4.00
seed 4 | ep 10 | loss=0.1599 | VAL ROC=0.6489 | VAL PR=0.1288 | posw=4.00
seed 4 | ep 11 | loss=0.1543 | VAL ROC=0.6443 | VAL PR=0.1312 | posw=4.00
seed 4 | ep 12 | loss=0.1572 | VAL ROC=0.6672 | VAL PR=0.1604 | posw=4.00
seed 4 | ep 13 | loss=0.1514 | VAL ROC=0.6527 | VAL PR=0.1569 | posw=4.00
seed 4 | ep 14 | loss=0.1530 | VAL ROC

In [25]:
# ===========================
# FULL RUNNABLE PIPELINE CELL
# ===========================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression

# ---------------------------
# 0) SAFETY: convert y to numpy
# ---------------------------
def _to_np_1d(y):
    if hasattr(y, "values"):  # pandas
        y = y.values
    y = np.asarray(y).reshape(-1)
    return y

y_tr = _to_np_1d(y_tr)
y_va = _to_np_1d(y_va)
y_te = _to_np_1d(y_te)

# ---------------------------
# 1) Infer shapes + cardinalities (robust)
# ---------------------------
Xtr_cont = np.asarray(Xtr_cont, dtype=np.float32)
Xva_cont = np.asarray(Xva_cont, dtype=np.float32)
Xte_cont = np.asarray(Xte_cont, dtype=np.float32)

Xtr_cat  = np.asarray(Xtr_cat, dtype=np.int64)
Xva_cat  = np.asarray(Xva_cat, dtype=np.int64)
Xte_cat  = np.asarray(Xte_cat, dtype=np.int64)

Xtr_bin  = np.asarray(Xtr_bin, dtype=np.float32)
Xva_bin  = np.asarray(Xva_bin, dtype=np.float32)
Xte_bin  = np.asarray(Xte_bin, dtype=np.float32)

n_cont = Xtr_cont.shape[1]
n_cat  = Xtr_cat.shape[1]
n_bin  = Xtr_bin.shape[1]

# infer per-column cardinalities from TRAIN (assumes remapped to [0..K] or [1..K], both ok)
cat_cardinalities = []
for j in range(n_cat):
    mx = int(Xtr_cat[:, j].max())
    # embedding needs size >= max_index+1
    cat_cardinalities.append(mx + 1)

print("device: cpu")
print("n_cont/n_cat/n_bin:", n_cont, n_cat, n_bin)
print("cat_cardinalities (train inferred):", cat_cardinalities)

# raw pos_weight
pos = (y_tr == 1).sum()
neg = (y_tr == 0).sum()
raw_posw = float(neg / max(pos, 1))
print("raw pos_weight:", raw_posw)

# ---------------------------
# 2) Dataset
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.Xc = torch.tensor(X_cont, dtype=torch.float32)
        self.Xk = torch.tensor(X_cat, dtype=torch.long)
        self.Xb = torch.tensor(X_bin, dtype=torch.float32)
        self.y  = torch.tensor(_to_np_1d(y), dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 3) Model: tokenizer + Transformer encoder for tabular tokens
#    (simple but strong; matches your current "ConceptFT-like" behavior)
# ---------------------------
class FeatureTokenizer(nn.Module):
    """
    Produces token embeddings for:
      - cont features -> n_cont tokens
      - cat features  -> n_cat tokens
      - bin features  -> n_bin tokens
    Total tokens = n_cont + n_cat + n_bin
    """
    def __init__(self, n_cont, cat_cards, n_bin, d):
        super().__init__()
        self.n_cont = n_cont
        self.n_cat = len(cat_cards)
        self.n_bin = n_bin
        self.d = d

        # Continuous: per-feature linear projection via weight+bias (like FT-Transformer style)
        self.cont_w = nn.Parameter(torch.randn(n_cont, d) * 0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        # Binary: same style
        self.bin_w = nn.Parameter(torch.randn(n_bin, d) * 0.02)
        self.bin_b = nn.Parameter(torch.zeros(n_bin, d))

        # Cat embeddings
        self.cat_embeds = nn.ModuleList([nn.Embedding(int(card), d) for card in cat_cards])

    def forward(self, x_cont, x_cat, x_bin):
        # x_cont: (B, n_cont)
        # cont tokens: (B, n_cont, d)
        cont_tok = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        # x_bin: (B, n_bin) -> (B, n_bin, d)
        bin_tok  = x_bin.unsqueeze(-1)  * self.bin_w.unsqueeze(0)  + self.bin_b.unsqueeze(0)

        # cats: list of (B, d) -> stack -> (B, n_cat, d)
        cat_toks = []
        for j, emb in enumerate(self.cat_embeds):
            cat_toks.append(emb(x_cat[:, j]))
        cat_tok = torch.stack(cat_toks, dim=1)

        # concat: (B, T, d)
        return torch.cat([cont_tok, cat_tok, bin_tok], dim=1)

class TabTransformer(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d=48, heads=4, layers=1, dropout=0.1):
        super().__init__()
        self.tok = FeatureTokenizer(n_cont, cat_cards, n_bin, d)

        T = n_cont + len(cat_cards) + n_bin
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=heads,
            dim_feedforward=4*d,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)

        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1),
        )

    def forward(self, x_cont, x_cat, x_bin):
        x = self.tok(x_cont, x_cat, x_bin)
        x = x + self.pos
        x = self.enc(x)
        x = self.norm(x)
        # mean pooling
        x = x.mean(dim=1)
        logit = self.head(x).squeeze(-1)
        return logit

# ---------------------------
# 4) Train/Eval utilities
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader, device="cpu"):
    model.eval()
    ps, ys = [], []
    for xc, xk, xb, y in loader:
        xc, xk, xb = xc.to(device), xk.to(device), xb.to(device)
        logit = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def train_one_seed_return_probs(
    seed=0,
    cfg=None,
    batch_size=256,
    max_epochs=35,
    patience=6,
):
    """
    Returns: model, out_dict, p_val, y_val, p_test, y_test
    """
    if cfg is None:
        cfg = dict(d=48, heads=4, layers=1, dropout=0.1, lr=3e-3, wd=3e-4, posw_clip=4.0)

    # seeding
    torch.manual_seed(seed)
    np.random.seed(seed)

    # loaders
    tr_ds = TabDataset(Xtr_cont, Xtr_cat, Xtr_bin, y_tr)
    va_ds = TabDataset(Xva_cont, Xva_cat, Xva_bin, y_va)
    te_ds = TabDataset(Xte_cont, Xte_cat, Xte_bin, y_te)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

    device = "cpu"

    model = TabTransformer(
        n_cont=n_cont,
        cat_cards=cat_cardinalities,
        n_bin=n_bin,
        d=int(cfg["d"]),
        heads=int(cfg["heads"]),
        layers=int(cfg["layers"]),
        dropout=float(cfg["dropout"])
    ).to(device)

    # pos_weight (clipped)
    posw_used = float(min(raw_posw, float(cfg.get("posw_clip", 4.0))))
    posw_t = torch.tensor(posw_used, dtype=torch.float32)

    crit = nn.BCEWithLogitsLoss(pos_weight=posw_t)

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["wd"]))

    best_state = None
    best_val_pr = -1
    bad = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xc, xk, xb, y in tr_loader:
            xc, xk, xb, y = xc.to(device), xk.to(device), xb.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logit = model(xc, xk, xb)
            loss = crit(logit, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        # val
        p_va, y_va2 = predict_proba(model, va_loader, device=device)
        val_roc = roc_auc_score(y_va2, p_va)
        val_pr  = average_precision_score(y_va2, p_va)

        print(f"seed {seed} | ep {ep:02d} | loss={np.mean(losses):.4f} | VAL ROC={val_roc:.4f} | VAL PR={val_pr:.4f} | posw={posw_used:.2f}")

        # early stopping on VAL PR (works better for your imbalance)
        if val_pr > best_val_pr + 1e-4:
            best_val_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # outputs
    p_va, y_va2 = predict_proba(model, va_loader, device=device)
    p_te, y_te2 = predict_proba(model, te_loader, device=device)

    out = {
        "seed": seed,
        "posw_used": posw_used,
        "val_roc": float(roc_auc_score(y_va2, p_va)),
        "val_pr": float(average_precision_score(y_va2, p_va)),
        "test_roc": float(roc_auc_score(y_te2, p_te)),
        "test_pr": float(average_precision_score(y_te2, p_te)),
    }
    print(f"seed {seed} | TEST ROC/PR: {out['test_roc']:.4f} {out['test_pr']:.4f}")
    return model, out, p_va, y_va2, p_te, y_te2

# ---------------------------
# 5) Ensemble helpers
# ---------------------------
def mean_ensemble(P):  # P shape (S, N)
    return P.mean(axis=0)

def stack_logreg(P_va_all, y_va, P_te_all):
    # Features are per-seed probs => (N, S)
    X_va = P_va_all.T
    X_te = P_te_all.T

    lr = LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")
    lr.fit(X_va, y_va)
    p_va = lr.predict_proba(X_va)[:, 1]
    p_te = lr.predict_proba(X_te)[:, 1]
    return p_va, p_te, lr

def random_search_weights(P_va_all, y_va, P_te_all, y_te, n_trials=8000, seed=0):
    rng = np.random.default_rng(seed)
    S = P_va_all.shape[0]

    best = None
    best_score = -1

    for _ in range(n_trials):
        w = rng.random(S)
        w = w / (w.sum() + 1e-12)

        p_va = (w[:, None] * P_va_all).sum(axis=0)
        score = average_precision_score(y_va, p_va)  # optimize PR-AUC on VAL

        if score > best_score:
            best_score = score
            p_te = (w[:, None] * P_te_all).sum(axis=0)
            best = (w, p_va, p_te)

    w, p_va, p_te = best
    out = {
        "best_val_pr": float(best_score),
        "val_roc": float(roc_auc_score(y_va, p_va)),
        "val_pr": float(average_precision_score(y_va, p_va)),
        "test_roc": float(roc_auc_score(y_te, p_te)),
        "test_pr": float(average_precision_score(y_te, p_te)),
        "w": w
    }
    return out, p_va, p_te

# ---------------------------
# 6) Run seeds -> build P_va_all / P_te_all
# ---------------------------
cfg = dict(d=48, heads=4, layers=1, dropout=0.1, lr=3e-3, wd=3e-4, posw_clip=4.0)

P_va_all = []
P_te_all = []
outs = []

y_va_ref, y_te_ref = None, None

print("\n" + "="*70)
for seed in [0,1,2,3,4]:
    print("\n======================================================================\n")
    model, out, p_va, y_va2, p_te, y_te2 = train_one_seed_return_probs(seed=seed, cfg=cfg)

    P_va_all.append(p_va)
    P_te_all.append(p_te)
    outs.append(out)

    if y_va_ref is None:
        y_va_ref = y_va2.copy()
        y_te_ref = y_te2.copy()

P_va_all = np.vstack(P_va_all)   # (S, Nval)
P_te_all = np.vstack(P_te_all)   # (S, Ntest)

print("\nP_va_all:", P_va_all.shape, "P_te_all:", P_te_all.shape)

# ---------------------------
# 7) Mean ensemble
# ---------------------------
p_va_mean = mean_ensemble(P_va_all)
p_te_mean = mean_ensemble(P_te_all)

print("\n" + "="*70)
print("MEAN ENSEMBLE VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_mean), average_precision_score(y_va_ref, p_va_mean))
print("MEAN ENSEMBLE TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_mean), average_precision_score(y_te_ref, p_te_mean))

# ---------------------------
# 8) Stacked ensemble (LogReg)
# ---------------------------
p_va_stack, p_te_stack, lr_stack = stack_logreg(P_va_all, y_va_ref, P_te_all)

print("\n" + "="*70)
print("STACKED (LogReg) VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_stack), average_precision_score(y_va_ref, p_va_stack))
print("STACKED (LogReg) TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_stack), average_precision_score(y_te_ref, p_te_stack))

# ---------------------------
# 9) Random-search weighted ensemble (optimizes VAL PR-AUC)
# ---------------------------
best_w_out, p_va_w, p_te_w = random_search_weights(
    P_va_all, y_va_ref, P_te_all, y_te_ref,
    n_trials=8000, seed=0
)

print("\n" + "="*70)
print("WEIGHTED-RS (opt VAL PR) VAL ROC/PR:", best_w_out["val_roc"], best_w_out["val_pr"])
print("WEIGHTED-RS (opt VAL PR) TEST ROC/PR:", best_w_out["test_roc"], best_w_out["test_pr"])
print("Best weights:", np.round(best_w_out["w"], 4))

# ---------------------------
# 10) Summary table
# ---------------------------
print("\n" + "="*70)
print("Per-seed summary:")
for o in outs:
    print(o)

device: cpu
n_cont/n_cat/n_bin: 22 13 6
cat_cardinalities (train inferred): [3, 6, 7, 3, 3, 6, 4, 2, 2, 2, 2, 2, 2]
raw pos_weight: 11.693965517241379





/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 0 | ep 01 | loss=0.7220 | VAL ROC=0.6027 | VAL PR=0.1376 | posw=4.00
seed 0 | ep 02 | loss=0.6866 | VAL ROC=0.6233 | VAL PR=0.1287 | posw=4.00
seed 0 | ep 03 | loss=0.6425 | VAL ROC=0.6687 | VAL PR=0.1430 | posw=4.00
seed 0 | ep 04 | loss=0.6269 | VAL ROC=0.6786 | VAL PR=0.1429 | posw=4.00
seed 0 | ep 05 | loss=0.6238 | VAL ROC=0.6897 | VAL PR=0.1466 | posw=4.00
seed 0 | ep 06 | loss=0.5974 | VAL ROC=0.6916 | VAL PR=0.1471 | posw=4.00
seed 0 | ep 07 | loss=0.6041 | VAL ROC=0.6961 | VAL PR=0.1567 | posw=4.00
seed 0 | ep 08 | loss=0.6057 | VAL ROC=0.6982 | VAL PR=0.1557 | posw=4.00
seed 0 | ep 09 | loss=0.5767 | VAL ROC=0.6987 | VAL PR=0.1540 | posw=4.00
seed 0 | ep 10 | loss=0.5891 | VAL ROC=0.6935 | VAL PR=0.1554 | posw=4.00
seed 0 | ep 11 | loss=0.5744 | VAL ROC=0.7066 | VAL PR=0.1659 | posw=4.00
seed 0 | ep 12 | loss=0.5700 | VAL ROC=0.6937 | VAL PR=0.1537 | posw=4.00
seed 0 | ep 13 | loss=0.5564 | VAL ROC=0.6997 | VAL PR=0.1628 | posw=4.00
seed 0 | ep 14 | loss=0.5549 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 1 | ep 01 | loss=0.7256 | VAL ROC=0.5808 | VAL PR=0.1352 | posw=4.00
seed 1 | ep 02 | loss=0.6836 | VAL ROC=0.6357 | VAL PR=0.1426 | posw=4.00
seed 1 | ep 03 | loss=0.6501 | VAL ROC=0.6558 | VAL PR=0.1474 | posw=4.00
seed 1 | ep 04 | loss=0.6222 | VAL ROC=0.6652 | VAL PR=0.1448 | posw=4.00
seed 1 | ep 05 | loss=0.6166 | VAL ROC=0.6862 | VAL PR=0.1588 | posw=4.00
seed 1 | ep 06 | loss=0.6134 | VAL ROC=0.6897 | VAL PR=0.1557 | posw=4.00
seed 1 | ep 07 | loss=0.5924 | VAL ROC=0.6773 | VAL PR=0.1562 | posw=4.00
seed 1 | ep 08 | loss=0.5853 | VAL ROC=0.6695 | VAL PR=0.1547 | posw=4.00
seed 1 | ep 09 | loss=0.5854 | VAL ROC=0.6844 | VAL PR=0.1572 | posw=4.00
seed 1 | ep 10 | loss=0.5748 | VAL ROC=0.6881 | VAL PR=0.1608 | posw=4.00
seed 1 | ep 11 | loss=0.5803 | VAL ROC=0.6818 | VAL PR=0.1490 | posw=4.00
seed 1 | ep 12 | loss=0.5862 | VAL ROC=0.6960 | VAL PR=0.1472 | posw=4.00
seed 1 | ep 13 | loss=0.5662 | VAL ROC=0.6683 | VAL PR=0.1432 | posw=4.00
seed 1 | ep 14 | loss=0.5573 | VAL ROC

...

seed 2 | ep 01 | loss=0.7123 | VAL ROC=0.5904 | VAL PR=0.1250 | posw=4.00
seed 2 | ep 02 | loss=0.6865 | VAL ROC=0.6093 | VAL PR=0.1366 | posw=4.00
seed 2 | ep 03 | loss=0.6601 | VAL ROC=0.6430 | VAL PR=0.1240 | posw=4.00
seed 2 | ep 04 | loss=0.6440 | VAL ROC=0.6471 | VAL PR=0.1319 | posw=4.00
seed 2 | ep 05 | loss=0.6274 | VAL ROC=0.6559 | VAL PR=0.1323 | posw=4.00
seed 2 | ep 06 | loss=0.6207 | VAL ROC=0.6645 | VAL PR=0.1354 | posw=4.00
seed 2 | ep 07 | loss=0.6035 | VAL ROC=0.6729 | VAL PR=0.1380 | posw=4.00
seed 2 | ep 08 | loss=0.5888 | VAL ROC=0.6707 | VAL PR=0.1369 | posw=4.00
seed 2 | ep 09 | loss=0.5803 | VAL ROC=0.6684 | VAL PR=0.1386 | posw=4.00
seed 2 | ep 10 | loss=0.5799 | VAL ROC=0.6796 | VAL PR=0.1496 | posw=4.00
seed 2 | ep 11 | loss=0.5747 | VAL ROC=0.6786 | VAL PR=0.1468 | posw=4.00
seed 2 | ep 12 | loss=0.5713 | VAL ROC=0.6797 | VAL PR=0.1449 | posw=4.00
seed 2 | ep 13 | loss=0.5741 | VAL ROC=0.6782 | VAL PR=0.1495 | posw=4.00
seed 2 | ep 14 | loss=0.5661 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 3 | ep 01 | loss=0.7242 | VAL ROC=0.6008 | VAL PR=0.1255 | posw=4.00
seed 3 | ep 02 | loss=0.6949 | VAL ROC=0.6273 | VAL PR=0.1345 | posw=4.00
seed 3 | ep 03 | loss=0.6507 | VAL ROC=0.6678 | VAL PR=0.1467 | posw=4.00
seed 3 | ep 04 | loss=0.6289 | VAL ROC=0.6749 | VAL PR=0.1542 | posw=4.00
seed 3 | ep 05 | loss=0.6164 | VAL ROC=0.6785 | VAL PR=0.1575 | posw=4.00
seed 3 | ep 06 | loss=0.5944 | VAL ROC=0.6842 | VAL PR=0.1692 | posw=4.00
seed 3 | ep 07 | loss=0.5990 | VAL ROC=0.6915 | VAL PR=0.1754 | posw=4.00
seed 3 | ep 08 | loss=0.5851 | VAL ROC=0.6940 | VAL PR=0.1780 | posw=4.00
seed 3 | ep 09 | loss=0.5805 | VAL ROC=0.6925 | VAL PR=0.1727 | posw=4.00
seed 3 | ep 10 | loss=0.5715 | VAL ROC=0.7027 | VAL PR=0.1718 | posw=4.00
seed 3 | ep 11 | loss=0.5731 | VAL ROC=0.7035 | VAL PR=0.1788 | posw=4.00
seed 3 | ep 12 | loss=0.5649 | VAL ROC=0.6963 | VAL PR=0.1664 | posw=4.00
seed 3 | ep 13 | loss=0.5521 | VAL ROC=0.6966 | VAL PR=0.1656 | posw=4.00
seed 3 | ep 14 | loss=0.5600 | VAL ROC

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 4 | ep 01 | loss=0.7328 | VAL ROC=0.5856 | VAL PR=0.1459 | posw=4.00
seed 4 | ep 02 | loss=0.6889 | VAL ROC=0.6382 | VAL PR=0.1529 | posw=4.00
seed 4 | ep 03 | loss=0.6466 | VAL ROC=0.6596 | VAL PR=0.1520 | posw=4.00
seed 4 | ep 04 | loss=0.6365 | VAL ROC=0.6708 | VAL PR=0.1499 | posw=4.00
seed 4 | ep 05 | loss=0.5994 | VAL ROC=0.6751 | VAL PR=0.1442 | posw=4.00
seed 4 | ep 06 | loss=0.5961 | VAL ROC=0.6954 | VAL PR=0.1612 | posw=4.00
seed 4 | ep 07 | loss=0.6064 | VAL ROC=0.6719 | VAL PR=0.1491 | posw=4.00
seed 4 | ep 08 | loss=0.5838 | VAL ROC=0.6934 | VAL PR=0.1517 | posw=4.00
seed 4 | ep 09 | loss=0.5784 | VAL ROC=0.6930 | VAL PR=0.1491 | posw=4.00
seed 4 | ep 10 | loss=0.5816 | VAL ROC=0.6848 | VAL PR=0.1504 | posw=4.00
seed 4 | ep 11 | loss=0.5779 | VAL ROC=0.6852 | VAL PR=0.1520 | posw=4.00
seed 4 | ep 12 | loss=0.5564 | VAL ROC=0.6880 | VAL PR=0.1417 | posw=4.00
seed 4 | TEST ROC/PR: 0.6169 0.1242

P_va_all: (5, 737) P_te_all: (5, 921)

MEAN ENSEMBLE VAL ROC/PR: 0.70191965

In [26]:
# ===========================
# FULL RUNNABLE PIPELINE CELL (CLS + FOCAL/BCE + ENSEMBLES)
# ===========================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression

# ---------------------------
# 0) SAFETY: convert y to numpy
# ---------------------------
def _to_np_1d(y):
    if hasattr(y, "values"):  # pandas
        y = y.values
    y = np.asarray(y).reshape(-1)
    return y

y_tr = _to_np_1d(y_tr)
y_va = _to_np_1d(y_va)
y_te = _to_np_1d(y_te)

# ---------------------------
# 1) Infer shapes + cardinalities (robust)
# ---------------------------
Xtr_cont = np.asarray(Xtr_cont, dtype=np.float32)
Xva_cont = np.asarray(Xva_cont, dtype=np.float32)
Xte_cont = np.asarray(Xte_cont, dtype=np.float32)

Xtr_cat  = np.asarray(Xtr_cat, dtype=np.int64)
Xva_cat  = np.asarray(Xva_cat, dtype=np.int64)
Xte_cat  = np.asarray(Xte_cat, dtype=np.int64)

Xtr_bin  = np.asarray(Xtr_bin, dtype=np.float32)
Xva_bin  = np.asarray(Xva_bin, dtype=np.float32)
Xte_bin  = np.asarray(Xte_bin, dtype=np.float32)

n_cont = Xtr_cont.shape[1]
n_cat  = Xtr_cat.shape[1]
n_bin  = Xtr_bin.shape[1]

cat_cardinalities = []
for j in range(n_cat):
    mx = int(Xtr_cat[:, j].max())
    cat_cardinalities.append(mx + 1)

print("device: cpu")
print("n_cont/n_cat/n_bin:", n_cont, n_cat, n_bin)
print("cat_cardinalities (train inferred):", cat_cardinalities)

pos = int((y_tr == 1).sum())
neg = int((y_tr == 0).sum())
raw_posw = float(neg / max(pos, 1))
print("raw pos_weight:", raw_posw)

# ---------------------------
# 2) Dataset
# ---------------------------
class TabDataset(Dataset):
    def __init__(self, X_cont, X_cat, X_bin, y):
        self.Xc = torch.tensor(X_cont, dtype=torch.float32)
        self.Xk = torch.tensor(X_cat, dtype=torch.long)
        self.Xb = torch.tensor(X_bin, dtype=torch.float32)
        self.y  = torch.tensor(_to_np_1d(y), dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xk[i], self.Xb[i], self.y[i]

# ---------------------------
# 3) Loss: Focal BCE with logits (optional)
# ---------------------------
class FocalBCEWithLogits(nn.Module):
    """
    Focal version of BCEWithLogits for binary classification.
    - pos_weight behaves like BCEWithLogitsLoss(pos_weight=...)
    """
    def __init__(self, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.gamma = float(gamma)
        self.register_buffer("pos_weight", torch.tensor(float(pos_weight)))

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        loss = ((1 - pt) ** self.gamma) * bce
        return loss.mean()

# ---------------------------
# 4) Model: tokenizer + Transformer encoder (CLS pooling)
# ---------------------------
class FeatureTokenizer(nn.Module):
    """
    Produces token embeddings for:
      - cont features -> n_cont tokens
      - cat features  -> n_cat tokens
      - bin features  -> n_bin tokens
    Total tokens = n_cont + n_cat + n_bin
    """
    def __init__(self, n_cont, cat_cards, n_bin, d):
        super().__init__()
        self.n_cont = n_cont
        self.n_cat = len(cat_cards)
        self.n_bin = n_bin
        self.d = d

        # Continuous: per-feature linear projection
        self.cont_w = nn.Parameter(torch.randn(n_cont, d) * 0.02)
        self.cont_b = nn.Parameter(torch.zeros(n_cont, d))

        # Binary: per-feature linear projection
        self.bin_w = nn.Parameter(torch.randn(n_bin, d) * 0.02)
        self.bin_b = nn.Parameter(torch.zeros(n_bin, d))

        # Cat embeddings
        self.cat_embeds = nn.ModuleList([nn.Embedding(int(card), d) for card in cat_cards])

    def forward(self, x_cont, x_cat, x_bin):
        # cont tokens: (B, n_cont, d)
        cont_tok = x_cont.unsqueeze(-1) * self.cont_w.unsqueeze(0) + self.cont_b.unsqueeze(0)

        # bin tokens: (B, n_bin, d)
        bin_tok  = x_bin.unsqueeze(-1)  * self.bin_w.unsqueeze(0)  + self.bin_b.unsqueeze(0)

        # cat tokens: (B, n_cat, d)
        cat_toks = []
        for j, emb in enumerate(self.cat_embeds):
            cat_toks.append(emb(x_cat[:, j]))
        cat_tok = torch.stack(cat_toks, dim=1)

        return torch.cat([cont_tok, cat_tok, bin_tok], dim=1)

class TabTransformerCLS(nn.Module):
    def __init__(self, n_cont, cat_cards, n_bin, d=48, heads=4, layers=1, dropout=0.1):
        super().__init__()
        self.tok = FeatureTokenizer(n_cont, cat_cards, n_bin, d)

        T = n_cont + len(cat_cards) + n_bin

        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.normal_(self.cls, std=0.02)

        self.pos = nn.Parameter(torch.zeros(1, T + 1, d))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=heads,
            dim_feedforward=4*d,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)

        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1),
        )

    def forward(self, x_cont, x_cat, x_bin):
        x = self.tok(x_cont, x_cat, x_bin)           # (B, T, d)
        B = x.size(0)
        cls = self.cls.expand(B, -1, -1)             # (B, 1, d)
        x = torch.cat([cls, x], dim=1)               # (B, T+1, d)
        x = x + self.pos
        x = self.enc(x)
        x = self.norm(x)
        cls_out = x[:, 0]                            # CLS pooling
        logit = self.head(cls_out).squeeze(-1)
        return logit

# ---------------------------
# 5) Train/Eval utilities
# ---------------------------
@torch.no_grad()
def predict_proba(model, loader, device="cpu"):
    model.eval()
    ps, ys = [], []
    for xc, xk, xb, y in loader:
        xc, xk, xb = xc.to(device), xk.to(device), xb.to(device)
        logit = model(xc, xk, xb)
        p = torch.sigmoid(logit).detach().cpu().numpy()
        ps.append(p)
        ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def train_one_seed_return_probs(
    seed=0,
    cfg=None,
    batch_size=256,
    max_epochs=40,
    patience=7,
):
    """
    Returns: model, out_dict, p_val, y_val, p_test, y_test
    """
    if cfg is None:
        cfg = dict(
            d=64, heads=4, layers=2, dropout=0.15,
            lr=2e-3, wd=5e-4,
            posw_clip=8.0,
            loss_type="focal",   # "bce" or "focal"
            focal_gamma=2.0
        )

    # seeding
    torch.manual_seed(seed)
    np.random.seed(seed)

    # loaders
    tr_ds = TabDataset(Xtr_cont, Xtr_cat, Xtr_bin, y_tr)
    va_ds = TabDataset(Xva_cont, Xva_cat, Xva_bin, y_va)
    te_ds = TabDataset(Xte_cont, Xte_cat, Xte_bin, y_te)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

    device = "cpu"

    model = TabTransformerCLS(
        n_cont=n_cont,
        cat_cards=cat_cardinalities,
        n_bin=n_bin,
        d=int(cfg["d"]),
        heads=int(cfg["heads"]),
        layers=int(cfg["layers"]),
        dropout=float(cfg["dropout"]),
    ).to(device)

    # pos_weight (clipped)
    posw_used = float(min(raw_posw, float(cfg.get("posw_clip", raw_posw))))
    posw_t = torch.tensor(posw_used, dtype=torch.float32)

    # loss
    if cfg.get("loss_type", "focal").lower() == "bce":
        crit = nn.BCEWithLogitsLoss(pos_weight=posw_t)
    else:
        crit = FocalBCEWithLogits(gamma=float(cfg.get("focal_gamma", 2.0)), pos_weight=posw_used)

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["wd"]))

    best_state = None
    best_val_pr = -1.0
    bad = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xc, xk, xb, y in tr_loader:
            xc, xk, xb, y = xc.to(device), xk.to(device), xb.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logit = model(xc, xk, xb)
            loss = crit(logit, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        # val
        p_va, y_va2 = predict_proba(model, va_loader, device=device)
        val_roc = roc_auc_score(y_va2, p_va)
        val_pr  = average_precision_score(y_va2, p_va)

        print(
            f"seed {seed} | ep {ep:02d} | loss={np.mean(losses):.4f} | "
            f"VAL ROC={val_roc:.4f} | VAL PR={val_pr:.4f} | posw={posw_used:.2f} | loss={cfg.get('loss_type')}"
        )

        if val_pr > best_val_pr + 1e-4:
            best_val_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # outputs
    p_va, y_va2 = predict_proba(model, va_loader, device=device)
    p_te, y_te2 = predict_proba(model, te_loader, device=device)

    out = {
        "seed": int(seed),
        "posw_used": float(posw_used),
        "val_roc": float(roc_auc_score(y_va2, p_va)),
        "val_pr": float(average_precision_score(y_va2, p_va)),
        "test_roc": float(roc_auc_score(y_te2, p_te)),
        "test_pr": float(average_precision_score(y_te2, p_te)),
    }
    print(f"seed {seed} | TEST ROC/PR: {out['test_roc']:.4f} {out['test_pr']:.4f}")
    return model, out, p_va, y_va2, p_te, y_te2

# ---------------------------
# 6) Ensemble helpers
# ---------------------------
def mean_ensemble(P):  # P shape (S, N)
    return P.mean(axis=0)

def stack_logreg(P_va_all, y_va, P_te_all):
    X_va = P_va_all.T
    X_te = P_te_all.T
    lr = LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")
    lr.fit(X_va, y_va)
    p_va = lr.predict_proba(X_va)[:, 1]
    p_te = lr.predict_proba(X_te)[:, 1]
    return p_va, p_te, lr

def random_search_weights(P_va_all, y_va, P_te_all, y_te, n_trials=8000, seed=0):
    rng = np.random.default_rng(seed)
    S = P_va_all.shape[0]
    best_w, best_val_pr, best_p_va, best_p_te = None, -1.0, None, None

    for _ in range(n_trials):
        w = rng.random(S)
        w = w / (w.sum() + 1e-12)
        p_va = (w[:, None] * P_va_all).sum(axis=0)
        score = average_precision_score(y_va, p_va)
        if score > best_val_pr:
            best_val_pr = score
            best_w = w
            best_p_va = p_va
            best_p_te = (w[:, None] * P_te_all).sum(axis=0)

    out = {
        "best_val_pr": float(best_val_pr),
        "val_roc": float(roc_auc_score(y_va, best_p_va)),
        "val_pr": float(average_precision_score(y_va, best_p_va)),
        "test_roc": float(roc_auc_score(y_te, best_p_te)),
        "test_pr": float(average_precision_score(y_te, best_p_te)),
        "w": best_w
    }
    return out, best_p_va, best_p_te

# ---------------------------
# 7) RUN
# ---------------------------
cfg = dict(
    d=64, heads=4, layers=2, dropout=0.15,
    lr=2e-3, wd=5e-4,
    posw_clip=8.0,
    loss_type="focal",     # "focal" or "bce"
    focal_gamma=2.0
)

seeds = [0, 1, 2, 3, 4]

P_va_all, P_te_all, outs = [], [], []
y_va_ref, y_te_ref = None, None

print("\n" + "="*70)
for seed in seeds:
    print("\n" + "="*70)
    model, out, p_va, y_va2, p_te, y_te2 = train_one_seed_return_probs(seed=seed, cfg=cfg)
    P_va_all.append(p_va)
    P_te_all.append(p_te)
    outs.append(out)
    if y_va_ref is None:
        y_va_ref = y_va2.copy()
        y_te_ref = y_te2.copy()

P_va_all = np.vstack(P_va_all)
P_te_all = np.vstack(P_te_all)

print("\nP_va_all:", P_va_all.shape, "P_te_all:", P_te_all.shape)

# Mean ensemble
p_va_mean = mean_ensemble(P_va_all)
p_te_mean = mean_ensemble(P_te_all)
print("\n" + "="*70)
print("MEAN ENSEMBLE VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_mean), average_precision_score(y_va_ref, p_va_mean))
print("MEAN ENSEMBLE TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_mean), average_precision_score(y_te_ref, p_te_mean))

# Stacked LogReg
p_va_stack, p_te_stack, lr_stack = stack_logreg(P_va_all, y_va_ref, P_te_all)
print("\n" + "="*70)
print("STACKED (LogReg) VAL ROC/PR:", roc_auc_score(y_va_ref, p_va_stack), average_precision_score(y_va_ref, p_va_stack))
print("STACKED (LogReg) TEST ROC/PR:", roc_auc_score(y_te_ref, p_te_stack), average_precision_score(y_te_ref, p_te_stack))

# Weighted random search
best_w_out, p_va_w, p_te_w = random_search_weights(P_va_all, y_va_ref, P_te_all, y_te_ref, n_trials=8000, seed=0)
print("\n" + "="*70)
print("WEIGHTED-RS (opt VAL PR) VAL ROC/PR:", best_w_out["val_roc"], best_w_out["val_pr"])
print("WEIGHTED-RS (opt VAL PR) TEST ROC/PR:", best_w_out["test_roc"], best_w_out["test_pr"])
print("Best weights:", np.round(best_w_out["w"], 4))

print("\n" + "="*70)
print("Per-seed summary:")
for o in outs:
    print(o)

device: cpu
n_cont/n_cat/n_bin: 22 13 6
cat_cardinalities (train inferred): [3, 6, 7, 3, 3, 6, 4, 2, 2, 2, 2, 2, 2]
raw pos_weight: 11.693965517241379




/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 0 | ep 01 | loss=0.2726 | VAL ROC=0.6299 | VAL PR=0.1301 | posw=8.00 | loss=focal
seed 0 | ep 02 | loss=0.2544 | VAL ROC=0.6663 | VAL PR=0.1471 | posw=8.00 | loss=focal
seed 0 | ep 03 | loss=0.2449 | VAL ROC=0.6750 | VAL PR=0.1538 | posw=8.00 | loss=focal
seed 0 | ep 04 | loss=0.2340 | VAL ROC=0.6856 | VAL PR=0.1556 | posw=8.00 | loss=focal
seed 0 | ep 05 | loss=0.2336 | VAL ROC=0.6899 | VAL PR=0.1474 | posw=8.00 | loss=focal
seed 0 | ep 06 | loss=0.2349 | VAL ROC=0.6910 | VAL PR=0.1539 | posw=8.00 | loss=focal
seed 0 | ep 07 | loss=0.2335 | VAL ROC=0.6777 | VAL PR=0.1447 | posw=8.00 | loss=focal
seed 0 | ep 08 | loss=0.2338 | VAL ROC=0.6913 | VAL PR=0.1650 | posw=8.00 | loss=focal
seed 0 | ep 09 | loss=0.2217 | VAL ROC=0.6811 | VAL PR=0.1764 | posw=8.00 | loss=focal
seed 0 | ep 10 | loss=0.2302 | VAL ROC=0.6568 | VAL PR=0.1555 | posw=8.00 | loss=focal
seed 0 | ep 11 | loss=0.2368 | VAL ROC=0.6440 | VAL PR=0.1554 | posw=8.00 | loss=focal
seed 0 | ep 12 | loss=0.2317 | VAL ROC=0.67

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 1 | ep 01 | loss=0.2745 | VAL ROC=0.5743 | VAL PR=0.1458 | posw=8.00 | loss=focal
seed 1 | ep 02 | loss=0.2625 | VAL ROC=0.6071 | VAL PR=0.1525 | posw=8.00 | loss=focal
seed 1 | ep 03 | loss=0.2523 | VAL ROC=0.6508 | VAL PR=0.1452 | posw=8.00 | loss=focal
seed 1 | ep 04 | loss=0.2393 | VAL ROC=0.6662 | VAL PR=0.1407 | posw=8.00 | loss=focal
seed 1 | ep 05 | loss=0.2323 | VAL ROC=0.6684 | VAL PR=0.1520 | posw=8.00 | loss=focal
seed 1 | ep 06 | loss=0.2323 | VAL ROC=0.6645 | VAL PR=0.1556 | posw=8.00 | loss=focal
seed 1 | ep 07 | loss=0.2274 | VAL ROC=0.6826 | VAL PR=0.1475 | posw=8.00 | loss=focal
seed 1 | ep 08 | loss=0.2358 | VAL ROC=0.6747 | VAL PR=0.1546 | posw=8.00 | loss=focal
seed 1 | ep 09 | loss=0.2282 | VAL ROC=0.6683 | VAL PR=0.1461 | posw=8.00 | loss=focal
seed 1 | ep 10 | loss=0.2233 | VAL ROC=0.6744 | VAL PR=0.1469 | posw=8.00 | loss=focal
seed 1 | ep 11 | loss=0.2192 | VAL ROC=0.6669 | VAL PR=0.1598 | posw=8.00 | loss=focal
seed 1 | ep 12 | loss=0.2231 | VAL ROC=0.66

...

seed 2 | ep 01 | loss=0.2691 | VAL ROC=0.5576 | VAL PR=0.1402 | posw=8.00 | loss=focal
seed 2 | ep 02 | loss=0.2637 | VAL ROC=0.5979 | VAL PR=0.1382 | posw=8.00 | loss=focal
seed 2 | ep 03 | loss=0.2533 | VAL ROC=0.6070 | VAL PR=0.1249 | posw=8.00 | loss=focal
seed 2 | ep 04 | loss=0.2501 | VAL ROC=0.6615 | VAL PR=0.1529 | posw=8.00 | loss=focal
seed 2 | ep 05 | loss=0.2316 | VAL ROC=0.6556 | VAL PR=0.1520 | posw=8.00 | loss=focal
seed 2 | ep 06 | loss=0.2289 | VAL ROC=0.6622 | VAL PR=0.1407 | posw=8.00 | loss=focal
seed 2 | ep 07 | loss=0.2321 | VAL ROC=0.6668 | VAL PR=0.1476 | posw=8.00 | loss=focal
seed 2 | ep 08 | loss=0.2432 | VAL ROC=0.6734 | VAL PR=0.1399 | posw=8.00 | loss=focal
seed 2 | ep 09 | loss=0.2266 | VAL ROC=0.6544 | VAL PR=0.1369 | posw=8.00 | loss=focal
seed 2 | ep 10 | loss=0.2342 | VAL ROC=0.6389 | VAL PR=0.1491 | posw=8.00 | loss=focal
seed 2 | ep 11 | loss=0.2375 | VAL ROC=0.6828 | VAL PR=0.1593 | posw=8.00 | loss=focal
seed 2 | ep 12 | loss=0.2288 | VAL ROC=0.67

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 3 | ep 01 | loss=0.2729 | VAL ROC=0.6297 | VAL PR=0.1345 | posw=8.00 | loss=focal
seed 3 | ep 02 | loss=0.2529 | VAL ROC=0.6374 | VAL PR=0.1388 | posw=8.00 | loss=focal
seed 3 | ep 03 | loss=0.2421 | VAL ROC=0.6706 | VAL PR=0.1529 | posw=8.00 | loss=focal
seed 3 | ep 04 | loss=0.2395 | VAL ROC=0.6824 | VAL PR=0.1331 | posw=8.00 | loss=focal
seed 3 | ep 05 | loss=0.2340 | VAL ROC=0.6756 | VAL PR=0.1373 | posw=8.00 | loss=focal
seed 3 | ep 06 | loss=0.2297 | VAL ROC=0.6625 | VAL PR=0.1402 | posw=8.00 | loss=focal
seed 3 | ep 07 | loss=0.2226 | VAL ROC=0.6842 | VAL PR=0.1501 | posw=8.00 | loss=focal
seed 3 | ep 08 | loss=0.2342 | VAL ROC=0.6909 | VAL PR=0.1549 | posw=8.00 | loss=focal
seed 3 | ep 09 | loss=0.2233 | VAL ROC=0.6880 | VAL PR=0.1562 | posw=8.00 | loss=focal
seed 3 | ep 10 | loss=0.2222 | VAL ROC=0.6843 | VAL PR=0.1413 | posw=8.00 | loss=focal
seed 3 | ep 11 | loss=0.2216 | VAL ROC=0.6833 | VAL PR=0.1502 | posw=8.00 | loss=focal
seed 3 | ep 12 | loss=0.2139 | VAL ROC=0.67

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


seed 4 | ep 01 | loss=0.2692 | VAL ROC=0.5857 | VAL PR=0.1422 | posw=8.00 | loss=focal
seed 4 | ep 02 | loss=0.2617 | VAL ROC=0.5626 | VAL PR=0.1374 | posw=8.00 | loss=focal
seed 4 | ep 03 | loss=0.2525 | VAL ROC=0.6557 | VAL PR=0.1448 | posw=8.00 | loss=focal
seed 4 | ep 04 | loss=0.2426 | VAL ROC=0.6882 | VAL PR=0.1516 | posw=8.00 | loss=focal
seed 4 | ep 05 | loss=0.2335 | VAL ROC=0.6720 | VAL PR=0.1458 | posw=8.00 | loss=focal
seed 4 | ep 06 | loss=0.2351 | VAL ROC=0.6850 | VAL PR=0.1472 | posw=8.00 | loss=focal
seed 4 | ep 07 | loss=0.2293 | VAL ROC=0.6907 | VAL PR=0.1491 | posw=8.00 | loss=focal
seed 4 | ep 08 | loss=0.2249 | VAL ROC=0.6991 | VAL PR=0.1539 | posw=8.00 | loss=focal
seed 4 | ep 09 | loss=0.2190 | VAL ROC=0.6782 | VAL PR=0.1468 | posw=8.00 | loss=focal
seed 4 | ep 10 | loss=0.2196 | VAL ROC=0.6695 | VAL PR=0.1508 | posw=8.00 | loss=focal
seed 4 | ep 11 | loss=0.2239 | VAL ROC=0.7016 | VAL PR=0.1377 | posw=8.00 | loss=focal
seed 4 | ep 12 | loss=0.2367 | VAL ROC=0.69

In [27]:
# ============================================================
# CATBOOST PR-AUC BOOSTER + CALIBRATION + REPEATED STRATIFIED CV
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, confusion_matrix, f1_score, precision_score, recall_score, balanced_accuracy_score
)
from sklearn.isotonic import IsotonicRegression

from catboost import CatBoostClassifier

# -------------------------
# 0) CONFIG
# -------------------------
DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"   # <-- change if your path differs
TARGET_COL = "stroke"
DROP_COLS = ["id"]  # keep if present; safe to ignore if missing

N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42

# If you want stronger search later, increase these modestly
CATBOOST_BASE_PARAMS = dict(
    loss_function="Logloss",
    eval_metric="PRAUC",   # PR-AUC optimization (CatBoost supports PRAUC)
    iterations=6000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=6.0,
    min_data_in_leaf=40,
    subsample=0.8,               # stochastic
    rsm=0.9,                     # feature subsample per split
    random_strength=1.0,
    bagging_temperature=0.5,     # helps generalization
    od_type="Iter",
    od_wait=200,                # early stop patience
    verbose=False
)

# -------------------------
# 1) LOAD DATA
# -------------------------
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

for c in DROP_COLS:
    if c in df.columns:
        df = df.drop(columns=[c])

assert TARGET_COL in df.columns, f"Target '{TARGET_COL}' not found. Columns={list(df.columns)}"
y = df[TARGET_COL].astype(int).values
X = df.drop(columns=[TARGET_COL]).copy()

print("Shape:", X.shape, " Pos rate:", y.mean().round(4))

# -------------------------
# 2) FEATURE ENGINEERING (safe + clinically plausible)
# -------------------------
def add_feature_engineering(X_in: pd.DataFrame) -> pd.DataFrame:
    X = X_in.copy()

    # Example columns that exist in common Kaggle stroke dataset
    # age, avg_glucose_level, bmi, hypertension, heart_disease
    if "bmi" in X.columns:
        X["bmi_missing"] = X["bmi"].isna().astype(int)

    if "avg_glucose_level" in X.columns:
        # log transform reduces skew; helps trees too
        X["glucose_log1p"] = np.log1p(X["avg_glucose_level"].astype(float))

    if "age" in X.columns:
        # 5-year bins as category-like signal
        # NOTE: keep as integer; CatBoost can treat as numeric; we also create a categorical bin feature.
        bins = np.arange(0, 105, 5)
        X["age_bin_5y"] = pd.cut(X["age"].astype(float), bins=bins, include_lowest=True).astype(str)

    # clinically meaningful interactions
    def safe_mul(a, b, name):
        if a in X.columns and b in X.columns:
            X[name] = X[a].astype(float) * X[b].astype(float)

    safe_mul("age", "hypertension", "age_x_hypertension")
    safe_mul("age", "heart_disease", "age_x_heart_disease")
    safe_mul("avg_glucose_level", "bmi", "glucose_x_bmi")

    # optional: BMI bucket as categorical (often helps)
    if "bmi" in X.columns:
        X["bmi_bin"] = pd.cut(
            X["bmi"].astype(float),
            bins=[0, 18.5, 25, 30, 35, 1000],
            labels=["under", "normal", "over", "obese1", "obese2"]
        ).astype(str)

    return X

X = add_feature_engineering(X)

# -------------------------
# 3) IDENTIFY CATEGORICAL FEATURES FOR CATBOOST
# -------------------------
def infer_cat_cols(X: pd.DataFrame, max_unique_int_as_cat=10):
    cat_cols = []
    for c in X.columns:
        if X[c].dtype == "object":
            cat_cols.append(c)
        elif str(X[c].dtype).startswith("category"):
            cat_cols.append(c)
        else:
            # treat low-cardinality integer-like columns as categorical (optional)
            # e.g., hypertension, heart_disease often 0/1
            if pd.api.types.is_integer_dtype(X[c]) and X[c].nunique(dropna=True) <= max_unique_int_as_cat:
                cat_cols.append(c)
    return cat_cols

cat_cols = infer_cat_cols(X)
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

print("Categorical columns:", cat_cols)

# -------------------------
# 4) BASIC IMPUTATION FOR CATBOOST
#    - Cat features: fill missing with string token
#    - Num features: fill missing with median
# -------------------------
def catboost_ready_dataframe(X: pd.DataFrame, cat_cols):
    X2 = X.copy()
    for c in X2.columns:
        if c in cat_cols:
            X2[c] = X2[c].astype("object").fillna("MISSING").astype(str)
        else:
            # numeric
            X2[c] = pd.to_numeric(X2[c], errors="coerce")
            med = X2[c].median()
            X2[c] = X2[c].fillna(med)
    return X2

X = catboost_ready_dataframe(X, cat_cols)

# -------------------------
# 5) ECE (Expected Calibration Error)
# -------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return ece

# -------------------------
# 6) THRESHOLD SELECTION (on calibration/valid only)
# -------------------------
def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    # precision_recall_curve returns thresholds of length len(prec)-1
    f1s = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
    j = int(np.nanargmax(f1s))
    return float(thr[j]), float(f1s[j])

# -------------------------
# 7) REPEATED STRATIFIED CV with leakage-safe calibration
#    Outer fold: evaluation
#    Inner split: early stopping & isotonic calibration fit
# -------------------------
def run_repeated_cv(X_df, y, params, cat_idx, n_splits=5, n_repeats=3, seed=42):
    rng = np.random.RandomState(seed)
    rows = []

    X_np = X_df.values

    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X_np, y), start=1):
            X_tr_full, y_tr_full = X_np[tr_idx], y[tr_idx]
            X_te, y_te = X_np[te_idx], y[te_idx]

            # inner split from training fold: train_sub vs calib_sub
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, calib_idx = next(sss.split(X_tr_full, y_tr_full))

            X_tr, y_tr = X_tr_full[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_cal, y_cal = X_tr_full[calib_idx], y_tr_full[calib_idx]

            # class weight scaling (helps PR-AUC)
            pos = max(1, int(y_tr.sum()))
            neg = max(1, int((1 - y_tr).sum()))
            scale_pos_weight = neg / pos

            model = CatBoostClassifier(
                **params,
                scale_pos_weight=scale_pos_weight,
                random_seed=int(rng.randint(0, 10_000_000))
            )

            # early stopping on calibration split (inner)
            model.fit(
                X_tr, y_tr,
                cat_features=cat_idx,
                eval_set=(X_cal, y_cal),
                use_best_model=True
            )

            # raw probabilities
            p_cal_raw = model.predict_proba(X_cal)[:, 1]
            p_te_raw  = model.predict_proba(X_te)[:, 1]

            # isotonic calibration fit ONLY on calibration split (leakage-safe)
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(p_cal_raw, y_cal)
            p_te = iso.transform(p_te_raw)

            # metrics on outer test fold
            pr = average_precision_score(y_te, p_te)
            roc = roc_auc_score(y_te, p_te)
            brier = brier_score_loss(y_te, p_te)
            ece = expected_calibration_error(y_te, p_te, n_bins=15)

            # threshold (picked on calibration set)
            thr, f1_cal = best_f1_threshold(y_cal, iso.transform(p_cal_raw))
            yhat = (p_te >= thr).astype(int)

            f1 = f1_score(y_te, yhat)
            prec = precision_score(y_te, yhat, zero_division=0)
            rec = recall_score(y_te, yhat, zero_division=0)
            balacc = balanced_accuracy_score(y_te, yhat)
            tn, fp, fn, tp = confusion_matrix(y_te, yhat).ravel()

            rows.append({
                "rep": rep + 1,
                "fold": fold,
                "pos_rate_test": float(y_te.mean()),
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "thr_from_cal": float(thr),
                "f1_test": float(f1),
                "precision_test": float(prec),
                "recall_test": float(rec),
                "balacc_test": float(balacc),
                "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                "scale_pos_weight": float(scale_pos_weight),
            })

            print(f"[rep {rep+1}/{n_repeats} fold {fold}/{n_splits}] "
                  f"PR-AUC={pr:.4f} ROC-AUC={roc:.4f} Brier={brier:.4f} ECE={ece:.4f} "
                  f"F1={f1:.4f} P={prec:.4f} R={rec:.4f}")

    return pd.DataFrame(rows)

results = run_repeated_cv(
    X, y,
    params=CATBOOST_BASE_PARAMS,
    cat_idx=cat_idx,
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    seed=RANDOM_STATE
)

# -------------------------
# 8) SUMMARY + CONFIDENCE INTERVALS (bootstrap)
# -------------------------
def bootstrap_ci(x, n_boot=5000, alpha=0.05, seed=123):
    x = np.asarray(x, dtype=float)
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boots.append(x[idx].mean())
    boots = np.sort(boots)
    lo = boots[int((alpha/2) * n_boot)]
    hi = boots[int((1-alpha/2) * n_boot)]
    return float(lo), float(hi)

print("\n====================")
print("OVERALL SUMMARY")
print("====================")
for k in ["pr_auc", "roc_auc", "brier", "ece", "f1_test", "precision_test", "recall_test", "balacc_test"]:
    m = results[k].mean()
    s = results[k].std(ddof=1)
    lo, hi = bootstrap_ci(results[k].values, n_boot=3000, seed=RANDOM_STATE)
    print(f"{k:>14s}: mean={m:.4f}  std={s:.4f}  95%CI(mean)=[{lo:.4f}, {hi:.4f}]")

print("\nTop 10 folds by PR-AUC:")
display(results.sort_values("pr_auc", ascending=False).head(10))

print("\nResults head:")
display(results.head())

Shape: (4603, 35)  Pos rate: 0.0786
Categorical columns: ['gender', 'age', 'Race', 'Marital status', 'alcohol', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'diabetes', 'hypertension', 'high cholesterol', 'Coronary Heart Disease', 'Body Mass Index', 'age_bin_5y']


/tmp/ipykernel_55/2106588905.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X2[c] = X2[c].astype("object").fillna("MISSING").astype(str)
/tmp/ipykernel_55/2106588905.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X2[c] = X2[c].astype("object").fillna("MISSING").astype(str)
/tmp/ipykernel_55/2106588905.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no

[rep 1/3 fold 1/5] PR-AUC=0.1676 ROC-AUC=0.7261 Brier=0.0684 ECE=0.0141 F1=0.2556 P=0.2130 R=0.3194
[rep 1/3 fold 2/5] PR-AUC=0.1513 ROC-AUC=0.7314 Brier=0.0711 ECE=0.0108 F1=0.2609 P=0.1654 R=0.6164
[rep 1/3 fold 3/5] PR-AUC=0.1302 ROC-AUC=0.6582 Brier=0.0724 ECE=0.0156 F1=0.1449 P=0.1538 R=0.1370
[rep 1/3 fold 4/5] PR-AUC=0.1101 ROC-AUC=0.6358 Brier=0.0796 ECE=0.0355 F1=0.1880 P=0.1358 R=0.3056
[rep 1/3 fold 5/5] PR-AUC=0.1468 ROC-AUC=0.7199 Brier=0.0706 ECE=0.0244 F1=0.1991 P=0.1477 R=0.3056
[rep 2/3 fold 1/5] PR-AUC=0.1607 ROC-AUC=0.7184 Brier=0.0702 ECE=0.0129 F1=0.2455 P=0.1824 R=0.3750
[rep 2/3 fold 2/5] PR-AUC=0.1525 ROC-AUC=0.6958 Brier=0.0722 ECE=0.0125 F1=0.2424 P=0.1675 R=0.4384
[rep 2/3 fold 3/5] PR-AUC=0.1252 ROC-AUC=0.6429 Brier=0.0735 ECE=0.0226 F1=0.1538 P=0.1571 R=0.1507
[rep 2/3 fold 4/5] PR-AUC=0.1169 ROC-AUC=0.6368 Brier=0.0754 ECE=0.0318 F1=0.1707 P=0.1207 R=0.2917
[rep 2/3 fold 5/5] PR-AUC=0.1332 ROC-AUC=0.6912 Brier=0.0713 ECE=0.0288 F1=0.2026 P=0.1484 R=0.3194


,rep,fold,pos_rate_test,pr_auc,roc_auc,brier,ece,thr_from_cal,f1_test,precision_test,recall_test,balacc_test,tn,fp,fn,tp,scale_pos_weight
0,1,1,0.078176,0.167555,0.726083,0.068355,0.014056,0.203125,0.255556,0.212963,0.319444,0.609663,764,85,49,23,11.693966
5,2,1,0.078176,0.160738,0.718419,0.070235,0.012859,0.153846,0.245455,0.182432,0.375000,0.616240,728,121,45,27,11.693966
11,3,2,0.079262,0.153682,0.712401,0.070073,0.010961,0.196721,0.244344,0.182432,0.369863,0.613587,727,121,46,27,11.748918
6,2,2,0.079262,0.152481,0.695835,0.072197,0.012505,0.161290,0.242424,0.167539,0.438356,0.625428,689,159,41,32,11.748918
1,1,2,0.079262,0.151266,0.731447,0.071064,0.010768,0.120000,0.260870,0.165441,0.616438,0.674375,621,227,28,45,11.748918
4,1,5,0.078261,0.146843,0.719880,0.070637,0.024372,0.147059,0.199095,0.147651,0.305556,0.577896,721,127,50,22,11.698276
12,3,3,0.079262,0.138908,0.680279,0.073369,0.023755,0.184211,0.201005,0.158730,0.273973,0.574486,742,106,53,20,11.748918
10,3,1,0.078176,0.136712,0.704432,0.072895,0.025173,0.428571,0.082474,0.160000,0.055556,0.515410,828,21,68,4,11.693966
9,2,5,0.078261,0.133209,0.691161,0.071276,0.028830,0.166667,0.202643,0.148387,0.319444,0.581892,716,132,49,23,11.698276
13,3,4,0.078261,0.132298,0.652344,0.070583,0.020742,0.234043,0.200000,0.192308,0.208333,0.567020,785,63,57,15,11.698276



Results head:


,rep,fold,pos_rate_test,pr_auc,roc_auc,brier,ece,thr_from_cal,f1_test,precision_test,recall_test,balacc_test,tn,fp,fn,tp,scale_pos_weight
0,1,1,0.078176,0.167555,0.726083,0.068355,0.014056,0.203125,0.255556,0.212963,0.319444,0.609663,764,85,49,23,11.693966
1,1,2,0.079262,0.151266,0.731447,0.071064,0.010768,0.120000,0.260870,0.165441,0.616438,0.674375,621,227,28,45,11.748918
2,1,3,0.079262,0.130174,0.658221,0.072414,0.015583,0.185185,0.144928,0.153846,0.136986,0.536064,793,55,63,10,11.748918
3,1,4,0.078261,0.110058,0.635777,0.079620,0.035456,0.166667,0.188034,0.135802,0.305556,0.570231,708,140,50,22,11.698276
4,1,5,0.078261,0.146843,0.719880,0.070637,0.024372,0.147059,0.199095,0.147651,0.305556,0.577896,721,127,50,22,11.698276


In [28]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

from catboost import CatBoostClassifier

# -------------------------
# 0) HELPERS
# -------------------------
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def bootstrap_ci_mean(x, n_boot=2000, alpha=0.05, seed=0):
    x = np.asarray(x, dtype=float)
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boots.append(x[idx].mean())
    boots = np.sort(boots)
    lo = boots[int((alpha/2) * n_boot)]
    hi = boots[int((1-alpha/2) * n_boot)]
    return float(lo), float(hi)

# -------------------------
# 1) IMPORTANT:
#   Use YOUR prepared X, y, cat_idx from earlier cell.
#   (This code assumes X is a pandas DataFrame already imputed for CatBoost
#    and categorical columns converted to strings with "MISSING".)
# -------------------------
# X: pd.DataFrame
# y: np.array shape (n,)
# cat_idx: list[int] indices of categorical columns

# -------------------------
# 2) CV EVALUATION (RAW PR-AUC primary)
# -------------------------
def eval_params_cv(X_df, y, cat_idx, params, n_splits=5, n_repeats=3, seed=42):
    rng = np.random.RandomState(seed)
    X_np = X_df.values
    rows = []

    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=int(rng.randint(0, 10_000_000)))
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X_np, y), start=1):
            X_tr_full, y_tr_full = X_np[tr_idx], y[tr_idx]
            X_te, y_te = X_np[te_idx], y[te_idx]

            # inner split for early stopping + (optional) calibration
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20,
                                         random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, cal_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_cal, y_cal = X_tr_full[cal_idx], y_tr_full[cal_idx]

            pos = max(1, int(y_tr.sum()))
            neg = max(1, int((1 - y_tr).sum()))
            spw = neg / pos

            model = CatBoostClassifier(
                **params,
                scale_pos_weight=spw,
                random_seed=int(rng.randint(0, 10_000_000)),
                verbose=False
            )

            model.fit(
                X_tr, y_tr,
                cat_features=cat_idx,
                eval_set=(X_cal, y_cal),
                use_best_model=True
            )

            # RAW probabilities (optimize PR-AUC with these)
            p_te_raw = model.predict_proba(X_te)[:, 1]
            pr_raw = average_precision_score(y_te, p_te_raw)
            roc_raw = roc_auc_score(y_te, p_te_raw)

            # Calibration (for trustworthiness reporting, not for maximizing PR)
            iso = IsotonicRegression(out_of_bounds="clip")
            p_cal_raw = model.predict_proba(X_cal)[:, 1]
            iso.fit(p_cal_raw, y_cal)
            p_te_cal = iso.transform(p_te_raw)

            pr_cal = average_precision_score(y_te, p_te_cal)
            roc_cal = roc_auc_score(y_te, p_te_cal)
            brier = brier_score_loss(y_te, p_te_cal)
            ece = expected_calibration_error(y_te, p_te_cal, n_bins=15)

            rows.append({
                "rep": rep+1, "fold": fold,
                "pr_raw": float(pr_raw), "roc_raw": float(roc_raw),
                "pr_cal": float(pr_cal), "roc_cal": float(roc_cal),
                "brier": float(brier), "ece": float(ece),
                "spw": float(spw)
            })

    df = pd.DataFrame(rows)
    out = {
        "pr_raw_mean": df["pr_raw"].mean(),
        "pr_raw_std": df["pr_raw"].std(ddof=1),
        "pr_raw_ci": bootstrap_ci_mean(df["pr_raw"].values, seed=seed),
        "pr_cal_mean": df["pr_cal"].mean(),
        "pr_cal_std": df["pr_cal"].std(ddof=1),
        "pr_cal_ci": bootstrap_ci_mean(df["pr_cal"].values, seed=seed+1),
        "roc_raw_mean": df["roc_raw"].mean(),
        "roc_cal_mean": df["roc_cal"].mean(),
        "brier_mean": df["brier"].mean(),
        "ece_mean": df["ece"].mean(),
        "detail": df
    }
    return out

# -------------------------
# 3) RANDOM SEARCH SPACE (high-impact knobs)
# -------------------------
def sample_params(rng):
    # Key levers for CatBoost on imbalanced tabular:
    depth = int(rng.choice([4,5,6,7,8,9,10]))
    l2_leaf_reg = float(rng.choice([1,3,5,7,9,12,15,20]))
    min_data_in_leaf = int(rng.choice([10,20,30,40,60,80,120]))
    bagging_temperature = float(rng.choice([0.0,0.2,0.5,0.8,1.0,2.0,5.0]))
    random_strength = float(rng.choice([0.2,0.5,1.0,2.0,5.0,10.0]))
    border_count = int(rng.choice([64,128,254]))
    rsm = float(rng.choice([0.7,0.8,0.9,1.0]))
    subsample = float(rng.choice([0.7,0.8,0.9,1.0]))
    grow_policy = str(rng.choice(["SymmetricTree", "Depthwise", "Lossguide"]))

    return dict(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=8000,
        learning_rate=float(rng.choice([0.01, 0.02, 0.03, 0.05])),
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        min_data_in_leaf=min_data_in_leaf,
        subsample=subsample,
        rsm=rsm,
        border_count=border_count,
        random_strength=random_strength,
        bagging_temperature=bagging_temperature,
        grow_policy=grow_policy,
        od_type="Iter",
        od_wait=250
    )

def random_search_catboost(X, y, cat_idx, n_trials=15, seed=42, n_splits=5, n_repeats=3):
    rng = np.random.RandomState(seed)
    leaderboard = []

    for t in range(1, n_trials+1):
        params = sample_params(rng)
        out = eval_params_cv(X, y, cat_idx, params, n_splits=n_splits, n_repeats=n_repeats, seed=int(rng.randint(0, 10_000_000)))

        leaderboard.append({
            "trial": t,
            "pr_raw_mean": out["pr_raw_mean"],
            "pr_raw_std": out["pr_raw_std"],
            "pr_raw_ci_lo": out["pr_raw_ci"][0],
            "pr_raw_ci_hi": out["pr_raw_ci"][1],
            "pr_cal_mean": out["pr_cal_mean"],
            "roc_raw_mean": out["roc_raw_mean"],
            "brier_mean": out["brier_mean"],
            "ece_mean": out["ece_mean"],
            "params": params
        })

        print(f"Trial {t:02d}/{n_trials} | PR_RAW={out['pr_raw_mean']:.4f} ±{out['pr_raw_std']:.4f} "
              f"(CI {out['pr_raw_ci'][0]:.4f}-{out['pr_raw_ci'][1]:.4f}) | "
              f"PR_CAL={out['pr_cal_mean']:.4f} | ROC_RAW={out['roc_raw_mean']:.4f} | "
              f"Brier={out['brier_mean']:.4f} ECE={out['ece_mean']:.4f}")

    lb = pd.DataFrame(leaderboard).sort_values("pr_raw_mean", ascending=False).reset_index(drop=True)
    return lb

# -------------------------
# 4) RUN SEARCH
# -------------------------
lb = random_search_catboost(X, y, cat_idx, n_trials=15, seed=42, n_splits=5, n_repeats=2)
display(lb[["trial","pr_raw_mean","pr_raw_std","pr_raw_ci_lo","pr_raw_ci_hi","pr_cal_mean","roc_raw_mean","brier_mean","ece_mean"]].head(10))

best = lb.iloc[0]["params"]
print("\nBEST PARAMS (by PR_RAW mean):")
print(best)

Trial 01/15 | PR_RAW=0.1466 ±0.0228 (CI 0.1340-0.1602) | PR_CAL=0.1281 | ROC_RAW=0.6762 | Brier=0.0716 ECE=0.0169
Trial 02/15 | PR_RAW=0.1530 ±0.0207 (CI 0.1400-0.1649) | PR_CAL=0.1372 | ROC_RAW=0.6890 | Brier=0.0725 ECE=0.0202
Trial 03/15 | PR_RAW=0.1624 ±0.0249 (CI 0.1480-0.1767) | PR_CAL=0.1453 | ROC_RAW=0.6961 | Brier=0.0715 ECE=0.0152
Trial 04/15 | PR_RAW=0.1493 ±0.0244 (CI 0.1349-0.1628) | PR_CAL=0.1346 | ROC_RAW=0.6739 | Brier=0.0722 ECE=0.0194
Trial 05/15 | PR_RAW=0.1660 ±0.0171 (CI 0.1569-0.1763) | PR_CAL=0.1466 | ROC_RAW=0.7075 | Brier=0.0708 ECE=0.0170
Trial 06/15 | PR_RAW=0.1428 ±0.0276 (CI 0.1264-0.1591) | PR_CAL=0.1285 | ROC_RAW=0.6725 | Brier=0.0737 ECE=0.0252
Trial 07/15 | PR_RAW=0.1464 ±0.0216 (CI 0.1336-0.1582) | PR_CAL=0.1319 | ROC_RAW=0.6795 | Brier=0.0715 ECE=0.0160
Trial 08/15 | PR_RAW=0.1573 ±0.0248 (CI 0.1424-0.1726) | PR_CAL=0.1353 | ROC_RAW=0.6835 | Brier=0.0719 ECE=0.0178
Trial 09/15 | PR_RAW=0.1374 ±0.0290 (CI 0.1197-0.1548) | PR_CAL=0.1250 | ROC_RAW=0.6491 

,trial,pr_raw_mean,pr_raw_std,pr_raw_ci_lo,pr_raw_ci_hi,pr_cal_mean,roc_raw_mean,brier_mean,ece_mean
0,11,0.166621,0.017693,0.156264,0.176403,0.143140,0.698919,0.072535,0.017637
1,5,0.166020,0.017145,0.156950,0.176277,0.146586,0.707491,0.070842,0.017003
2,3,0.162374,0.024891,0.148036,0.176715,0.145344,0.696055,0.071469,0.015221
3,12,0.159452,0.016994,0.149370,0.168662,0.142646,0.695048,0.072504,0.022234
4,8,0.157298,0.024850,0.142433,0.172571,0.135261,0.683469,0.071865,0.017825
5,2,0.152968,0.020738,0.139988,0.164898,0.137158,0.689011,0.072489,0.020230
6,4,0.149337,0.024429,0.134857,0.162776,0.134554,0.673865,0.072230,0.019437
7,13,0.147565,0.029033,0.131185,0.165166,0.129728,0.680408,0.071159,0.016569
8,1,0.146567,0.022800,0.134027,0.160220,0.128097,0.676228,0.071644,0.016930
9,7,0.146430,0.021582,0.133622,0.158212,0.131888,0.679522,0.071481,0.015988



BEST PARAMS (by PR_RAW mean):
{'loss_function': 'Logloss', 'eval_metric': 'PRAUC', 'iterations': 8000, 'learning_rate': 0.01, 'depth': 5, 'l2_leaf_reg': 1.0, 'min_data_in_leaf': 40, 'subsample': 0.7, 'rsm': 0.7, 'border_count': 64, 'random_strength': 2.0, 'bagging_temperature': 0.8, 'grow_policy': 'Lossguide', 'od_type': 'Iter', 'od_wait': 250}


In [29]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

# assumes X (pd.DataFrame), y (np.array), cat_idx exist from your earlier preprocessing
X_np = X.values
y_np = y

def eval_params_cv_raw_pr(X_np, y_np, cat_idx, params, n_splits=5, n_repeats=2, seed=42):
    rng = np.random.RandomState(seed)
    prs = []
    rocs = []
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=int(rng.randint(0, 10_000_000)))
        for tr_idx, te_idx in skf.split(X_np, y_np):
            X_tr_full, y_tr_full = X_np[tr_idx], y_np[tr_idx]
            X_te, y_te = X_np[te_idx], y_np[te_idx]

            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20,
                                         random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, cal_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_cal, y_cal = X_tr_full[cal_idx], y_tr_full[cal_idx]

            pos = max(1, int(y_tr.sum()))
            neg = max(1, int((1 - y_tr).sum()))
            spw = neg / pos

            model = CatBoostClassifier(
                **params,
                scale_pos_weight=spw,
                random_seed=int(rng.randint(0, 10_000_000)),
                verbose=False
            )
            model.fit(X_tr, y_tr, cat_features=cat_idx, eval_set=(X_cal, y_cal), use_best_model=True)

            p_te = model.predict_proba(X_te)[:, 1]
            prs.append(average_precision_score(y_te, p_te))
            rocs.append(roc_auc_score(y_te, p_te))
    return float(np.mean(prs)), float(np.std(prs, ddof=1)), float(np.mean(rocs))

best_params = {'loss_function': 'Logloss', 'eval_metric': 'PRAUC', 'iterations': 12000, 'learning_rate': 0.01,
               'depth': 5, 'l2_leaf_reg': 1.0, 'min_data_in_leaf': 40, 'subsample': 0.7, 'rsm': 0.7,
               'border_count': 64, 'random_strength': 2.0, 'bagging_temperature': 0.8,
               'grow_policy': 'Lossguide', 'od_type': 'Iter', 'od_wait': 300}

# local grid around best pocket
depths = [4,5,6]
l2s = [1.0, 3.0, 5.0, 7.0]
mins = [20, 40, 60, 80]
subs = [0.7, 0.8, 0.9]
rsms = [0.7, 0.8, 0.9]
bags = [0.2, 0.8, 2.0]
borders = [64, 128, 254]
rstr = [1.0, 2.0, 5.0]

cands = []
rng = np.random.RandomState(0)

# sample ~25 candidates from this local space (fast)
for _ in range(25):
    p = dict(best_params)
    p["depth"] = int(rng.choice(depths))
    p["l2_leaf_reg"] = float(rng.choice(l2s))
    p["min_data_in_leaf"] = int(rng.choice(mins))
    p["subsample"] = float(rng.choice(subs))
    p["rsm"] = float(rng.choice(rsms))
    p["bagging_temperature"] = float(rng.choice(bags))
    p["border_count"] = int(rng.choice(borders))
    p["random_strength"] = float(rng.choice(rstr))
    cands.append(p)

rows = []
for i, p in enumerate(cands, 1):
    pr_mean, pr_std, roc_mean = eval_params_cv_raw_pr(X_np, y_np, cat_idx, p, n_splits=5, n_repeats=2, seed=42+i)
    rows.append({"i": i, "pr_mean": pr_mean, "pr_std": pr_std, "roc_mean": roc_mean, "params": p})
    print(f"[{i:02d}/25] PR={pr_mean:.4f} ±{pr_std:.4f} | ROC={roc_mean:.4f} | depth={p['depth']} "
          f"l2={p['l2_leaf_reg']} minleaf={p['min_data_in_leaf']} subs={p['subsample']} rsm={p['rsm']} "
          f"bagT={p['bagging_temperature']} border={p['border_count']} rs={p['random_strength']}")

lb_local = pd.DataFrame(rows).sort_values("pr_mean", ascending=False).reset_index(drop=True)
display(lb_local.head(10)[["i","pr_mean","pr_std","roc_mean"]])
best_local = lb_local.iloc[0]["params"]
print("\nBEST LOCAL PARAMS:")
print(best_local)

[01/25] PR=0.1524 ±0.0225 | ROC=0.6834 | depth=4 l2=7.0 minleaf=40 subs=0.7 rsm=0.8 bagT=0.8 border=254 rs=1.0
[02/25] PR=0.1416 ±0.0161 | ROC=0.6786 | depth=6 l2=1.0 minleaf=20 subs=0.7 rsm=0.9 bagT=0.8 border=254 rs=5.0
[03/25] PR=0.1613 ±0.0259 | ROC=0.6937 | depth=4 l2=3.0 minleaf=40 subs=0.8 rsm=0.8 bagT=0.2 border=128 rs=1.0
[04/25] PR=0.1638 ±0.0212 | ROC=0.6971 | depth=4 l2=7.0 minleaf=40 subs=0.9 rsm=0.7 bagT=2.0 border=64 rs=2.0
[05/25] PR=0.1539 ±0.0306 | ROC=0.6939 | depth=5 l2=7.0 minleaf=80 subs=0.9 rsm=0.7 bagT=0.8 border=128 rs=2.0
[06/25] PR=0.1472 ±0.0278 | ROC=0.6770 | depth=4 l2=7.0 minleaf=60 subs=0.7 rsm=0.9 bagT=2.0 border=64 rs=5.0
[07/25] PR=0.1491 ±0.0172 | ROC=0.6797 | depth=4 l2=1.0 minleaf=20 subs=0.8 rsm=0.8 bagT=2.0 border=64 rs=1.0
[08/25] PR=0.1601 ±0.0213 | ROC=0.6942 | depth=5 l2=7.0 minleaf=20 subs=0.8 rsm=0.9 bagT=2.0 border=64 rs=2.0
[09/25] PR=0.1559 ±0.0231 | ROC=0.6874 | depth=5 l2=7.0 minleaf=40 subs=0.8 rsm=0.9 bagT=2.0 border=254 rs=1.0
[10/2

,i,pr_mean,pr_std,roc_mean
0,16,0.171731,0.030316,0.693027
1,4,0.163802,0.021208,0.697150
2,20,0.162243,0.026519,0.695640
3,3,0.161255,0.025945,0.693724
4,8,0.160131,0.021251,0.694150
5,22,0.157133,0.028301,0.696788
6,25,0.156776,0.015085,0.683819
7,24,0.156480,0.021070,0.695924
8,9,0.155942,0.023057,0.687395
9,17,0.155717,0.020012,0.691476



BEST LOCAL PARAMS:
{'loss_function': 'Logloss', 'eval_metric': 'PRAUC', 'iterations': 12000, 'learning_rate': 0.01, 'depth': 6, 'l2_leaf_reg': 3.0, 'min_data_in_leaf': 60, 'subsample': 0.9, 'rsm': 0.9, 'border_count': 254, 'random_strength': 1.0, 'bagging_temperature': 2.0, 'grow_policy': 'Lossguide', 'od_type': 'Iter', 'od_wait': 300}


In [30]:
# ============================================================
# OUTER CV: CatBoost + LightGBM STACKING (leakage-safe)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
import lightgbm as lgb

# -------------------------
# Assumes you already have:
# X : pd.DataFrame (CatBoost-ready: cat cols as str; nums imputed)
# y : np.array (0/1)
# cat_idx : list of categorical indices
# -------------------------

X_df = X.copy()
y_np = y.astype(int)
cat_cols = [X_df.columns[i] for i in cat_idx]

BEST_LOCAL_PARAMS = {
    'loss_function': 'Logloss',
    'eval_metric': 'PRAUC',
    'iterations': 12000,
    'learning_rate': 0.01,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'min_data_in_leaf': 60,
    'subsample': 0.9,
    'rsm': 0.9,
    'border_count': 254,
    'random_strength': 1.0,
    'bagging_temperature': 2.0,
    'grow_policy': 'Lossguide',
    'od_type': 'Iter',
    'od_wait': 300
}

def make_lgb_data(df):
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].astype("category")
    return out

def lgb_params(seed):
    # strong baseline for imbalanced tabular
    return dict(
        objective="binary",
        learning_rate=0.03,
        n_estimators=8000,
        num_leaves=128,
        min_data_in_leaf=40,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=1,
        lambda_l2=5.0,
        max_depth=-1,
        random_state=seed
    )

def fit_catboost(X_tr, y_tr, X_va, y_va, spw, seed):
    model = CatBoostClassifier(
        **BEST_LOCAL_PARAMS,
        scale_pos_weight=spw,
        random_seed=seed,
        verbose=False
    )
    model.fit(
        X_tr.values, y_tr,
        cat_features=cat_idx,
        eval_set=(X_va.values, y_va),
        use_best_model=True
    )
    return model

def fit_lgbm(X_tr, y_tr, X_va, y_va, spw, seed):
    # LightGBM imbalance handling: scale_pos_weight
    model = lgb.LGBMClassifier(**lgb_params(seed))
    model.set_params(scale_pos_weight=spw)

    X_tr_l = make_lgb_data(X_tr)
    X_va_l = make_lgb_data(X_va)

    model.fit(
        X_tr_l, y_tr,
        eval_set=[(X_va_l, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=300, verbose=False)]
    )
    return model

def outer_cv_stacking(X_df, y_np, n_splits=5, n_repeats=3, seed=42):
    rng = np.random.RandomState(seed)
    rows = []

    for rep in range(1, n_repeats + 1):
        skf = StratifiedKFold(
            n_splits=n_splits, shuffle=True,
            random_state=int(rng.randint(0, 10_000_000))
        )

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X_df, y_np), start=1):
            X_tr_full, y_tr_full = X_df.iloc[tr_idx], y_np[tr_idx]
            X_te, y_te = X_df.iloc[te_idx], y_np[te_idx]

            # Inner split from training fold: train_sub vs val_sub (for early stop + meta)
            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=0.25,
                random_state=int(rng.randint(0, 10_000_000))
            )
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            pos = max(1, int(y_tr.sum()))
            neg = max(1, int((1 - y_tr).sum()))
            spw = neg / pos

            # Fit base models
            cb = fit_catboost(X_tr, y_tr, X_va, y_va, spw, seed=int(rng.randint(0, 10_000_000)))
            lgbm = fit_lgbm(X_tr, y_tr, X_va, y_va, spw, seed=int(rng.randint(0, 10_000_000)))

            # Meta features built ONLY from validation predictions (no leakage)
            p_va_cb = cb.predict_proba(X_va.values)[:, 1]
            p_va_lg = lgbm.predict_proba(make_lgb_data(X_va))[:, 1]
            Z_va = np.vstack([p_va_cb, p_va_lg]).T

            # Train meta-model on validation split
            meta = LogisticRegression(max_iter=2000, class_weight="balanced")
            meta.fit(Z_va, y_va)

            # Predict on test fold
            p_te_cb = cb.predict_proba(X_te.values)[:, 1]
            p_te_lg = lgbm.predict_proba(make_lgb_data(X_te))[:, 1]
            Z_te = np.vstack([p_te_cb, p_te_lg]).T

            p_te_stack = meta.predict_proba(Z_te)[:, 1]

            # Metrics
            pr_cb = average_precision_score(y_te, p_te_cb)
            pr_lg = average_precision_score(y_te, p_te_lg)
            pr_st = average_precision_score(y_te, p_te_stack)

            roc_cb = roc_auc_score(y_te, p_te_cb)
            roc_lg = roc_auc_score(y_te, p_te_lg)
            roc_st = roc_auc_score(y_te, p_te_stack)

            rows.append({
                "rep": rep, "fold": fold,
                "pr_cb": float(pr_cb), "pr_lgb": float(pr_lg), "pr_stack": float(pr_st),
                "roc_cb": float(roc_cb), "roc_lgb": float(roc_lg), "roc_stack": float(roc_st),
                "spw": float(spw)
            })

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"PR: CB={pr_cb:.4f} LGB={pr_lg:.4f} STACK={pr_st:.4f} | "
                  f"ROC: CB={roc_cb:.4f} LGB={roc_lg:.4f} STACK={roc_st:.4f}")

    res = pd.DataFrame(rows)

    def summarize(col):
        m = res[col].mean()
        s = res[col].std(ddof=1)
        return m, s

    print("\n====================")
    print("OVERALL MEAN ± STD")
    print("====================")
    for c in ["pr_cb", "pr_lgb", "pr_stack", "roc_cb", "roc_lgb", "roc_stack"]:
        m, s = summarize(c)
        print(f"{c:>10s}: {m:.4f} ± {s:.4f}")

    display(res.sort_values("pr_stack", ascending=False).head(10))
    return res

# Run it
res_stack = outer_cv_stacking(X_df, y_np, n_splits=5, n_repeats=2, seed=42)

[LightGBM] [Warning] min_data_in_leaf is set=40, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=40
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.85
[LightGBM] [Warning] lambda_l2 is set=5.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.0
[LightGBM] [Warning] bagging_fraction is set=0.85, subsample=1.0 will be ignored. Current value: bagging_fraction=0.85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=40, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=40
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.85, colsample_bytree=1.0 will be ignored. Current value: feature

,rep,fold,pr_cb,pr_lgb,pr_stack,roc_cb,roc_lgb,roc_stack,spw
6,2,2,0.197287,0.137376,0.197167,0.736802,0.661839,0.737190,11.723502
1,1,2,0.171907,0.117755,0.171906,0.737448,0.647107,0.737416,11.723502
2,1,3,0.167259,0.111948,0.167190,0.698711,0.630743,0.698582,11.723502
5,2,1,0.166254,0.107097,0.166323,0.725903,0.631691,0.725952,11.723502
9,2,5,0.159338,0.105333,0.156685,0.660410,0.596452,0.660066,11.728111
7,2,3,0.154172,0.115946,0.153221,0.710681,0.642244,0.710730,11.723502
8,2,4,0.138828,0.109027,0.138802,0.667076,0.614665,0.667109,11.728111
0,1,1,0.118061,0.118203,0.132696,0.608993,0.663076,0.672940,11.723502
4,1,5,0.130020,0.119404,0.130088,0.703059,0.666110,0.703191,11.728111
3,1,4,0.115680,0.103686,0.115153,0.647586,0.618301,0.647291,11.728111


In [32]:
# ============================================================
# FIXED: Tabular Transformer (FT-Transformer-like)
# + Balanced batches + Hybrid loss + MC Dropout
# + Proper fold preprocessing (fit on train, transform val/test)
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -------------------------
# 0) Expect X (pd.DataFrame) and y (np.array) already exist
# -------------------------
assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
y = np.asarray(y).astype(int)
assert len(X) == len(y)

# -------------------------
# 1) Identify categorical/numerical columns
# -------------------------
def infer_cat_cols(df: pd.DataFrame, max_unique_int_as_cat=10):
    cat_cols = []
    for c in df.columns:
        if df[c].dtype == "object":
            cat_cols.append(c)
        elif str(df[c].dtype).startswith("category"):
            cat_cols.append(c)
        else:
            if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique(dropna=True) <= max_unique_int_as_cat:
                cat_cols.append(c)
    return cat_cols

cat_cols = infer_cat_cols(X)
num_cols = [c for c in X.columns if c not in cat_cols]

print("n_cat:", len(cat_cols), "n_num:", len(num_cols))

# -------------------------
# 2) Fold preprocessor (fit on train, transform on val/test)
# -------------------------
class FoldPreprocessor:
    def __init__(self, cat_cols, num_cols):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.cat_maps = {}
        self.num_stats = {}
        self.cat_sizes = []

    def fit(self, X_tr: pd.DataFrame):
        Xtr = X_tr.copy()

        # categorical maps from train only
        self.cat_maps = {}
        self.cat_sizes = []
        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mp = {k: i+1 for i, k in enumerate(uniq)}  # 0 reserved for UNK
            self.cat_maps[c] = mp
            self.cat_sizes.append(len(mp) + 1)  # +1 for UNK=0

        # numeric stats from train only
        self.num_stats = {}
        for c in self.num_cols:
            s = pd.to_numeric(Xtr[c], errors="coerce")
            med = float(s.median())
            s = s.fillna(med).astype(np.float32)
            mu = float(s.mean())
            sd = float(s.std(ddof=0) + 1e-6)
            self.num_stats[c] = (med, mu, sd)

        return self

    def transform(self, X_any: pd.DataFrame):
        Xa = X_any.copy()

        # categorical -> int64 codes
        Xcat = None
        if len(self.cat_cols) > 0:
            cat_arr = []
            for c in self.cat_cols:
                s = Xa[c].astype("object").fillna("MISSING").astype(str)
                mp = self.cat_maps[c]
                codes = s.map(mp).fillna(0).astype(np.int64).values
                cat_arr.append(codes)
            Xcat = np.stack(cat_arr, axis=1)  # (N, n_cat)

        # numerical -> float32 standardized
        Xnum = None
        if len(self.num_cols) > 0:
            num_arr = []
            for c in self.num_cols:
                s = pd.to_numeric(Xa[c], errors="coerce")
                med, mu, sd = self.num_stats[c]
                s = s.fillna(med).astype(np.float32)
                s = (s - mu) / sd
                num_arr.append(s.values.astype(np.float32))
            Xnum = np.stack(num_arr, axis=1)  # (N, n_num)

        return Xcat, Xnum

# -------------------------
# 3) Dataset + Balanced Batch Sampler
# -------------------------
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = np.asarray(y).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        y = torch.tensor(self.y[i], dtype=torch.float32)
        x_cat = torch.tensor(self.X_cat[i], dtype=torch.long) if self.X_cat is not None else None
        x_num = torch.tensor(self.X_num[i], dtype=torch.float32) if self.X_num is not None else None
        return x_cat, x_num, y

class BalancedBatchSampler(Sampler):
    """
    Yield indices so each batch contains a fixed fraction of positives.
    """
    def __init__(self, y, batch_size=256, pos_frac=0.30, seed=0):
        self.y = np.asarray(y).astype(int)
        self.batch_size = int(batch_size)
        self.pos_bs = max(1, int(self.batch_size * pos_frac))
        self.neg_bs = self.batch_size - self.pos_bs
        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]
        self.rng = np.random.RandomState(seed)

    def __iter__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        for _ in range(n_batches):
            pos = self.rng.choice(self.pos_idx, size=self.pos_bs, replace=True)
            neg = self.rng.choice(self.neg_idx, size=self.neg_bs, replace=True)
            idx = np.concatenate([pos, neg])
            self.rng.shuffle(idx)
            yield from idx.tolist()

    def __len__(self):
        return int(np.ceil(len(self.y) / self.batch_size)) * self.batch_size

# -------------------------
# 4) Model: FT-Transformer-like
# -------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4*d_model, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + self.drop(h))
        h = self.ff(x)
        x = self.ln2(x + self.drop(h))
        return x

class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.20):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = int(n_num)
        self.d_model = int(d_model)

        self.cat_embeds = nn.ModuleList([nn.Embedding(sz, d_model) for sz in cat_sizes]) if self.n_cat > 0 else None
        self.num_proj = nn.Linear(1, d_model) if self.n_num > 0 else None

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x_cat, x_num):
        tokens = []

        if self.n_cat > 0:
            for j, emb in enumerate(self.cat_embeds):
                tokens.append(emb(x_cat[:, j]))  # (B,D)

        if self.n_num > 0:
            for j in range(self.n_num):
                v = x_num[:, j:j+1]            # (B,1)
                tokens.append(self.num_proj(v)) # (B,D)

        # (B, T, D)
        x = torch.stack(tokens, dim=1) if len(tokens) else torch.zeros((x_cat.size(0), 0, self.d_model), device=x_cat.device)
        B = x.shape[0]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, 1+T, D)

        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)

        z = x[:, 0, :]
        logit = self.head(z).squeeze(-1)
        return logit

# -------------------------
# 5) Loss: BCE + Focal
# -------------------------
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        pt = torch.where(y == 1, p, 1 - p)
        w = (1 - pt).pow(self.gamma)
        a = torch.where(y == 1, torch.tensor(self.alpha, device=y.device), torch.tensor(1 - self.alpha, device=y.device))
        return (w * a * bce).mean()

def train_model(Xtr_cat, Xtr_num, ytr, Xva_cat, Xva_num, yva, cat_sizes,
                d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30, seed=0):

    torch.manual_seed(seed); np.random.seed(seed)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, ytr)
    ds_va = TabDataset(Xva_cat, Xva_num, yva)

    sampler = BalancedBatchSampler(ytr, batch_size=batch_size, pos_frac=pos_frac, seed=seed)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=2048, shuffle=False, num_workers=0)

    model = TabTransformer(cat_sizes, n_num=(0 if Xtr_num is None else Xtr_num.shape[1]),
                           d_model=d_model, n_heads=n_heads, n_layers=n_layers, dropout=dropout).to(DEVICE)

    focal = FocalBCE(gamma=2.0, alpha=0.25)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_pr = -1.0
    best_state = None
    patience = 5
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        for x_cat, x_num, yb in dl_tr:
            yb = yb.to(DEVICE)
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None

            opt.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)

            loss_bce = F.binary_cross_entropy_with_logits(logits, yb)
            loss_focal = focal(logits, yb)
            loss = 0.7 * loss_bce + 0.3 * loss_focal

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # validation PR-AUC
        model.eval()
        p_list, y_list = [], []
        with torch.no_grad():
            for x_cat, x_num, yb in dl_va:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                logits = model(x_cat, x_num).detach().cpu().numpy()
                p = 1 / (1 + np.exp(-logits))
                p_list.append(p)
                y_list.append(yb.numpy())

        pva = np.concatenate(p_list)
        yva_np = np.concatenate(y_list).astype(int)
        pr = average_precision_score(yva_np, pva)

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def predict_proba_mc(model, Xcat, Xnum, mc_samples=30, use_mc=True):
    ds = TabDataset(Xcat, Xnum, np.zeros(len(Xcat) if Xcat is not None else len(Xnum), dtype=np.float32))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, num_workers=0)

    def one_pass(train_mode=False):
        model.train() if train_mode else model.eval()
        ps = []
        with torch.no_grad():
            for x_cat, x_num, _ in dl:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                logits = model(x_cat, x_num).detach().cpu().numpy()
                ps.append(1 / (1 + np.exp(-logits)))
        return np.concatenate(ps)

    if not use_mc:
        p = one_pass(train_mode=False)
        return p, None

    draws = [one_pass(train_mode=True) for _ in range(mc_samples)]
    draws = np.stack(draws, axis=0)  # (S,N)
    return draws.mean(axis=0), draws.std(axis=0)

# -------------------------
# 6) CV runner
# -------------------------
def run_cv_transformer_fixed(X, y, cat_cols, num_cols, n_splits=5, n_repeats=2, seed=42):
    rng = np.random.RandomState(seed)
    rows = []

    for rep in range(1, n_repeats + 1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # inner split for early stopping
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            # fit preprocessor on TRAIN SUB only
            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xte_cat, Xte_num = prep.transform(X_te)

            # sanity checks
            assert len(y_te) == (Xte_cat.shape[0] if Xte_cat is not None else Xte_num.shape[0])

            model = train_model(
                Xtr_cat, Xtr_num, y_tr,
                Xva_cat, Xva_num, y_va,
                cat_sizes=prep.cat_sizes,
                d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30,
                seed=int(rng.randint(0, 10_000_000))
            )

            p_te, u_te = predict_proba_mc(model, Xte_cat, Xte_num, mc_samples=25, use_mc=True)
            pr = average_precision_score(y_te, p_te)
            roc = roc_auc_score(y_te, p_te)

            rows.append({"rep": rep, "fold": fold, "pr_auc": float(pr), "roc_auc": float(roc),
                         "unc_std_mean": float(np.mean(u_te))})

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] PR-AUC={pr:.4f} ROC-AUC={roc:.4f} (mc_std_mean={np.mean(u_te):.4f})")

    res = pd.DataFrame(rows)
    print("\n==== SUMMARY ====")
    print(f"PR-AUC:  {res['pr_auc'].mean():.4f} ± {res['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC: {res['roc_auc'].mean():.4f} ± {res['roc_auc'].std(ddof=1):.4f}")
    return res

# RUN
res_tt = run_cv_transformer_fixed(X, y, cat_cols=cat_cols, num_cols=num_cols, n_splits=5, n_repeats=2, seed=42)
display(res_tt.sort_values("pr_auc", ascending=False).head(10))

DEVICE: cpu
n_cat: 16 n_num: 21
[rep 1/2 fold 1/5] PR-AUC=0.2026 ROC-AUC=0.7424 (mc_std_mean=0.0524)
[rep 1/2 fold 2/5] PR-AUC=0.1696 ROC-AUC=0.7035 (mc_std_mean=0.0390)
[rep 1/2 fold 3/5] PR-AUC=0.1350 ROC-AUC=0.6337 (mc_std_mean=0.0388)
[rep 1/2 fold 4/5] PR-AUC=0.1229 ROC-AUC=0.6626 (mc_std_mean=0.0515)
[rep 1/2 fold 5/5] PR-AUC=0.1338 ROC-AUC=0.6671 (mc_std_mean=0.0440)
[rep 2/2 fold 1/5] PR-AUC=0.2007 ROC-AUC=0.7026 (mc_std_mean=0.0553)
[rep 2/2 fold 2/5] PR-AUC=0.1592 ROC-AUC=0.6912 (mc_std_mean=0.0427)
[rep 2/2 fold 3/5] PR-AUC=0.1516 ROC-AUC=0.6805 (mc_std_mean=0.0541)
[rep 2/2 fold 4/5] PR-AUC=0.2037 ROC-AUC=0.7149 (mc_std_mean=0.0639)
[rep 2/2 fold 5/5] PR-AUC=0.1407 ROC-AUC=0.6824 (mc_std_mean=0.0589)

==== SUMMARY ====
PR-AUC:  0.1620 ± 0.0308
ROC-AUC: 0.6881 ± 0.0303


,rep,fold,pr_auc,roc_auc,unc_std_mean
8,2,4,0.203659,0.714868,0.063883
0,1,1,0.202618,0.742377,0.052444
5,2,1,0.200658,0.702591,0.055336
1,1,2,0.169581,0.703460,0.038999
6,2,2,0.159160,0.691215,0.042676
7,2,3,0.151559,0.680538,0.054103
9,2,5,0.140650,0.682374,0.058893
2,1,3,0.135002,0.633691,0.038765
4,1,5,0.133779,0.667125,0.044013
3,1,4,0.122917,0.662605,0.051531


In [33]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# Reuse: FoldPreprocessor, train_model, predict_proba_mc from your last working cell.

def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def selective_metrics(y, p, u, coverages=(1.0,0.9,0.8,0.7,0.6,0.5)):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    u = np.asarray(u).astype(float)
    order = np.argsort(u)  # low uncertainty first

    rows = []
    n = len(y)
    for c in coverages:
        k = max(1, int(round(c * n)))
        idx = order[:k]
        pr = average_precision_score(y[idx], p[idx])
        roc = roc_auc_score(y[idx], p[idx]) if len(np.unique(y[idx])) > 1 else np.nan
        brier = brier_score_loss(y[idx], p[idx])
        ece = ece_score(y[idx], p[idx], n_bins=15)
        pos_rate = float(y[idx].mean())
        rows.append({"coverage": c, "n": k, "pos_rate": pos_rate, "pr_auc": pr, "roc_auc": roc, "brier": brier, "ece": ece})
    return pd.DataFrame(rows)

def run_cv_and_collect(X, y, cat_cols, num_cols, n_splits=5, n_repeats=2, seed=42, mc_samples=25):
    rng = np.random.RandomState(seed)
    all_rows = []
    all_sel = []

    for rep in range(1, n_repeats+1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xte_cat, Xte_num = prep.transform(X_te)

            model = train_model(
                Xtr_cat, Xtr_num, y_tr,
                Xva_cat, Xva_num, y_va,
                cat_sizes=prep.cat_sizes,
                d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30,
                seed=int(rng.randint(0, 10_000_000))
            )

            p_te, u_te = predict_proba_mc(model, Xte_cat, Xte_num, mc_samples=mc_samples, use_mc=True)

            pr = average_precision_score(y_te, p_te)
            roc = roc_auc_score(y_te, p_te)

            all_rows.append({
                "rep": rep, "fold": fold,
                "n": len(y_te),
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "unc_mean": float(np.mean(u_te)),
                "unc_median": float(np.median(u_te)),
                "pos_rate": float(np.mean(y_te)),
            })

            sel = selective_metrics(y_te, p_te, u_te)
            sel["rep"] = rep
            sel["fold"] = fold
            all_sel.append(sel)

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] PR={pr:.4f} ROC={roc:.4f} | unc_mean={np.mean(u_te):.4f}")

    df = pd.DataFrame(all_rows)
    sel_df = pd.concat(all_sel, axis=0, ignore_index=True)

    print("\n==== OVERALL (full coverage) ====")
    print(f"PR-AUC:  {df['pr_auc'].mean():.4f} ± {df['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC: {df['roc_auc'].mean():.4f} ± {df['roc_auc'].std(ddof=1):.4f}")

    # Aggregate selective curves across folds
    agg = sel_df.groupby("coverage").agg(
        pr_auc_mean=("pr_auc","mean"),
        pr_auc_std=("pr_auc","std"),
        roc_auc_mean=("roc_auc","mean"),
        brier_mean=("brier","mean"),
        ece_mean=("ece","mean"),
        pos_rate_mean=("pos_rate","mean"),
        n_mean=("n","mean")
    ).reset_index()

    print("\n==== SELECTIVE (uncertainty-based) ====")
    display(agg)
 
    return df, sel_df, agg

# RUN collection + selective curve table
df_full, sel_full, agg_sel = run_cv_and_collect(X, y, cat_cols, num_cols, n_splits=5, n_repeats=2, seed=42, mc_samples=25)

[rep 1/2 fold 1/5] PR=0.2026 ROC=0.7424 | unc_mean=0.0524
[rep 1/2 fold 2/5] PR=0.1696 ROC=0.7035 | unc_mean=0.0390
[rep 1/2 fold 3/5] PR=0.1350 ROC=0.6337 | unc_mean=0.0388
[rep 1/2 fold 4/5] PR=0.1229 ROC=0.6626 | unc_mean=0.0515
[rep 1/2 fold 5/5] PR=0.1338 ROC=0.6671 | unc_mean=0.0440
[rep 2/2 fold 1/5] PR=0.2007 ROC=0.7026 | unc_mean=0.0553
[rep 2/2 fold 2/5] PR=0.1592 ROC=0.6912 | unc_mean=0.0427
[rep 2/2 fold 3/5] PR=0.1516 ROC=0.6805 | unc_mean=0.0541
[rep 2/2 fold 4/5] PR=0.2037 ROC=0.7149 | unc_mean=0.0639
[rep 2/2 fold 5/5] PR=0.1407 ROC=0.6824 | unc_mean=0.0589

==== OVERALL (full coverage) ====
PR-AUC:  0.1620 ± 0.0308
ROC-AUC: 0.6881 ± 0.0303

==== SELECTIVE (uncertainty-based) ====


,coverage,pr_auc_mean,pr_auc_std,roc_auc_mean,brier_mean,ece_mean,pos_rate_mean,n_mean
0,0.5,0.157037,0.042209,0.709982,0.095954,0.189546,0.062174,460.0
1,0.6,0.153905,0.040269,0.709318,0.100802,0.196516,0.064964,552.6
2,0.7,0.157446,0.041621,0.705903,0.106122,0.198769,0.070739,644.6
3,0.8,0.163793,0.037250,0.698701,0.110355,0.202170,0.074260,736.6
4,0.9,0.161366,0.034991,0.693649,0.113809,0.205024,0.076996,828.6
5,1.0,0.161958,0.030845,0.688084,0.116650,0.208541,0.078644,920.6


In [34]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- calibration helpers ----------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def fit_temperature_scaling(p_val, y_val, max_iter=200, lr=0.05):
    """
    Fit scalar temperature T on validation logits to minimize NLL.
    We calibrate: p_cal = sigmoid(logit(p)/T).
    """
    y = torch.tensor(y_val.astype(np.float32), device=DEVICE)
    z = torch.tensor(logit(p_val).astype(np.float32), device=DEVICE)

    # optimize logT to keep T positive
    logT = torch.zeros((), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([logT], lr=lr)

    for _ in range(max_iter):
        T = torch.exp(logT) + 1e-6
        logits = z / T
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    T = float((torch.exp(logT) + 1e-6).detach().cpu().item())
    return T

def apply_temperature(p, T):
    return sigmoid(logit(p) / T)

def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

def selective_metrics_by_conf(y, p, conf, coverages=(1.0,0.9,0.8,0.7,0.6,0.5)):
    """
    conf: higher = more confident. We keep top-k by conf.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    conf = np.asarray(conf).astype(float)

    order = np.argsort(-conf)  # descending confidence
    n = len(y)

    rows = []
    for c in coverages:
        k = max(1, int(round(c * n)))
        idx = order[:k]
        pr = average_precision_score(y[idx], p[idx])
        roc = roc_auc_score(y[idx], p[idx]) if len(np.unique(y[idx])) > 1 else np.nan
        brier = brier_score_loss(y[idx], p[idx])
        ece = ece_score(y[idx], p[idx], n_bins=15)
        rows.append({
            "coverage": c, "n": k,
            "pos_rate": float(y[idx].mean()),
            "pr_auc": float(pr),
            "roc_auc": float(roc),
            "brier": float(brier),
            "ece": float(ece),
        })
    return pd.DataFrame(rows)

# ---------- main runner ----------
def run_cv_deep_ensemble_selective(X, y, cat_cols, num_cols,
                                  n_splits=5, n_repeats=2,
                                  ensemble_M=5, mc_samples=0,   # mc_samples=0 => no MC-dropout, pure ensemble
                                  seed=42):

    rng = np.random.RandomState(seed)
    all_full = []
    all_sel = []

    for rep in range(1, n_repeats+1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xte_cat, Xte_num = prep.transform(X_te)

            # ---- Train ensemble ----
            p_va_list = []
            p_te_list = []

            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_model(
                    Xtr_cat, Xtr_num, y_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                    lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30,
                    seed=seed_m
                )

                # Use deterministic eval probs (no MC dropout) for stability
                p_va, _ = predict_proba_mc(model, Xva_cat, Xva_num, mc_samples=1, use_mc=False)
                p_te, _ = predict_proba_mc(model, Xte_cat, Xte_num, mc_samples=1, use_mc=False)

                p_va_list.append(p_va)
                p_te_list.append(p_te)

            p_va_ens = np.mean(np.stack(p_va_list, axis=0), axis=0)
            p_te_ens = np.mean(np.stack(p_te_list, axis=0), axis=0)

            # Uncertainty proxy from ensemble disagreement
            u_te = np.std(np.stack(p_te_list, axis=0), axis=0)

            # ---- Temperature scaling on validation ----
            T = fit_temperature_scaling(p_va_ens, y_va, max_iter=200, lr=0.05)
            p_te_cal = apply_temperature(p_te_ens, T)

            # Confidence score: margin (works very well)
            conf_margin = np.abs(p_te_cal - 0.5)

            # Full metrics
            pr_full = average_precision_score(y_te, p_te_cal)
            roc_full = roc_auc_score(y_te, p_te_cal)
            brier_full = brier_score_loss(y_te, p_te_cal)
            ece_full = ece_score(y_te, p_te_cal)

            all_full.append({
                "rep": rep, "fold": fold,
                "pr_auc": float(pr_full),
                "roc_auc": float(roc_full),
                "brier": float(brier_full),
                "ece": float(ece_full),
                "T": float(T),
                "u_mean": float(np.mean(u_te))
            })

            # Selective by confidence
            sel = selective_metrics_by_conf(y_te, p_te_cal, conf_margin,
                                            coverages=(1.0,0.9,0.8,0.7,0.6,0.5))
            sel["rep"] = rep
            sel["fold"] = fold
            all_sel.append(sel)

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"PR(full)={pr_full:.4f} ROC(full)={roc_full:.4f} "
                  f"Brier={brier_full:.4f} ECE={ece_full:.4f} T={T:.3f}")

    df_full = pd.DataFrame(all_full)
    sel_df = pd.concat(all_sel, axis=0, ignore_index=True)

    print("\n==== OVERALL (calibrated ensemble, full coverage) ====")
    print(f"PR-AUC:  {df_full['pr_auc'].mean():.4f} ± {df_full['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC: {df_full['roc_auc'].mean():.4f} ± {df_full['roc_auc'].std(ddof=1):.4f}")
    print(f"Brier:   {df_full['brier'].mean():.4f} ± {df_full['brier'].std(ddof=1):.4f}")
    print(f"ECE:     {df_full['ece'].mean():.4f} ± {df_full['ece'].std(ddof=1):.4f}")

    agg = sel_df.groupby("coverage").agg(
        pr_auc_mean=("pr_auc","mean"),
        pr_auc_std=("pr_auc","std"),
        roc_auc_mean=("roc_auc","mean"),
        brier_mean=("brier","mean"),
        ece_mean=("ece","mean"),
        pos_rate_mean=("pos_rate","mean"),
        n_mean=("n","mean")
    ).reset_index()

    print("\n==== SELECTIVE (confidence-based, calibrated) ====")
    display(agg)

    return df_full, sel_df, agg

# RUN (CPU-friendly: M=3 first; then increase to 5 if looks promising)
dfE, selE, aggE = run_cv_deep_ensemble_selective(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,   # start with 3 on CPU
    seed=42
)

[rep 1/2 fold 1/5] PR(full)=0.1859 ROC(full)=0.7397 Brier=0.0875 ECE=0.0930 T=0.492
[rep 1/2 fold 2/5] PR(full)=0.2117 ROC(full)=0.7541 Brier=0.0746 ECE=0.0430 T=0.475
[rep 1/2 fold 3/5] PR(full)=0.1783 ROC(full)=0.6796 Brier=0.0912 ECE=0.0951 T=0.454
[rep 1/2 fold 4/5] PR(full)=0.1303 ROC(full)=0.6346 Brier=0.0807 ECE=0.0574 T=0.415
[rep 1/2 fold 5/5] PR(full)=0.1307 ROC(full)=0.6698 Brier=0.1007 ECE=0.0958 T=0.465
[rep 2/2 fold 1/5] PR(full)=0.1499 ROC(full)=0.6738 Brier=0.0953 ECE=0.1000 T=0.460
[rep 2/2 fold 2/5] PR(full)=0.1944 ROC(full)=0.7476 Brier=0.0783 ECE=0.0643 T=0.417
[rep 2/2 fold 3/5] PR(full)=0.1376 ROC(full)=0.6512 Brier=0.0947 ECE=0.1038 T=0.439
[rep 2/2 fold 4/5] PR(full)=0.1991 ROC(full)=0.7503 Brier=0.0788 ECE=0.0516 T=0.443
[rep 2/2 fold 5/5] PR(full)=0.1556 ROC(full)=0.7121 Brier=0.0884 ECE=0.0708 T=0.431

==== OVERALL (calibrated ensemble, full coverage) ====
PR-AUC:  0.1674 ± 0.0302
ROC-AUC: 0.7013 ± 0.0449
Brier:   0.0870 ± 0.0086
ECE:     0.0775 ± 0.0225

===

,coverage,pr_auc_mean,pr_auc_std,roc_auc_mean,brier_mean,ece_mean,pos_rate_mean,n_mean
0,0.5,0.076044,0.030624,0.639979,0.034781,0.016428,0.035870,460.0
1,0.6,0.098111,0.050404,0.656837,0.042077,0.019395,0.043614,552.6
2,0.7,0.093259,0.028345,0.657393,0.048162,0.024203,0.049644,644.6
3,0.8,0.120813,0.031872,0.677561,0.057542,0.034183,0.059054,736.6
4,0.9,0.147013,0.039533,0.687600,0.070679,0.053751,0.067946,828.6
5,1.0,0.167350,0.030218,0.701289,0.087017,0.077489,0.078644,920.6


In [36]:
# ============================================================
# FULL RUNNABLE PIPELINE:
# Tabular Transformer (FT-Transformer-like) + Deep Ensemble
# + Temperature Scaling Calibration + Risk Stratification
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -------------------------
# 0) EXPECTED INPUTS:
# X: pd.DataFrame
# y: array-like (0/1)
# If you already have X,y in your notebook, keep them as-is.
# Otherwise, uncomment and edit loader below.
# -------------------------
# DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"
# TARGET_COL = "stroke"
# df = pd.read_csv(DATA_PATH)
# df.columns = [c.strip() for c in df.columns]
# if "id" in df.columns: df = df.drop(columns=["id"])
# y = df[TARGET_COL].astype(int).values
# X = df.drop(columns=[TARGET_COL]).copy()

assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
y = np.asarray(y).astype(int)
assert len(X) == len(y), "X and y length mismatch"

# -------------------------
# 1) column typing
# -------------------------
def infer_cat_cols(df: pd.DataFrame, max_unique_int_as_cat=10):
    cat_cols = []
    for c in df.columns:
        if df[c].dtype == "object":
            cat_cols.append(c)
        elif str(df[c].dtype).startswith("category"):
            cat_cols.append(c)
        else:
            if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique(dropna=True) <= max_unique_int_as_cat:
                cat_cols.append(c)
    return cat_cols

cat_cols = infer_cat_cols(X)
num_cols = [c for c in X.columns if c not in cat_cols]
print("n_cat:", len(cat_cols), "n_num:", len(num_cols))

# -------------------------
# 2) Fold Preprocessor (fit on train, transform val/test)
# -------------------------
class FoldPreprocessor:
    def __init__(self, cat_cols, num_cols):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.cat_maps = {}
        self.num_stats = {}
        self.cat_sizes = []

    def fit(self, X_tr: pd.DataFrame):
        Xtr = X_tr.copy()

        # categorical mappings
        self.cat_maps = {}
        self.cat_sizes = []
        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mp = {k: i + 1 for i, k in enumerate(uniq)}  # 0 = UNK
            self.cat_maps[c] = mp
            self.cat_sizes.append(len(mp) + 1)

        # numeric stats
        self.num_stats = {}
        for c in self.num_cols:
            s = pd.to_numeric(Xtr[c], errors="coerce")
            med = float(s.median())
            s = s.fillna(med).astype(np.float32)
            mu = float(s.mean())
            sd = float(s.std(ddof=0) + 1e-6)
            self.num_stats[c] = (med, mu, sd)

        return self

    def transform(self, X_any: pd.DataFrame):
        Xa = X_any.copy()

        Xcat = None
        if len(self.cat_cols) > 0:
            arr = []
            for c in self.cat_cols:
                s = Xa[c].astype("object").fillna("MISSING").astype(str)
                mp = self.cat_maps[c]
                codes = s.map(mp).fillna(0).astype(np.int64).values
                arr.append(codes)
            Xcat = np.stack(arr, axis=1)

        Xnum = None
        if len(self.num_cols) > 0:
            arr = []
            for c in self.num_cols:
                s = pd.to_numeric(Xa[c], errors="coerce")
                med, mu, sd = self.num_stats[c]
                s = s.fillna(med).astype(np.float32)
                s = (s - mu) / sd
                arr.append(s.values.astype(np.float32))
            Xnum = np.stack(arr, axis=1)

        return Xcat, Xnum

# -------------------------
# 3) Dataset + Balanced Batch Sampler
# -------------------------
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = np.asarray(y).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        y = torch.tensor(self.y[i], dtype=torch.float32)
        x_cat = torch.tensor(self.X_cat[i], dtype=torch.long) if self.X_cat is not None else None
        x_num = torch.tensor(self.X_num[i], dtype=torch.float32) if self.X_num is not None else None
        return x_cat, x_num, y

class BalancedBatchSampler(Sampler):
    def __init__(self, y, batch_size=256, pos_frac=0.30, seed=0):
        self.y = np.asarray(y).astype(int)
        self.batch_size = int(batch_size)
        self.pos_bs = max(1, int(self.batch_size * pos_frac))
        self.neg_bs = self.batch_size - self.pos_bs
        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]
        self.rng = np.random.RandomState(seed)

    def __iter__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        for _ in range(n_batches):
            pos = self.rng.choice(self.pos_idx, size=self.pos_bs, replace=True)
            neg = self.rng.choice(self.neg_idx, size=self.neg_bs, replace=True)
            idx = np.concatenate([pos, neg])
            self.rng.shuffle(idx)
            yield from idx.tolist()

    def __len__(self):
        return int(np.ceil(len(self.y) / self.batch_size)) * self.batch_size

# -------------------------
# 4) Model: FT-Transformer-like
# -------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + self.drop(h))
        h = self.ff(x)
        x = self.ln2(x + self.drop(h))
        return x

class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.20):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = int(n_num)
        self.d_model = int(d_model)

        self.cat_embeds = nn.ModuleList([nn.Embedding(sz, d_model) for sz in cat_sizes]) if self.n_cat > 0 else None
        self.num_proj = nn.Linear(1, d_model) if self.n_num > 0 else None

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_cat, x_num):
        tokens = []

        if self.n_cat > 0:
            for j, emb in enumerate(self.cat_embeds):
                tokens.append(emb(x_cat[:, j]))  # (B,D)

        if self.n_num > 0:
            for j in range(self.n_num):
                v = x_num[:, j:j+1]               # (B,1)
                tokens.append(self.num_proj(v))   # (B,D)

        # Stack tokens => (B,T,D)
        if len(tokens) == 0:
            # shouldn't happen in your data, but safe
            B = x_cat.size(0) if x_cat is not None else x_num.size(0)
            x = torch.zeros((B, 0, self.d_model), device=(x_cat.device if x_cat is not None else x_num.device))
        else:
            x = torch.stack(tokens, dim=1)

        B = x.shape[0]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, 1+T, D)

        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)

        z = x[:, 0, :]
        logit = self.head(z).squeeze(-1)
        return logit

# -------------------------
# 5) Loss: BCE + Focal
# -------------------------
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        pt = torch.where(y == 1, p, 1 - p)
        w = (1 - pt).pow(self.gamma)
        a = torch.where(y == 1, torch.tensor(self.alpha, device=y.device), torch.tensor(1 - self.alpha, device=y.device))
        return (w * a * bce).mean()

def train_model(Xtr_cat, Xtr_num, ytr, Xva_cat, Xva_num, yva, cat_sizes,
                d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30, seed=0):

    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, ytr)
    ds_va = TabDataset(Xva_cat, Xva_num, yva)

    sampler = BalancedBatchSampler(ytr, batch_size=batch_size, pos_frac=pos_frac, seed=seed)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=2048, shuffle=False, num_workers=0)

    model = TabTransformer(cat_sizes, n_num=(0 if Xtr_num is None else Xtr_num.shape[1]),
                           d_model=d_model, n_heads=n_heads, n_layers=n_layers, dropout=dropout).to(DEVICE)

    focal = FocalBCE(gamma=2.0, alpha=0.25)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_pr = -1.0
    best_state = None
    patience = 5
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        for x_cat, x_num, yb in dl_tr:
            yb = yb.to(DEVICE)
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None

            opt.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)

            loss_bce = F.binary_cross_entropy_with_logits(logits, yb)
            loss_focal = focal(logits, yb)
            loss = 0.7 * loss_bce + 0.3 * loss_focal

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # validation PR-AUC for early stop
        model.eval()
        p_list, y_list = [], []
        with torch.no_grad():
            for x_cat, x_num, yb in dl_va:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                logits = model(x_cat, x_num).detach().cpu().numpy()
                p_list.append(1 / (1 + np.exp(-logits)))
                y_list.append(yb.numpy())

        pva = np.concatenate(p_list)
        yva_np = np.concatenate(y_list).astype(int)
        pr = average_precision_score(yva_np, pva)

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def predict_proba(model, Xcat, Xnum):
    ds = TabDataset(Xcat, Xnum, np.zeros(len(Xcat) if Xcat is not None else len(Xnum), dtype=np.float32))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, num_workers=0)

    model.eval()
    ps = []
    with torch.no_grad():
        for x_cat, x_num, _ in dl:
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None
            logits = model(x_cat, x_num).detach().cpu().numpy()
            ps.append(1 / (1 + np.exp(-logits)))
    return np.concatenate(ps)

# -------------------------
# 6) Calibration: Temperature scaling on validation
# -------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def fit_temperature_scaling(p_val, y_val, max_iter=200, lr=0.05):
    """
    Fit scalar temperature T > 0 on validation logits to minimize NLL:
    p_cal = sigmoid(logit(p)/T)
    """
    z = torch.tensor(logit(p_val).astype(np.float32), device=DEVICE)
    y = torch.tensor(y_val.astype(np.float32), device=DEVICE)

    logT = torch.zeros((), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([logT], lr=lr)

    for _ in range(max_iter):
        T = torch.exp(logT) + 1e-6
        logits = z / T
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    T = float((torch.exp(logT) + 1e-6).detach().cpu().item())
    return T

def apply_temperature(p, T):
    return sigmoid(logit(p) / T)

def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)

# -------------------------
# 7) Risk stratification metrics (Top-K% highest risk)
# -------------------------
def risk_stratification_metrics(y, p, top_fracs=(0.05, 0.10, 0.20, 0.30, 0.50)):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    order = np.argsort(-p)
    base_rate = float(y.mean())
    total_pos = max(1, int(y.sum()))
    n = len(y)

    rows = []
    for frac in top_fracs:
        k = int(np.ceil(frac * n))
        idx = order[:k]

        group_risk = float(y[idx].mean())
        lift = group_risk / base_rate if base_rate > 0 else np.nan
        capture = float(y[idx].sum()) / total_pos
        brier = float(brier_score_loss(y[idx], p[idx]))

        rows.append({
            "top_frac": frac,
            "n": k,
            "base_rate": base_rate,
            "risk_in_group": group_risk,
            "lift": lift,
            "stroke_capture_rate": capture,
            "brier_in_group": brier
        })
    return pd.DataFrame(rows)

# -------------------------
# 8) Main CV Runner:
# Deep Ensemble + Calibration + Global metrics + Risk stratification
# -------------------------
def run_cv_ensemble_calibrated_with_risk(
    X, y, cat_cols, num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,
    seed=42
):
    rng = np.random.RandomState(seed)

    all_fold_metrics = []
    all_risk_rows = []

    # Store per fold for later if needed
    y_store = {}
    p_store = {}

    for rep in range(1, n_repeats + 1):
        skf = StratifiedKFold(
            n_splits=n_splits, shuffle=True,
            random_state=int(rng.randint(0, 10_000_000))
        )

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # inner val split for early stop + calibration
            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=0.25,
                random_state=int(rng.randint(0, 10_000_000))
            )
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xte_cat, Xte_num = prep.transform(X_te)

            # Deep ensemble
            p_va_list = []
            p_te_list = []
            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_model(
                    Xtr_cat, Xtr_num, y_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                    lr=2e-3, wd=1e-5, epochs=25,
                    batch_size=256, pos_frac=0.30,
                    seed=seed_m
                )
                p_va_list.append(predict_proba(model, Xva_cat, Xva_num))
                p_te_list.append(predict_proba(model, Xte_cat, Xte_num))

            p_va_ens = np.mean(np.stack(p_va_list, axis=0), axis=0)
            p_te_ens = np.mean(np.stack(p_te_list, axis=0), axis=0)

            # Temperature scaling (per fold)
            T = fit_temperature_scaling(p_va_ens, y_va, max_iter=200, lr=0.05)
            p_te_cal = apply_temperature(p_te_ens, T)

            # metrics (full coverage)
            pr = average_precision_score(y_te, p_te_cal)
            roc = roc_auc_score(y_te, p_te_cal)
            brier = brier_score_loss(y_te, p_te_cal)
            ece = ece_score(y_te, p_te_cal, n_bins=15)

            all_fold_metrics.append({
                "rep": rep, "fold": fold,
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "T": float(T),
                "pos_rate": float(np.mean(y_te))
            })

            # store predictions for risk stratification
            key = (rep, fold)
            y_store[key] = y_te
            p_store[key] = p_te_cal

            # risk stratification for this fold
            r = risk_stratification_metrics(y_te, p_te_cal, top_fracs=(0.05,0.10,0.20,0.30,0.50))
            r["rep"] = rep
            r["fold"] = fold
            all_risk_rows.append(r)

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} ECE={ece:.4f} T={T:.3f}")

    df_metrics = pd.DataFrame(all_fold_metrics)
    df_risk = pd.concat(all_risk_rows, axis=0, ignore_index=True)

    print("\n==== OVERALL (Calibrated Ensemble) ====")
    print(f"PR-AUC:  {df_metrics['pr_auc'].mean():.4f} ± {df_metrics['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC: {df_metrics['roc_auc'].mean():.4f} ± {df_metrics['roc_auc'].std(ddof=1):.4f}")
    print(f"Brier:   {df_metrics['brier'].mean():.4f} ± {df_metrics['brier'].std(ddof=1):.4f}")
    print(f"ECE:     {df_metrics['ece'].mean():.4f} ± {df_metrics['ece'].std(ddof=1):.4f}")

    # aggregate risk stratification across folds
    risk_agg = df_risk.groupby("top_frac").agg(
        base_rate_mean=("base_rate","mean"),
        risk_mean=("risk_in_group","mean"),
        risk_std=("risk_in_group","std"),
        lift_mean=("lift","mean"),
        lift_std=("lift","std"),
        capture_mean=("stroke_capture_rate","mean"),
        capture_std=("stroke_capture_rate","std"),
        brier_mean=("brier_in_group","mean"),
    ).reset_index()

    print("\n==== RISK STRATIFICATION (Top-K% highest predicted risk) ====")
    display(risk_agg)

    return df_metrics, df_risk, risk_agg

# -------------------------
# 9) RUN (CPU-friendly defaults)
# -------------------------
# If CPU is slow, set n_repeats=1 first, then increase.
df_metrics, df_risk, risk_agg = run_cv_ensemble_calibrated_with_risk(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,   # CPU-friendly; set 5 on GPU
    seed=42
)

DEVICE: cpu
n_cat: 16 n_num: 21
[rep 1/2 fold 1/5] PR=0.1859 ROC=0.7397 Brier=0.0875 ECE=0.0930 T=0.492
[rep 1/2 fold 2/5] PR=0.2117 ROC=0.7541 Brier=0.0746 ECE=0.0430 T=0.475
[rep 1/2 fold 4/5] PR=0.1303 ROC=0.6346 Brier=0.0807 ECE=0.0574 T=0.415
[rep 1/2 fold 5/5] PR=0.1307 ROC=0.6698 Brier=0.1007 ECE=0.0958 T=0.465
[rep 2/2 fold 1/5] PR=0.1499 ROC=0.6738 Brier=0.0953 ECE=0.1000 T=0.460
[rep 2/2 fold 2/5] PR=0.1944 ROC=0.7476 Brier=0.0783 ECE=0.0643 T=0.417
[rep 2/2 fold 3/5] PR=0.1376 ROC=0.6512 Brier=0.0947 ECE=0.1038 T=0.439
[rep 2/2 fold 4/5] PR=0.1991 ROC=0.7503 Brier=0.0788 ECE=0.0516 T=0.443
[rep 2/2 fold 5/5] PR=0.1556 ROC=0.7121 Brier=0.0884 ECE=0.0708 T=0.431

==== OVERALL (Calibrated Ensemble) ====
PR-AUC:  0.1674 ± 0.0302
ROC-AUC: 0.7013 ± 0.0449
Brier:   0.0870 ± 0.0086
ECE:     0.0775 ± 0.0225

==== RISK STRATIFICATION (Top-K% highest predicted risk) ====


,top_frac,base_rate_mean,risk_mean,risk_std,lift_mean,lift_std,capture_mean,capture_std,brier_mean
0,0.05,0.078644,0.220768,0.048075,2.805781,0.604460,0.142180,0.031518,0.320487
1,0.10,0.078644,0.195395,0.029321,2.483358,0.363122,0.249867,0.037144,0.270028
2,0.20,0.078644,0.158722,0.027965,2.017692,0.351356,0.404585,0.070456,0.210605
3,0.30,0.078644,0.145691,0.020764,1.852666,0.264956,0.556659,0.079700,0.178820
4,0.50,0.078644,0.121361,0.009259,1.543084,0.116027,0.772051,0.058138,0.139368


In [38]:
# sanity check: brier must be <= 1, and typically around 0.05-0.20 in these tasks
print("Overall brier mean:", df_metrics["brier"].mean(), "max:", df_metrics["brier"].max())
print("Risk table brier mean range:", risk_agg["brier_mean"].min(), risk_agg["brier_mean"].max())

Overall brier mean: 0.0870172692626838 max: 0.10068720402413843
Risk table brier mean range: 0.13936760445495894 0.32048723000990226


In [1]:
# ============================================================
# TRUST-STROKE (CDE-TT) — FINAL BASELINE CODE (FROM START)
# ------------------------------------------------------------
# A trustworthy stroke risk stratification framework:
# - Tabular Transformer backbone (attention over cat+num tokens)
# - Imbalance-aware training (balanced batches + BCE/Focal hybrid)
# - Deep Ensemble (epistemic uncertainty)
# - Temperature Scaling calibration (per-fold)
# - Metrics: PR-AUC, ROC-AUC, Brier, ECE
# - Risk Stratification: lift + capture + Brier Skill Score (BSS)
# - Decision Curve Analysis (Net Benefit)
#
# NOTE:
# - This is your "final code base" to continue expanding:
#   interpretability, fairness, robustness/attacks will be added on top.
# - CPU-safe defaults. If GPU available, increase ensemble_M, epochs.
# ============================================================

# -------------------------
# 0) Imports
# -------------------------
import os
import gc
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# -------------------------
# 1) Global Config
# -------------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# CV + training
N_SPLITS = 5
N_REPEATS = 2
ENSEMBLE_M = 3          # try 5+ on GPU
EPOCHS = 25             # try 35+ on GPU
BATCH_SIZE = 256
POS_FRAC = 0.30

# model
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.20
LR = 2e-3
WD = 1e-5

# metrics
ECE_BINS = 15

# risk stratification groups
TOP_FRACS = (0.05, 0.10, 0.20, 0.30, 0.50)

# decision curve thresholds (tune later)
DCA_THRESHOLDS = np.linspace(0.01, 0.40, 40)


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# -------------------------
# 2) Data Loading Template
# -------------------------
# If you already have X (DataFrame) and y (0/1 array), SKIP this section.
# Otherwise, set DATA_PATH and TARGET_COL properly.

# Example (common Kaggle stroke dataset):
DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"
TARGET_COL = "stroke"
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
if "id" in df.columns: df = df.drop(columns=["id"])
y = df[TARGET_COL].astype(int).values
X = df.drop(columns=[TARGET_COL]).copy()

# Safety checks:
assert "X" in globals() and "y" in globals(), "Please define X (DataFrame) and y (0/1 array) before running."
assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
y = np.asarray(y).astype(int)
assert len(X) == len(y), "X and y must have same length"
print("N:", len(X), "Pos rate:", float(y.mean()))


# -------------------------
# 3) Column typing
# -------------------------
def infer_cat_cols(df: pd.DataFrame, max_unique_int_as_cat=10):
    cat_cols = []
    for c in df.columns:
        if df[c].dtype == "object":
            cat_cols.append(c)
        elif str(df[c].dtype).startswith("category"):
            cat_cols.append(c)
        else:
            # treat small-cardinality integers as categorical (optional)
            if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique(dropna=True) <= max_unique_int_as_cat:
                cat_cols.append(c)
    return cat_cols


cat_cols = infer_cat_cols(X)
num_cols = [c for c in X.columns if c not in cat_cols]
print("n_cat:", len(cat_cols), "n_num:", len(num_cols))


# -------------------------
# 4) Fold Preprocessor (fit on train only)
# -------------------------
class FoldPreprocessor:
    """
    - Categorical: map category strings -> ids, with 0 reserved for UNK
    - Numerical: median-impute + z-score standardize (fit on train)
    """
    def __init__(self, cat_cols, num_cols):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.cat_maps = {}
        self.num_stats = {}
        self.cat_sizes = []

    def fit(self, X_tr: pd.DataFrame):
        Xtr = X_tr.copy()

        # categorical mappings
        self.cat_maps = {}
        self.cat_sizes = []
        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mp = {k: i + 1 for i, k in enumerate(uniq)}  # 0 = UNK
            self.cat_maps[c] = mp
            self.cat_sizes.append(len(mp) + 1)

        # numerical stats
        self.num_stats = {}
        for c in self.num_cols:
            s = pd.to_numeric(Xtr[c], errors="coerce")
            med = float(s.median())
            s = s.fillna(med).astype(np.float32)
            mu = float(s.mean())
            sd = float(s.std(ddof=0) + 1e-6)
            self.num_stats[c] = (med, mu, sd)

        return self

    def transform(self, X_any: pd.DataFrame):
        Xa = X_any.copy()

        Xcat = None
        if len(self.cat_cols) > 0:
            arr = []
            for c in self.cat_cols:
                s = Xa[c].astype("object").fillna("MISSING").astype(str)
                mp = self.cat_maps[c]
                codes = s.map(mp).fillna(0).astype(np.int64).values
                arr.append(codes)
            Xcat = np.stack(arr, axis=1)

        Xnum = None
        if len(self.num_cols) > 0:
            arr = []
            for c in self.num_cols:
                s = pd.to_numeric(Xa[c], errors="coerce")
                med, mu, sd = self.num_stats[c]
                s = s.fillna(med).astype(np.float32)
                s = (s - mu) / sd
                arr.append(s.values.astype(np.float32))
            Xnum = np.stack(arr, axis=1)

        return Xcat, Xnum


# -------------------------
# 5) Dataset + Balanced Sampler
# -------------------------
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = np.asarray(y).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        yi = torch.tensor(self.y[i], dtype=torch.float32)
        x_cat = torch.tensor(self.X_cat[i], dtype=torch.long) if self.X_cat is not None else None
        x_num = torch.tensor(self.X_num[i], dtype=torch.float32) if self.X_num is not None else None
        return x_cat, x_num, yi


class BalancedBatchSampler(Sampler):
    """
    Samples each batch with approx pos_frac positives and (1-pos_frac) negatives.
    This is a simple, strong technique for rare-event training.
    """
    def __init__(self, y, batch_size=256, pos_frac=0.30, seed=0):
        self.y = np.asarray(y).astype(int)
        self.batch_size = int(batch_size)
        self.pos_bs = max(1, int(self.batch_size * pos_frac))
        self.neg_bs = self.batch_size - self.pos_bs
        self.pos_idx = np.where(self.y == 1)[0]
        self.neg_idx = np.where(self.y == 0)[0]
        self.rng = np.random.RandomState(seed)

    def __iter__(self):
        n_batches = int(np.ceil(len(self.y) / self.batch_size))
        for _ in range(n_batches):
            pos = self.rng.choice(self.pos_idx, size=self.pos_bs, replace=True)
            neg = self.rng.choice(self.neg_idx, size=self.neg_bs, replace=True)
            idx = np.concatenate([pos, neg])
            self.rng.shuffle(idx)
            yield from idx.tolist()

    def __len__(self):
        return int(np.ceil(len(self.y) / self.batch_size)) * self.batch_size


# -------------------------
# 6) Model: Tabular Transformer (FT-Transformer-like)
# -------------------------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + self.drop(h))
        h = self.ff(x)
        x = self.ln2(x + self.drop(h))
        return x


class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.20):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = int(n_num)
        self.d_model = int(d_model)

        self.cat_embeds = nn.ModuleList([nn.Embedding(sz, d_model) for sz in cat_sizes]) if self.n_cat > 0 else None
        self.num_proj = nn.Linear(1, d_model) if self.n_num > 0 else None

        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_cat, x_num):
        tokens = []

        if self.n_cat > 0:
            for j, emb in enumerate(self.cat_embeds):
                tokens.append(emb(x_cat[:, j]))  # (B,D)

        if self.n_num > 0:
            for j in range(self.n_num):
                v = x_num[:, j:j+1]              # (B,1)
                tokens.append(self.num_proj(v))  # (B,D)

        if len(tokens) == 0:
            B = x_cat.size(0) if x_cat is not None else x_num.size(0)
            x = torch.zeros((B, 0, self.d_model), device=(x_cat.device if x_cat is not None else x_num.device))
        else:
            x = torch.stack(tokens, dim=1)       # (B,T,D)

        B = x.shape[0]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)           # (B,1+T,D)

        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)

        z = x[:, 0, :]
        logit = self.head(z).squeeze(-1)
        return logit


# -------------------------
# 7) Loss: BCE + Focal (hybrid)
# -------------------------
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p = torch.sigmoid(logits)
        pt = torch.where(y == 1, p, 1 - p)
        w = (1 - pt).pow(self.gamma)
        a = torch.where(
            y == 1,
            torch.tensor(self.alpha, device=y.device),
            torch.tensor(1 - self.alpha, device=y.device),
        )
        return (w * a * bce).mean()


def train_one_model(
    Xtr_cat, Xtr_num, ytr,
    Xva_cat, Xva_num, yva,
    cat_sizes,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
    lr=LR, wd=WD, epochs=EPOCHS, batch_size=BATCH_SIZE, pos_frac=POS_FRAC,
    seed=0
):
    set_seed(seed)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, ytr)
    ds_va = TabDataset(Xva_cat, Xva_num, yva)

    sampler = BalancedBatchSampler(ytr, batch_size=batch_size, pos_frac=pos_frac, seed=seed)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=2048, shuffle=False, num_workers=0)

    model = TabTransformer(
        cat_sizes=cat_sizes,
        n_num=(0 if Xtr_num is None else Xtr_num.shape[1]),
        d_model=d_model, n_heads=n_heads, n_layers=n_layers, dropout=dropout
    ).to(DEVICE)

    focal = FocalBCE(gamma=2.0, alpha=0.25)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_pr = -1.0
    best_state = None
    patience = 5
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        for x_cat, x_num, yb in dl_tr:
            yb = yb.to(DEVICE)
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None

            opt.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)

            loss_bce = F.binary_cross_entropy_with_logits(logits, yb)
            loss_focal = focal(logits, yb)
            loss = 0.7 * loss_bce + 0.3 * loss_focal

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # early stop on val PR-AUC
        model.eval()
        p_list, y_list = [], []
        with torch.no_grad():
            for x_cat, x_num, yb in dl_va:
                x_cat = x_cat.to(DEVICE) if x_cat is not None else None
                x_num = x_num.to(DEVICE) if x_num is not None else None
                logits = model(x_cat, x_num).detach().cpu().numpy()
                p_list.append(1.0 / (1.0 + np.exp(-logits)))
                y_list.append(yb.numpy())

        pva = np.concatenate(p_list)
        yva_np = np.concatenate(y_list).astype(int)
        pr = average_precision_score(yva_np, pva)

        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def predict_proba(model, Xcat, Xnum):
    n = len(Xcat) if Xcat is not None else len(Xnum)
    ds = TabDataset(Xcat, Xnum, np.zeros(n, dtype=np.float32))
    dl = DataLoader(ds, batch_size=2048, shuffle=False, num_workers=0)

    model.eval()
    ps = []
    with torch.no_grad():
        for x_cat, x_num, _ in dl:
            x_cat = x_cat.to(DEVICE) if x_cat is not None else None
            x_num = x_num.to(DEVICE) if x_num is not None else None
            logits = model(x_cat, x_num).detach().cpu().numpy()
            ps.append(1.0 / (1.0 + np.exp(-logits)))
    return np.concatenate(ps)


# -------------------------
# 8) Calibration: Temperature Scaling
# -------------------------
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_temperature_scaling(p_val, y_val, max_iter=200, lr=0.05):
    """
    Fit scalar T>0 on validation logits to minimize NLL.
    p_cal = sigmoid(logit(p)/T)
    """
    z = torch.tensor(logit(p_val).astype(np.float32), device=DEVICE)
    y = torch.tensor(y_val.astype(np.float32), device=DEVICE)

    logT = torch.zeros((), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([logT], lr=lr)

    for _ in range(max_iter):
        T = torch.exp(logT) + 1e-6
        logits = z / T
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    T = float((torch.exp(logT) + 1e-6).detach().cpu().item())
    return T


def apply_temperature(p, T):
    return sigmoid(logit(p) / T)


def ece_score(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y_true[m].mean()
        conf = p[m].mean()
        ece += (m.sum() / len(p)) * abs(acc - conf)
    return float(ece)


# -------------------------
# 9) Risk Stratification: Lift + Capture + Brier Skill Score
# -------------------------
def risk_stratification_metrics(y, p, top_fracs=TOP_FRACS):
    """
    Computes risk enrichment and capture rate in top-K risk groups,
    plus Brier Skill Score (BSS) per group.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    order = np.argsort(-p)
    base_rate = float(y.mean())
    total_pos = max(1, int(y.sum()))
    n = len(y)

    rows = []
    for frac in top_fracs:
        k = int(np.ceil(frac * n))
        idx = order[:k]

        y_g = y[idx]
        p_g = p[idx]

        group_rate = float(y_g.mean())
        lift = group_rate / base_rate if base_rate > 0 else np.nan
        capture = float(y_g.sum()) / total_pos

        brier_model = float(brier_score_loss(y_g, p_g))
        brier_null = float(np.mean((group_rate - y_g) ** 2))  # constant predictor = group prevalence
        bss = float(1.0 - (brier_model / (brier_null + 1e-12)))

        rows.append({
            "top_frac": frac,
            "n": k,
            "base_rate": base_rate,
            "risk_in_group": group_rate,
            "lift": lift,
            "stroke_capture_rate": capture,
            "brier_model": brier_model,
            "brier_null": brier_null,
            "brier_skill": bss
        })

    return pd.DataFrame(rows)


# -------------------------
# 10) Decision Curve Analysis (Net Benefit)
# -------------------------
def decision_curve_net_benefit(y, p, thresholds):
    """
    Net Benefit = TP/N - FP/N * (pt/(1-pt))
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    N = len(y)
    out = []
    for pt in thresholds:
        pred = (p >= pt).astype(int)
        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        nb = (tp / N) - (fp / N) * (pt / (1 - pt + 1e-12))
        out.append(nb)
    return np.array(out)


# -------------------------
# 11) Main Runner: Repeated CV + Ensemble + Calib + Trust Metrics
# -------------------------
def run_trust_stroke_cv(
    X, y,
    cat_cols, num_cols,
    n_splits=N_SPLITS, n_repeats=N_REPEATS,
    ensemble_M=ENSEMBLE_M,
    seed=SEED
):
    rng = np.random.RandomState(seed)

    all_fold_metrics = []
    all_risk_rows = []
    all_dca_rows = []

    # store per fold preds to continue later
    y_store = {}
    p_store = {}

    for rep in range(1, n_repeats + 1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # inner validation split (for early-stop + temperature scaling)
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, va_idx = next(sss.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_va, y_va = X_tr_full.iloc[va_idx], y_tr_full[va_idx]

            # preprocess per fold
            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xte_cat, Xte_num = prep.transform(X_te)

            # ensemble training
            p_va_list, p_te_list = [], []
            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_one_model(
                    Xtr_cat, Xtr_num, y_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
                    lr=LR, wd=WD, epochs=EPOCHS,
                    batch_size=BATCH_SIZE, pos_frac=POS_FRAC,
                    seed=seed_m
                )
                p_va_list.append(predict_proba(model, Xva_cat, Xva_num))
                p_te_list.append(predict_proba(model, Xte_cat, Xte_num))

                # cleanup
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            p_va_ens = np.mean(np.stack(p_va_list, axis=0), axis=0)
            p_te_ens = np.mean(np.stack(p_te_list, axis=0), axis=0)

            # temperature scaling
            T = fit_temperature_scaling(p_va_ens, y_va, max_iter=200, lr=0.05)
            p_te_cal = apply_temperature(p_te_ens, T)

            # fold metrics (global)
            pr = average_precision_score(y_te, p_te_cal)
            roc = roc_auc_score(y_te, p_te_cal)
            brier = brier_score_loss(y_te, p_te_cal)
            ece = ece_score(y_te, p_te_cal, n_bins=ECE_BINS)

            all_fold_metrics.append({
                "rep": rep, "fold": fold,
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "T": float(T),
                "pos_rate": float(np.mean(y_te)),
                "ensemble_M": int(ensemble_M)
            })

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} ECE={ece:.4f} T={T:.3f}")

            # store predictions
            key = (rep, fold)
            y_store[key] = y_te
            p_store[key] = p_te_cal

            # risk stratification rows
            r = risk_stratification_metrics(y_te, p_te_cal, top_fracs=TOP_FRACS)
            r["rep"] = rep
            r["fold"] = fold
            all_risk_rows.append(r)

            # decision curve rows
            nb_model = decision_curve_net_benefit(y_te, p_te_cal, DCA_THRESHOLDS)
            prev = float(np.mean(y_te))
            nb_all = prev - (1 - prev) * (DCA_THRESHOLDS / (1 - DCA_THRESHOLDS + 1e-12))
            nb_none = np.zeros_like(DCA_THRESHOLDS)

            dca_df = pd.DataFrame({
                "rep": rep, "fold": fold,
                "threshold": DCA_THRESHOLDS,
                "nb_model": nb_model,
                "nb_all": nb_all,
                "nb_none": nb_none
            })
            all_dca_rows.append(dca_df)

    df_metrics = pd.DataFrame(all_fold_metrics)
    df_risk = pd.concat(all_risk_rows, axis=0, ignore_index=True)
    df_dca = pd.concat(all_dca_rows, axis=0, ignore_index=True)

    # overall summary
    print("\n==== OVERALL (Calibrated Ensemble) ====")
    print(f"PR-AUC:  {df_metrics['pr_auc'].mean():.4f} ± {df_metrics['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC: {df_metrics['roc_auc'].mean():.4f} ± {df_metrics['roc_auc'].std(ddof=1):.4f}")
    print(f"Brier:   {df_metrics['brier'].mean():.4f} ± {df_metrics['brier'].std(ddof=1):.4f}")
    print(f"ECE:     {df_metrics['ece'].mean():.4f} ± {df_metrics['ece'].std(ddof=1):.4f}")

    # aggregate risk stratification across folds
    risk_agg = df_risk.groupby("top_frac").agg(
        base_rate_mean=("base_rate","mean"),
        risk_mean=("risk_in_group","mean"),
        risk_std=("risk_in_group","std"),
        lift_mean=("lift","mean"),
        lift_std=("lift","std"),
        capture_mean=("stroke_capture_rate","mean"),
        capture_std=("stroke_capture_rate","std"),
        bss_mean=("brier_skill","mean"),
        bss_std=("brier_skill","std"),
    ).reset_index()

    print("\n==== RISK STRATIFICATION (Top-K% highest predicted risk) ====")
    display(risk_agg)

    # aggregate DCA across folds
    dca_agg = df_dca.groupby("threshold").agg(
        nb_model_mean=("nb_model","mean"),
        nb_model_std=("nb_model","std"),
        nb_all_mean=("nb_all","mean"),
        nb_none_mean=("nb_none","mean"),
    ).reset_index()

    print("\n==== DECISION CURVE (aggregated) ====")
    display(dca_agg.head(10))

    return df_metrics, df_risk, risk_agg, df_dca, dca_agg, y_store, p_store


# -------------------------
# 12) RUN
# -------------------------
df_metrics, df_risk, risk_agg, df_dca, dca_agg, y_store, p_store = run_trust_stroke_cv(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=N_SPLITS, n_repeats=N_REPEATS,
    ensemble_M=ENSEMBLE_M,
    seed=SEED
)


# -------------------------
# 13) Save artifacts (optional)
# -------------------------
df_metrics.to_csv("trust_stroke_metrics.csv", index=False)
df_risk.to_csv("trust_stroke_risk_strat_rows.csv", index=False)
risk_agg.to_csv("trust_stroke_risk_strat_agg.csv", index=False)
df_dca.to_csv("trust_stroke_dca_rows.csv", index=False)
dca_agg.to_csv("trust_stroke_dca_agg.csv", index=False)

print("\nDONE. This is the finalized base code for TRUST-STROKE (CDE-TT).")
print("Next steps will be added on top: interpretability, fairness, robustness/attacks.")

DEVICE: cpu
N: 4603 Pos rate: 0.07864436237236584
n_cat: 15 n_num: 20
[rep 1/2 fold 1/5] PR=0.1800 ROC=0.7321 Brier=0.0732 ECE=0.0438 T=0.384
[rep 1/2 fold 2/5] PR=0.2151 ROC=0.7547 Brier=0.0899 ECE=0.0977 T=0.502
[rep 1/2 fold 3/5] PR=0.2140 ROC=0.6818 Brier=0.1154 ECE=0.1456 T=0.495
[rep 1/2 fold 4/5] PR=0.1402 ROC=0.6738 Brier=0.0910 ECE=0.0842 T=0.498
[rep 1/2 fold 5/5] PR=0.1415 ROC=0.6747 Brier=0.0958 ECE=0.1025 T=0.377
[rep 2/2 fold 1/5] PR=0.1562 ROC=0.6920 Brier=0.0929 ECE=0.0914 T=0.480
[rep 2/2 fold 2/5] PR=0.1514 ROC=0.7245 Brier=0.0943 ECE=0.0906 T=0.453
[rep 2/2 fold 3/5] PR=0.1311 ROC=0.6414 Brier=0.0923 ECE=0.0905 T=0.541
[rep 2/2 fold 4/5] PR=0.2303 ROC=0.7561 Brier=0.0815 ECE=0.0784 T=0.409
[rep 2/2 fold 5/5] PR=0.1563 ROC=0.7039 Brier=0.0843 ECE=0.0602 T=0.411

==== OVERALL (Calibrated Ensemble) ====
PR-AUC:  0.1716 ± 0.0359
ROC-AUC: 0.7035 ± 0.0377
Brier:   0.0911 ± 0.0110
ECE:     0.0885 ± 0.0268

==== RISK STRATIFICATION (Top-K% highest predicted risk) ====


,top_frac,base_rate_mean,risk_mean,risk_std,lift_mean,lift_std,capture_mean,capture_std,bss_mean,bss_std
0,0.05,0.078644,0.221045,0.052491,2.810269,0.662693,0.142237,0.033584,-1.057217,0.528482
1,0.10,0.078644,0.197616,0.031714,2.511862,0.396472,0.252664,0.039950,-0.854828,0.356052
2,0.20,0.078644,0.167406,0.031823,2.129062,0.405605,0.426884,0.081130,-0.681129,0.318582
3,0.30,0.078644,0.146420,0.021505,1.862150,0.275491,0.559494,0.082757,-0.553091,0.280476
4,0.50,0.078644,0.120493,0.006288,1.532213,0.080769,0.766610,0.040507,-0.387408,0.200712



==== DECISION CURVE (aggregated) ====


,threshold,nb_model_mean,nb_model_std,nb_all_mean,nb_none_mean
0,0.01,0.067932,0.002685,0.069338,0.0
1,0.02,0.058303,0.004132,0.059841,0.0
2,0.03,0.050912,0.004067,0.050149,0.0
3,0.04,0.044554,0.003592,0.040254,0.0
4,0.05,0.038030,0.004042,0.030152,0.0
5,0.06,0.032425,0.003783,0.019834,0.0
6,0.07,0.027056,0.003442,0.009295,0.0
7,0.08,0.022202,0.004082,-0.001474,0.0
8,0.09,0.018181,0.004064,-0.012479,0.0
9,0.10,0.012721,0.003765,-0.023729,0.0



DONE. This is the finalized base code for TRUST-STROKE (CDE-TT).
Next steps will be added on top: interpretability, fairness, robustness/attacks.


In [2]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

def nll_loss(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.clip(np.asarray(p).astype(float), 1e-6, 1 - 1e-6)
    return float(-(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)).mean())


def tail_ece(y_true, p, top_frac=0.10, n_bins=10):
    """
    ECE computed only on the top_frac highest predicted risk samples.
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)

    n = len(p)
    k = max(1, int(np.ceil(top_frac * n)))
    idx = np.argsort(-p)[:k]
    return ece_score(y_true[idx], p[idx], n_bins=n_bins)


def fit_platt_scaling(p_cal, y_cal):
    """
    Platt scaling using logistic regression on logits.
    p' = sigmoid(a*logit(p) + b)
    """
    z = logit(np.clip(p_cal, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(z, y_cal.astype(int))
    return lr


def apply_platt(lr_model, p):
    z = logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    return lr_model.predict_proba(z)[:, 1]


def fit_isotonic(p_cal, y_cal):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_cal.astype(float), y_cal.astype(int))
    return iso


def apply_isotonic(iso_model, p):
    return iso_model.transform(p.astype(float))

def run_trust_stroke_cv(
    X, y,
    cat_cols, num_cols,
    n_splits=N_SPLITS, n_repeats=N_REPEATS,
    ensemble_M=ENSEMBLE_M,
    seed=SEED
):
    rng = np.random.RandomState(seed)

    all_fold_metrics = []
    all_risk_rows = []
    all_dca_rows = []

    y_store = {}
    p_store = {}
    cal_choice_store = {}

    for rep in range(1, n_repeats + 1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # ----------------------------
            # Inner 3-way split: train / val(early stop) / cal(calibration)
            # ----------------------------
            # Step 1: split off "hold" portion
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, hold_idx = next(sss1.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_hold, y_hold = X_tr_full.iloc[hold_idx], y_tr_full[hold_idx]

            # Step 2: split hold into val and cal (20% + 20% of full training fold)
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(rng.randint(0, 10_000_000)))
            val_idx, cal_idx = next(sss2.split(X_hold, y_hold))
            X_va, y_va = X_hold.iloc[val_idx], y_hold[val_idx]
            X_cal, y_cal = X_hold.iloc[cal_idx], y_hold[cal_idx]

            # preprocess fit on TRAIN ONLY
            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xcal_cat, Xcal_num = prep.transform(X_cal)
            Xte_cat, Xte_num = prep.transform(X_te)

            # ----------------------------
            # Train ensemble (early stop on VAL)
            # ----------------------------
            p_cal_list, p_te_list = [], []

            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_one_model(
                    Xtr_cat, Xtr_num, y_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
                    lr=LR, wd=WD, epochs=EPOCHS,
                    batch_size=BATCH_SIZE, pos_frac=POS_FRAC,
                    seed=seed_m
                )
                p_cal_list.append(predict_proba(model, Xcal_cat, Xcal_num))
                p_te_list.append(predict_proba(model, Xte_cat, Xte_num))

                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            p_cal_ens = np.mean(np.stack(p_cal_list, axis=0), axis=0)
            p_te_ens = np.mean(np.stack(p_te_list, axis=0), axis=0)

            # ----------------------------
            # Fit calibrators on CAL (not VAL, not TEST)
            # ----------------------------
            # Temperature
            T = fit_temperature_scaling(p_cal_ens, y_cal, max_iter=250, lr=0.05)
            p_te_temp = apply_temperature(p_te_ens, T)

            # Platt
            platt = fit_platt_scaling(p_cal_ens, y_cal)
            p_te_platt = apply_platt(platt, p_te_ens)

            # Isotonic
            iso = fit_isotonic(p_cal_ens, y_cal)
            p_te_iso = apply_isotonic(iso, p_te_ens)

            # Choose calibrator based on CAL NLL (no test leakage)
            nll_temp = nll_loss(y_cal, apply_temperature(p_cal_ens, T))
            nll_platt = nll_loss(y_cal, apply_platt(platt, p_cal_ens))
            nll_iso = nll_loss(y_cal, apply_isotonic(iso, p_cal_ens))

            choices = {
                "temp": (nll_temp, p_te_temp),
                "platt": (nll_platt, p_te_platt),
                "isotonic": (nll_iso, p_te_iso),
            }
            best_name = sorted(choices.items(), key=lambda kv: kv[1][0])[0][0]
            best_nll, p_te_cal = choices[best_name]

            # ----------------------------
            # Metrics on TEST
            # ----------------------------
            pr = average_precision_score(y_te, p_te_cal)
            roc = roc_auc_score(y_te, p_te_cal)
            brier = brier_score_loss(y_te, p_te_cal)
            ece = ece_score(y_te, p_te_cal, n_bins=ECE_BINS)
            tece10 = tail_ece(y_te, p_te_cal, top_frac=0.10, n_bins=10)
            tece20 = tail_ece(y_te, p_te_cal, top_frac=0.20, n_bins=10)

            all_fold_metrics.append({
                "rep": rep, "fold": fold,
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "tail_ece10": float(tece10),
                "tail_ece20": float(tece20),
                "calibrator": best_name,
                "cal_nll": float(best_nll),
                "T_if_temp": float(T),
                "pos_rate": float(np.mean(y_te)),
                "ensemble_M": int(ensemble_M)
            })

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"Cal={best_name} PR={pr:.4f} ROC={roc:.4f} "
                  f"Brier={brier:.4f} ECE={ece:.4f} TailECE10={tece10:.4f} TailECE20={tece20:.4f}")

            # store preds
            key = (rep, fold)
            y_store[key] = y_te
            p_store[key] = p_te_cal
            cal_choice_store[key] = best_name

            # risk stratification rows (no BSS emphasis—keep lift/capture + brier_skill if you want)
            r = risk_stratification_metrics(y_te, p_te_cal, top_fracs=TOP_FRACS)
            r["rep"] = rep
            r["fold"] = fold
            r["calibrator"] = best_name
            all_risk_rows.append(r)

            # decision curve rows
            nb_model = decision_curve_net_benefit(y_te, p_te_cal, DCA_THRESHOLDS)
            prev = float(np.mean(y_te))
            nb_all = prev - (1 - prev) * (DCA_THRESHOLDS / (1 - DCA_THRESHOLDS + 1e-12))
            nb_none = np.zeros_like(DCA_THRESHOLDS)

            dca_df = pd.DataFrame({
                "rep": rep, "fold": fold,
                "threshold": DCA_THRESHOLDS,
                "nb_model": nb_model,
                "nb_all": nb_all,
                "nb_none": nb_none
            })
            all_dca_rows.append(dca_df)

    df_metrics = pd.DataFrame(all_fold_metrics)
    df_risk = pd.concat(all_risk_rows, axis=0, ignore_index=True)
    df_dca = pd.concat(all_dca_rows, axis=0, ignore_index=True)

    print("\n==== OVERALL (Best Calibrator chosen on CAL NLL) ====")
    print("Calibrator counts:\n", df_metrics["calibrator"].value_counts())
    print(f"PR-AUC:     {df_metrics['pr_auc'].mean():.4f} ± {df_metrics['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC:    {df_metrics['roc_auc'].mean():.4f} ± {df_metrics['roc_auc'].std(ddof=1):.4f}")
    print(f"Brier:      {df_metrics['brier'].mean():.4f} ± {df_metrics['brier'].std(ddof=1):.4f}")
    print(f"ECE:        {df_metrics['ece'].mean():.4f} ± {df_metrics['ece'].std(ddof=1):.4f}")
    print(f"TailECE10:  {df_metrics['tail_ece10'].mean():.4f} ± {df_metrics['tail_ece10'].std(ddof=1):.4f}")
    print(f"TailECE20:  {df_metrics['tail_ece20'].mean():.4f} ± {df_metrics['tail_ece20'].std(ddof=1):.4f}")

    risk_agg = df_risk.groupby("top_frac").agg(
        base_rate_mean=("base_rate","mean"),
        risk_mean=("risk_in_group","mean"),
        risk_std=("risk_in_group","std"),
        lift_mean=("lift","mean"),
        lift_std=("lift","std"),
        capture_mean=("stroke_capture_rate","mean"),
        capture_std=("stroke_capture_rate","std"),
        bss_mean=("brier_skill","mean"),
        bss_std=("brier_skill","std"),
    ).reset_index()

    print("\n==== RISK STRATIFICATION (Top-K% highest predicted risk) ====")
    display(risk_agg)

    dca_agg = df_dca.groupby("threshold").agg(
        nb_model_mean=("nb_model","mean"),
        nb_model_std=("nb_model","std"),
        nb_all_mean=("nb_all","mean"),
        nb_none_mean=("nb_none","mean"),
    ).reset_index()

    print("\n==== DECISION CURVE (aggregated) ====")
    display(dca_agg.head(10))

    return df_metrics, df_risk, risk_agg, df_dca, dca_agg, y_store, p_store, cal_choice_store

df_metrics, df_risk, risk_agg, df_dca, dca_agg, y_store, p_store, cal_choice_store = run_trust_stroke_cv(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=N_SPLITS, n_repeats=N_REPEATS,
    ensemble_M=ENSEMBLE_M,
    seed=SEED
)

[rep 1/2 fold 1/5] Cal=isotonic PR=0.1733 ROC=0.7205 Brier=0.0692 ECE=0.0154 TailECE10=0.0257 TailECE20=0.0073
[rep 1/2 fold 2/5] Cal=isotonic PR=0.1644 ROC=0.7358 Brier=0.0689 ECE=0.0078 TailECE10=0.0286 TailECE20=0.0373
[rep 1/2 fold 3/5] Cal=isotonic PR=0.1548 ROC=0.6666 Brier=0.0711 ECE=0.0132 TailECE10=0.0196 TailECE20=0.0028
[rep 1/2 fold 4/5] Cal=isotonic PR=0.1333 ROC=0.6677 Brier=0.0705 ECE=0.0149 TailECE10=0.0257 TailECE20=0.0049
[rep 1/2 fold 5/5] Cal=isotonic PR=0.1302 ROC=0.6637 Brier=0.0733 ECE=0.0281 TailECE10=0.1202 TailECE20=0.0801
[rep 2/2 fold 1/5] Cal=isotonic PR=0.1797 ROC=0.7170 Brier=0.0686 ECE=0.0253 TailECE10=0.0995 TailECE20=0.0687
[rep 2/2 fold 2/5] Cal=isotonic PR=0.1404 ROC=0.7005 Brier=0.0711 ECE=0.0179 TailECE10=0.0823 TailECE20=0.0565
[rep 2/2 fold 3/5] Cal=isotonic PR=0.1615 ROC=0.7242 Brier=0.0699 ECE=0.0160 TailECE10=0.0783 TailECE20=0.0709
[rep 2/2 fold 4/5] Cal=isotonic PR=0.1227 ROC=0.6730 Brier=0.0773 ECE=0.0398 TailECE10=0.1766 TailECE20=0.1077
[

,top_frac,base_rate_mean,risk_mean,risk_std,lift_mean,lift_std,capture_mean,capture_std,bss_mean,bss_std
0,0.05,0.078644,0.231406,0.062250,2.942236,0.790757,0.149144,0.040972,-0.181351,0.345163
1,0.10,0.078644,0.186723,0.027110,2.373647,0.339285,0.238870,0.035029,-0.075899,0.163468
2,0.20,0.078644,0.158693,0.017231,2.017297,0.212910,0.404566,0.043256,-0.036398,0.099275
3,0.30,0.078644,0.142790,0.012929,1.815606,0.163857,0.545548,0.049712,-0.023874,0.068779
4,0.50,0.078644,0.120923,0.007900,1.537320,0.095076,0.769178,0.047866,-0.009100,0.047921



==== DECISION CURVE (aggregated) ====


,threshold,nb_model_mean,nb_model_std,nb_all_mean,nb_none_mean
0,0.01,0.068010,0.001989,0.069338,0.0
1,0.02,0.059713,0.001625,0.059841,0.0
2,0.03,0.051757,0.002117,0.050149,0.0
3,0.04,0.044332,0.002754,0.040254,0.0
4,0.05,0.037182,0.005166,0.030152,0.0
5,0.06,0.030851,0.005262,0.019834,0.0
6,0.07,0.026971,0.004903,0.009295,0.0
7,0.08,0.021714,0.004745,-0.001474,0.0
8,0.09,0.017549,0.004708,-0.012479,0.0
9,0.10,0.012961,0.003933,-0.023729,0.0


In [6]:
!pip -q install catboost

import numpy as np
import pandas as pd
import gc, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

def _clip(p):
    return np.clip(np.asarray(p).astype(float), 1e-6, 1-1e-6)

def logit(p):
    p = _clip(p)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def nll_loss(y_true, p):
    y = np.asarray(y_true).astype(int)
    p = _clip(p)
    return float(-(y*np.log(p) + (1-y)*np.log(1-p)).mean())

def ece_score(y_true, p, n_bins=15):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        acc = y[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def tail_ece(y_true, p, top_frac=0.10, n_bins=10):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    n = len(p)
    k = max(1, int(np.ceil(top_frac*n)))
    idx = np.argsort(-p)[:k]
    return ece_score(y[idx], p[idx], n_bins=n_bins)

# ---- Temperature scaling from your notebook assumed: fit_temperature_scaling, apply_temperature ----
# If you already have them, no need to redefine.
# If not, paste your existing ones here.

def fit_platt_scaling(p_cal, y_cal):
    z = logit(p_cal).reshape(-1, 1)
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(z, y_cal.astype(int))
    return lr

def apply_platt(lr_model, p):
    z = logit(p).reshape(-1, 1)
    return lr_model.predict_proba(z)[:, 1]

def fit_isotonic(p_cal, y_cal):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_cal.astype(float), y_cal.astype(int))
    return iso

def apply_isotonic(iso_model, p):
    return iso_model.transform(p.astype(float))


def select_calibrator_by_trust(p_cal_raw, y_cal, ece_bins=15):
    """
    Choose Temp/Platt/Isotonic by trust-score on CAL:
      score = ECE + 0.5*TailECE10 + 0.2*Brier
    Tie-break by NLL.
    Returns: name, fitted_obj, optional_T
    """
    from sklearn.metrics import brier_score_loss

    p_cal_raw = _clip(p_cal_raw)
    y_cal = np.asarray(y_cal).astype(int)

    # TEMP
    T = fit_temperature_scaling(p_cal_raw, y_cal, max_iter=250, lr=0.05)
    p_temp = apply_temperature(p_cal_raw, T)

    # PLATT
    platt = fit_platt_scaling(p_cal_raw, y_cal)
    p_platt = apply_platt(platt, p_cal_raw)

    # ISO
    iso = fit_isotonic(p_cal_raw, y_cal)
    p_iso = apply_isotonic(iso, p_cal_raw)

    def trust_score(p):
        e = ece_score(y_cal, p, n_bins=ece_bins)
        te10 = tail_ece(y_cal, p, top_frac=0.10, n_bins=10)
        b = float(brier_score_loss(y_cal, p))
        return float(e + 0.5*te10 + 0.2*b)

    candidates = {
        "temp":   {"p": p_temp,  "obj": T,     "T": T},
        "platt":  {"p": p_platt, "obj": platt, "T": None},
        "isotonic":{"p": p_iso,  "obj": iso,   "T": None}
    }

    rows = []
    for name, d in candidates.items():
        p = d["p"]
        rows.append({
            "name": name,
            "trust": trust_score(p),
            "nll": nll_loss(y_cal, p),
            "ece": ece_score(y_cal, p, n_bins=ece_bins),
            "tail10": tail_ece(y_cal, p, top_frac=0.10, n_bins=10),
        })
    df = pd.DataFrame(rows).sort_values(["trust","nll"]).reset_index(drop=True)
    best = df.loc[0, "name"]
    return best, candidates[best]["obj"], candidates[best]["T"], df

def make_cb_ready_df(X, cat_cols, num_cols):
    X2 = X.copy()
    for c in cat_cols:
        X2[c] = X2[c].astype("string").fillna("MISSING").astype(str)
    for c in num_cols:
        # numeric fill with median computed outside (we'll compute per fold)
        X2[c] = pd.to_numeric(X2[c], errors="coerce")
    return X2

def train_catboost_teacher(X_tr, y_tr, X_va, y_va, cat_cols, num_cols, seed=42):
    Xtr = make_cb_ready_df(X_tr, cat_cols, num_cols)
    Xva = make_cb_ready_df(X_va, cat_cols, num_cols)

    # per-fold numeric imputation
    med = Xtr[num_cols].median()
    Xtr[num_cols] = Xtr[num_cols].fillna(med)
    Xva[num_cols] = Xva[num_cols].fillna(med)

    cat_features = [Xtr.columns.get_loc(c) for c in cat_cols]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=6000,
        learning_rate=0.02,
        depth=6,
        l2_leaf_reg=5.0,
        subsample=0.8,
        rsm=0.8,
        random_strength=1.0,
        bagging_temperature=1.0,
        grow_policy="Lossguide",
        random_seed=seed,
        verbose=False
    )

    tr_pool = Pool(Xtr, y_tr, cat_features=cat_features)
    va_pool = Pool(Xva, y_va, cat_features=cat_features)

    model.fit(tr_pool, eval_set=va_pool, use_best_model=True, early_stopping_rounds=300)
    return model, med

def cb_predict_proba(model, X_any, cat_cols, num_cols, med):
    Xp = make_cb_ready_df(X_any, cat_cols, num_cols)
    Xp[num_cols] = Xp[num_cols].fillna(med)
    return model.predict_proba(Xp)[:, 1]

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class DistillDataset(Dataset):
    def __init__(self, X_cat, X_num, y, tprob):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.t = torch.tensor(tprob, dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.X_cat[i], self.X_num[i], self.y[i], self.t[i]

def train_one_model_distill(
    Xtr_cat, Xtr_num, y_tr, t_tr,
    Xva_cat, Xva_num, y_va,
    cat_sizes,
    d_model=64, n_heads=4, n_layers=3, dropout=0.2,
    lr=2e-3, wd=1e-5, epochs=25, batch_size=256,
    pos_frac=0.30,
    alpha=0.7,          # weight on true-label loss
    tau=2.0,            # distillation temperature
    seed=42,
    device="cpu",
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    # ---- you must use YOUR transformer class here ----
    # It should take (x_cat, x_num) and output logits (N,)
    model = TabTransformer(
        cat_sizes=cat_sizes,
        n_num=Xtr_num.shape[1],
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        dropout=dropout
    ).to(device)

    # imbalance
    ytr = np.asarray(y_tr).astype(int)
    pos = ytr.sum()
    neg = len(ytr) - pos
    pos_weight = float(neg / max(1, pos))

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    ds_tr = DistillDataset(Xtr_cat, Xtr_num, y_tr, t_tr)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, drop_last=False)

    # validation (for early stop)
    Xva_cat_t = torch.tensor(Xva_cat, dtype=torch.long, device=device)
    Xva_num_t = torch.tensor(Xva_num, dtype=torch.float32, device=device)
    yva_t = torch.tensor(y_va, dtype=torch.float32, device=device)

    best = None
    best_metric = -1e9
    patience = 6
    bad = 0

    for ep in range(1, epochs+1):
        model.train()
        for xb_cat, xb_num, yb, tb in dl_tr:
            xb_cat = xb_cat.to(device)
            xb_num = xb_num.to(device)
            yb = yb.to(device)
            tb = tb.to(device)

            logits = model(xb_cat, xb_num).view(-1)

            # label loss (weighted BCE)
            bce = F.binary_cross_entropy_with_logits(
                logits, yb, pos_weight=torch.tensor(pos_weight, device=device)
            )

            # distill loss (KL on softened probs)
            with torch.no_grad():
                t_logits = logit(tb.cpu().numpy())
                t_logits = torch.tensor(t_logits, dtype=torch.float32, device=device)

            s_logp = F.logsigmoid(logits / tau)
            s_log1mp = F.logsigmoid(-logits / tau)

            t_p = torch.sigmoid(t_logits / tau)
            # KL between Bernoulli distributions
            kl = (t_p * (torch.log(t_p + 1e-12) - s_logp) +
                  (1 - t_p) * (torch.log(1 - t_p + 1e-12) - s_log1mp)).mean()

            loss = alpha * bce + (1 - alpha) * (tau * tau) * kl

            opt.zero_grad()
            loss.backward()
            opt.step()

        # ---- early stop on VAL PR-AUC (ranking) ----
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(Xva_cat_t, Xva_num_t).view(-1)).detach().cpu().numpy()
        pr_va = average_precision_score(y_va, p_va)

        if pr_va > best_metric + 1e-4:
            best_metric = pr_va
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best is not None:
        model.load_state_dict(best)
    return model

from sklearn.metrics import brier_score_loss

def run_cv_distill_trust(
    X, y, cat_cols, num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,
    seed=42,
    device="cpu",
    ece_bins=15,
    mc_samples=0   # keep 0 for speed, we can add MC later
):
    rng = np.random.RandomState(seed)
    all_rows = []
    all_risk = []

    for rep in range(1, n_repeats+1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.randint(0, 10_000_000)))

        for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # --- 3-way inner split ---
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=int(rng.randint(0, 10_000_000)))
            tr_sub_idx, hold_idx = next(sss1.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub_idx], y_tr_full[tr_sub_idx]
            X_hold, y_hold = X_tr_full.iloc[hold_idx], y_tr_full[hold_idx]

            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(rng.randint(0, 10_000_000)))
            val_idx, cal_idx = next(sss2.split(X_hold, y_hold))
            X_va, y_va = X_hold.iloc[val_idx], y_hold[val_idx]
            X_cal, y_cal = X_hold.iloc[cal_idx], y_hold[cal_idx]

            # --- teacher CatBoost (trained on TRAIN, early stop on VAL) ---
            teacher, med = train_catboost_teacher(X_tr, y_tr, X_va, y_va, cat_cols, num_cols, seed=int(rng.randint(0, 10_000_000)))

            # teacher probs for distillation (TRAIN only)
            t_tr = cb_predict_proba(teacher, X_tr, cat_cols, num_cols, med)
            # also for CAL and TEST (for evaluation only)
            t_cal = cb_predict_proba(teacher, X_cal, cat_cols, num_cols, med)
            t_te  = cb_predict_proba(teacher, X_te,  cat_cols, num_cols, med)

            # --- preprocessing for transformer (fit only on TRAIN) ---
            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xcal_cat, Xcal_num = prep.transform(X_cal)
            Xte_cat, Xte_num = prep.transform(X_te)

            # --- ensemble students ---
            p_cal_list, p_te_list = [], []

            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_one_model_distill(
                    Xtr_cat, Xtr_num, y_tr, t_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                    lr=2e-3, wd=1e-5, epochs=25, batch_size=256, pos_frac=0.30,
                    alpha=0.7, tau=2.0,
                    seed=seed_m, device=device
                )

                p_cal_list.append(predict_proba(model, Xcal_cat, Xcal_num))
                p_te_list.append(predict_proba(model, Xte_cat, Xte_num))

                del model
                gc.collect()

            p_cal_raw = np.mean(np.stack(p_cal_list, axis=0), axis=0)
            p_te_raw  = np.mean(np.stack(p_te_list, axis=0), axis=0)

            # --- choose calibrator by TRUST score on CAL ---
            cal_name, cal_obj, T, cal_table = select_calibrator_by_trust(p_cal_raw, y_cal, ece_bins=ece_bins)

            if cal_name == "temp":
                p_te = apply_temperature(p_te_raw, T)
            elif cal_name == "platt":
                p_te = apply_platt(cal_obj, p_te_raw)
            else:
                p_te = apply_isotonic(cal_obj, p_te_raw)

            pr = average_precision_score(y_te, p_te)
            roc = roc_auc_score(y_te, p_te)
            brier = float(brier_score_loss(y_te, p_te))
            ece = ece_score(y_te, p_te, n_bins=ece_bins)
            te10 = tail_ece(y_te, p_te, top_frac=0.10, n_bins=10)
            te20 = tail_ece(y_te, p_te, top_frac=0.20, n_bins=10)

            all_rows.append({
                "rep": rep, "fold": fold,
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "tail_ece10": float(te10),
                "tail_ece20": float(te20),
                "calibrator": cal_name,
                "pos_rate": float(np.mean(y_te)),
            })

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"Distill+Cal({cal_name}) PR={pr:.4f} ROC={roc:.4f} "
                  f"Brier={brier:.4f} ECE={ece:.4f} Tail10={te10:.4f}")

    df = pd.DataFrame(all_rows)

    print("\n==== OVERALL (Distillation + Trust-Calibrator) ====")
    print("Calibrator counts:\n", df["calibrator"].value_counts())
    print(f"PR-AUC:     {df['pr_auc'].mean():.4f} ± {df['pr_auc'].std(ddof=1):.4f}")
    print(f"ROC-AUC:    {df['roc_auc'].mean():.4f} ± {df['roc_auc'].std(ddof=1):.4f}")
    print(f"Brier:      {df['brier'].mean():.4f} ± {df['brier'].std(ddof=1):.4f}")
    print(f"ECE:        {df['ece'].mean():.4f} ± {df['ece'].std(ddof=1):.4f}")
    print(f"TailECE10:  {df['tail_ece10'].mean():.4f} ± {df['tail_ece10'].std(ddof=1):.4f}")
    print(f"TailECE20:  {df['tail_ece20'].mean():.4f} ± {df['tail_ece20'].std(ddof=1):.4f}")

    return df

df_distill = run_cv_distill_trust(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,
    seed=42,
    device="cpu",
    ece_bins=15
)
df_distill.sort_values("pr_auc", ascending=False).head(10)

[rep 1/2 fold 1/5] Distill+Cal(isotonic) PR=0.1366 ROC=0.6791 Brier=0.0697 ECE=0.0091 Tail10=0.0077
[rep 1/2 fold 2/5] Distill+Cal(isotonic) PR=0.1584 ROC=0.7062 Brier=0.0693 ECE=0.0076 Tail10=0.0556
[rep 1/2 fold 3/5] Distill+Cal(platt) PR=0.2062 ROC=0.6857 Brier=0.0700 ECE=0.0059 Tail10=0.0644
[rep 1/2 fold 4/5] Distill+Cal(isotonic) PR=0.1350 ROC=0.6784 Brier=0.0713 ECE=0.0169 Tail10=0.0798
[rep 1/2 fold 5/5] Distill+Cal(isotonic) PR=0.1183 ROC=0.6252 Brier=0.0725 ECE=0.0247 Tail10=0.1198
[rep 2/2 fold 1/5] Distill+Cal(isotonic) PR=0.1664 ROC=0.7480 Brier=0.0683 ECE=0.0182 Tail10=0.0570
[rep 2/2 fold 2/5] Distill+Cal(isotonic) PR=0.1323 ROC=0.6308 Brier=0.0756 ECE=0.0252 Tail10=0.1088
[rep 2/2 fold 3/5] Distill+Cal(isotonic) PR=0.1467 ROC=0.6788 Brier=0.0710 ECE=0.0252 Tail10=0.0448
[rep 2/2 fold 4/5] Distill+Cal(isotonic) PR=0.1379 ROC=0.6855 Brier=0.0714 ECE=0.0156 Tail10=0.0951
[rep 2/2 fold 5/5] Distill+Cal(isotonic) PR=0.1499 ROC=0.6969 Brier=0.0733 ECE=0.0286 Tail10=0.1142

==

,rep,fold,pr_auc,roc_auc,brier,ece,tail_ece10,tail_ece20,calibrator,pos_rate
2,1,3,0.206228,0.685723,0.070034,0.005888,0.064422,0.002325,platt,0.079262
5,2,1,0.166424,0.748037,0.068287,0.018165,0.057021,0.057030,isotonic,0.078176
1,1,2,0.158385,0.706158,0.069299,0.007630,0.055607,0.043089,isotonic,0.079262
9,2,5,0.149905,0.696918,0.073308,0.028647,0.114178,0.058775,isotonic,0.078261
7,2,3,0.146745,0.678753,0.070956,0.025246,0.044755,0.020413,isotonic,0.079262
8,2,4,0.137903,0.685526,0.071430,0.015553,0.095086,0.062481,isotonic,0.078261
0,1,1,0.136556,0.679132,0.069693,0.009138,0.007713,0.000908,isotonic,0.078176
3,1,4,0.134995,0.678353,0.071312,0.016855,0.079796,0.037702,isotonic,0.078261
6,2,2,0.132298,0.630815,0.075639,0.025191,0.108768,0.074464,isotonic,0.079262
4,1,5,0.118259,0.625205,0.072547,0.024704,0.119799,0.088864,isotonic,0.078261


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip -q install scikit-learn pandas numpy matplotlib

In [2]:
import os, gc, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

OUTDIR = "/kaggle/working/paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Example placeholder:
df = pd.read_csv("/kaggle/input/stroke-dataset/Stroke.csv")
y = df["stroke"].values.astype(int)
X = df.drop(columns=["stroke"])

# --- YOU MUST SET X and y here ---
assert "X" in globals() and "y" in globals(), "Please define X (DataFrame) and y (0/1 array) before proceeding."

y = np.asarray(y).astype(int).reshape(-1)
X = X.reset_index(drop=True)

print("N:", len(X), "Pos rate:", y.mean())
print("X shape:", X.shape)

OUTDIR: /kaggle/working/paper_outputs
DEVICE: cpu
N: 4603 Pos rate: 0.07864436237236584
X shape: (4603, 35)


In [4]:
def infer_cat_num_cols(X: pd.DataFrame, max_unique_as_cat=30):
    cat_cols, num_cols = [], []
    for c in X.columns:
        s = X[c]
        if s.dtype == "object" or str(s.dtype).startswith("category") or str(s.dtype).startswith("string"):
            cat_cols.append(c)
        else:
            # if integer with low unique count, treat as categorical
            nun = s.nunique(dropna=True)
            if pd.api.types.is_integer_dtype(s) and nun <= max_unique_as_cat:
                cat_cols.append(c)
            else:
                num_cols.append(c)
    return cat_cols, num_cols

cat_cols, num_cols = infer_cat_num_cols(X)
print("n_cat:", len(cat_cols), "n_num:", len(num_cols))

class FoldPreprocessor:
    def __init__(self, cat_cols, num_cols):
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.cat_maps = {}
        self.cat_sizes = []
        self.num_median = None
        self.num_mean = None
        self.num_std = None

    def fit(self, Xtr: pd.DataFrame):
        # numeric stats
        Xn = Xtr[self.num_cols].copy()
        Xn = Xn.apply(pd.to_numeric, errors="coerce")
        self.num_median = Xn.median()
        Xn = Xn.fillna(self.num_median)
        self.num_mean = Xn.mean()
        self.num_std = Xn.std().replace(0, 1.0)

        # cat maps
        self.cat_maps = {}
        self.cat_sizes = []
        for c in self.cat_cols:
            s = Xtr[c].astype("object").fillna("MISSING").astype(str)
            uniq = pd.Index(s.unique())
            mapping = {k: i+1 for i, k in enumerate(uniq)}  # 0 reserved for OOV
            self.cat_maps[c] = mapping
            self.cat_sizes.append(len(mapping) + 1)  # +OOV
        return self

    def transform(self, X: pd.DataFrame):
        # cats
        Xc = []
        for c in self.cat_cols:
            s = X[c].astype("object").fillna("MISSING").astype(str)
            mp = self.cat_maps[c]
            arr = s.map(mp).fillna(0).astype(int).values
            Xc.append(arr)
        Xc = np.stack(Xc, axis=1) if len(Xc) else np.zeros((len(X), 0), dtype=np.int64)

        # nums
        Xn = X[self.num_cols].copy()
        Xn = Xn.apply(pd.to_numeric, errors="coerce")
        Xn = Xn.fillna(self.num_median)
        Xn = (Xn - self.num_mean) / self.num_std
        Xn = Xn.values.astype(np.float32)

        return Xc.astype(np.int64), Xn.astype(np.float32)

n_cat: 15 n_num: 20


In [9]:
class TabTransformer(nn.Module):
    def __init__(self, cat_sizes, n_num, d_model=64, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.n_cat = len(cat_sizes)
        self.n_num = n_num

        # embeddings for each categorical feature
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(sz, d_model) for sz in cat_sizes
        ])

        # numeric token
        self.num_proj = nn.Linear(n_num, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True,
            activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x_cat, x_num):
        tokens = []
        # cat tokens
        for j, emb in enumerate(self.cat_embeds):
            tokens.append(emb(x_cat[:, j]))
        # numeric token
        tokens.append(self.num_proj(x_num))

        T = torch.stack(tokens, dim=1)  # (B, n_cat+1, d_model)
        Z = self.encoder(T)             # (B, tokens, d_model)
        pooled = Z.mean(dim=1)          # mean pool
        logits = self.head(pooled).squeeze(-1)
        return logits

class TabDataset(Dataset):
    def __init__(self, Xc, Xn, y):
        self.Xc = torch.tensor(Xc, dtype=torch.long)
        self.Xn = torch.tensor(Xn, dtype=torch.float32)
        self.y  = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.Xc[i], self.Xn[i], self.y[i]

@torch.no_grad()
def predict_proba(model, Xc, Xn, batch_size=1024, device=DEVICE):
    model.eval()
    dl = DataLoader(TabDataset(Xc, Xn, np.zeros(len(Xc))), batch_size=batch_size, shuffle=False)
    out = []
    for xb_cat, xb_num, _ in dl:
        xb_cat = xb_cat.to(device)
        xb_num = xb_num.to(device)
        logits = model(xb_cat, xb_num)
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out)

def train_model(
    Xtr_cat, Xtr_num, y_tr,
    Xva_cat, Xva_num, y_va,
    cat_sizes,
    d_model=64, n_heads=4, n_layers=3, dropout=0.2,
    lr=2e-3, wd=1e-5, epochs=30, batch_size=256,
    seed=42,
    device=DEVICE
):
    seed_everything(seed)

    model = TabTransformer(cat_sizes, n_num=Xtr_num.shape[1], d_model=d_model, n_heads=n_heads,
                           n_layers=n_layers, dropout=dropout).to(device)

    # imbalance
    ytr = np.asarray(y_tr).astype(int)
    pos = ytr.sum()
    neg = len(ytr) - pos
    pos_weight = torch.tensor([neg / max(1, pos)], dtype=torch.float32, device=device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    ds_tr = TabDataset(Xtr_cat, Xtr_num, y_tr)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, drop_last=False)

    # early stop on val PR-AUC
    best_state = None
    best_pr = -1
    bad = 0
    patience = 6

    Xva_cat_t = torch.tensor(Xva_cat, dtype=torch.long, device=device)
    Xva_num_t = torch.tensor(Xva_num, dtype=torch.float32, device=device)

    for ep in range(1, epochs+1):
        model.train()
        for xb_cat, xb_num, yb in dl_tr:
            xb_cat = xb_cat.to(device)
            xb_num = xb_num.to(device)
            yb = yb.to(device)

            logits = model(xb_cat, xb_num)
            loss = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_weight)

            opt.zero_grad()
            loss.backward()
            opt.step()

        # val
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(Xva_cat_t, Xva_num_t)).detach().cpu().numpy()

        pr = average_precision_score(y_va, p_va)
        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def _clip(p): return np.clip(np.asarray(p).astype(float), 1e-6, 1-1e-6)

def ece_score(y_true, p, n_bins=15):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        m = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if m.sum() == 0: 
            continue
        acc = y[m].mean()
        conf = p[m].mean()
        ece += (m.sum()/len(p)) * abs(acc - conf)
    return float(ece)

def tail_ece(y_true, p, top_frac=0.10, n_bins=10):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    k = max(1, int(np.ceil(top_frac*len(p))))
    idx = np.argsort(-p)[:k]
    return ece_score(y[idx], p[idx], n_bins=n_bins)

def nll_loss(y_true, p):
    y = np.asarray(y_true).astype(int)
    p = _clip(p)
    return float(-(y*np.log(p)+(1-y)*np.log(1-p)).mean())

# --- Temperature scaling ---
def fit_temperature_scaling(p_cal, y_cal, max_iter=200, lr=0.05):
    # optimize T on logits
    p_cal = _clip(p_cal)
    z = np.log(p_cal/(1-p_cal)).astype(np.float32)
    y = torch.tensor(y_cal.astype(np.float32), device=DEVICE)
    zt = torch.tensor(z, device=DEVICE)

    T = torch.tensor([1.0], device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([T], lr=lr)

    for _ in range(max_iter):
        opt.zero_grad()
        logits = zt / T.clamp_min(1e-3)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        opt.step()

    return float(T.detach().cpu().numpy()[0])

def apply_temperature(p, T):
    p = _clip(p)
    z = np.log(p/(1-p))
    z = z / max(1e-3, T)
    return _clip(1/(1+np.exp(-z)))

# --- Platt scaling ---
from sklearn.linear_model import LogisticRegression
def fit_platt(p_cal, y_cal):
    z = np.log(_clip(p_cal)/(1-_clip(p_cal))).reshape(-1,1)
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(z, y_cal.astype(int))
    return lr

def apply_platt(lr_model, p):
    z = np.log(_clip(p)/(1-_clip(p))).reshape(-1,1)
    return lr_model.predict_proba(z)[:,1]

# --- Isotonic ---
from sklearn.isotonic import IsotonicRegression
def fit_isotonic(p_cal, y_cal):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(_clip(p_cal), y_cal.astype(int))
    return iso

def apply_isotonic(iso_model, p):
    return _clip(iso_model.transform(_clip(p)))

def select_calibrator_trust(p_cal, y_cal, ece_bins=15):
    # candidates
    T = fit_temperature_scaling(p_cal, y_cal)
    p_temp = apply_temperature(p_cal, T)

    pl = fit_platt(p_cal, y_cal)
    p_platt = apply_platt(pl, p_cal)

    iso = fit_isotonic(p_cal, y_cal)
    p_iso = apply_isotonic(iso, p_cal)

    def trust_score(p):
        e = ece_score(y_cal, p, n_bins=ece_bins)
        te10 = tail_ece(y_cal, p, top_frac=0.10, n_bins=10)
        b = brier_score_loss(y_cal, p)
        return float(e + 0.5*te10 + 0.2*b)

    cand = [
        ("temp", T, p_temp),
        ("platt", pl, p_platt),
        ("isotonic", iso, p_iso)
    ]

    rows=[]
    for name, obj, pc in cand:
        rows.append({
            "name": name,
            "trust": trust_score(pc),
            "nll": nll_loss(y_cal, pc),
            "ece": ece_score(y_cal, pc, n_bins=ece_bins),
            "tail10": tail_ece(y_cal, pc, top_frac=0.10),
        })
    df = pd.DataFrame(rows).sort_values(["trust","nll"]).reset_index(drop=True)
    best = df.loc[0,"name"]
    best_obj = {n:o for n,o,_ in cand}[best]
    return best, best_obj, df

def apply_calibrator(name, obj, p):
    if name=="temp":
        return apply_temperature(p, obj)
    if name=="platt":
        return apply_platt(obj, p)
    return apply_isotonic(obj, p)

def risk_stratification(y_true, p, top_fracs=(0.05,0.10,0.20,0.30,0.50)):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    base = y.mean()
    rows=[]
    order = np.argsort(-p)
    n = len(y)
    for f in top_fracs:
        k = max(1, int(np.ceil(f*n)))
        idx = order[:k]
        risk = y[idx].mean()
        lift = risk / max(1e-12, base)
        capture = y[idx].sum() / max(1, y.sum())
        rows.append({
            "top_frac": f,
            "base_rate": base,
            "risk": risk,
            "lift": lift,
            "capture": capture,
            "n_top": k
        })
    return pd.DataFrame(rows)

def decision_curve(y_true, p, thresholds=np.linspace(0.01, 0.10, 10)):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    N = len(y)
    rows=[]
    for t in thresholds:
        pred = (p >= t).astype(int)
        TP = int(((pred==1)&(y==1)).sum())
        FP = int(((pred==1)&(y==0)).sum())
        nb_model = (TP/N) - (FP/N)*(t/(1-t))
        # treat-all
        TP_all = int(y.sum())
        FP_all = int((1-y).sum())
        nb_all = (TP_all/N) - (FP_all/N)*(t/(1-t))
        rows.append({"threshold": float(t), "nb_model": nb_model, "nb_all": nb_all, "nb_none": 0.0})
    return pd.DataFrame(rows)

import matplotlib.pyplot as plt

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f"{name}.png"), dpi=600)
    plt.savefig(os.path.join(OUTDIR, f"{name}.pdf"))
    plt.close()

def plot_calibration(y, p, n_bins=10, title="Calibration"):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)

    bins = np.linspace(0, 1, n_bins+1)
    bin_ids = np.digitize(p, bins) - 1
    xs, ys, ns = [], [], []
    for b in range(n_bins):
        m = (bin_ids == b)
        if m.sum()==0: 
            continue
        xs.append(p[m].mean())
        ys.append(y[m].mean())
        ns.append(m.sum())

    plt.figure(figsize=(5.2,4.6))
    plt.plot([0,1],[0,1], linestyle="--")
    plt.scatter(xs, ys, s=40)
    plt.plot(xs, ys)
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed event rate")
    plt.title(title)

def plot_risk_table(df_risk, title="Risk stratification"):
    plt.figure(figsize=(6.2,4.6))
    plt.plot(df_risk["top_frac"]*100, df_risk["lift"], marker="o")
    plt.xlabel("Top-K% predicted risk")
    plt.ylabel("Lift vs base rate")
    plt.title(title)

def plot_dca(df_dca, title="Decision Curve"):
    plt.figure(figsize=(6.2,4.6))
    plt.plot(df_dca["threshold"], df_dca["nb_model"], marker="o", label="Model")
    plt.plot(df_dca["threshold"], df_dca["nb_all"], linestyle="--", label="Treat-all")
    plt.plot(df_dca["threshold"], df_dca["nb_none"], linestyle=":", label="Treat-none")
    plt.xlabel("Threshold probability")
    plt.ylabel("Net benefit")
    plt.title(title)
    plt.legend()

def run_cv_trust_stroke(
    X, y, cat_cols, num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,
    seed=42,
    ece_bins=15
):
    rng = np.random.RandomState(seed)

    rows=[]
    all_oof_p=[]
    all_oof_y=[]
    all_risk=[]
    all_dca=[]

    for rep in range(1, n_repeats+1):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=int(rng.randint(0, 10_000_000)))

        for fold,(tr_idx,te_idx) in enumerate(skf.split(X,y), start=1):
            X_tr_full, y_tr_full = X.iloc[tr_idx], y[tr_idx]
            X_te, y_te = X.iloc[te_idx], y[te_idx]

            # ---- inner split: train vs holdout (val+cal) ----
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.40, random_state=int(rng.randint(0, 10_000_000)))
            tr_sub, hold = next(sss1.split(X_tr_full, y_tr_full))
            X_tr, y_tr = X_tr_full.iloc[tr_sub], y_tr_full[tr_sub]
            X_hold, y_hold = X_tr_full.iloc[hold], y_tr_full[hold]

            # ---- split holdout into val and cal ----
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=int(rng.randint(0, 10_000_000)))
            va_i, cal_i = next(sss2.split(X_hold, y_hold))
            X_va, y_va = X_hold.iloc[va_i], y_hold[va_i]
            X_cal, y_cal = X_hold.iloc[cal_i], y_hold[cal_i]

            # ---- preprocess (fit on TRAIN only) ----
            prep = FoldPreprocessor(cat_cols, num_cols).fit(X_tr)
            Xtr_cat, Xtr_num = prep.transform(X_tr)
            Xva_cat, Xva_num = prep.transform(X_va)
            Xcal_cat, Xcal_num = prep.transform(X_cal)
            Xte_cat, Xte_num = prep.transform(X_te)

            # ---- ensemble training ----
            p_cal_list=[]
            p_te_list=[]

            for m in range(ensemble_M):
                seed_m = int(rng.randint(0, 10_000_000))
                model = train_model(
                    Xtr_cat, Xtr_num, y_tr,
                    Xva_cat, Xva_num, y_va,
                    cat_sizes=prep.cat_sizes,
                    d_model=64, n_heads=4, n_layers=3, dropout=0.20,
                    lr=2e-3, wd=1e-5, epochs=30, batch_size=256,
                    seed=seed_m, device=DEVICE
                )

                p_cal_list.append(predict_proba(model, Xcal_cat, Xcal_num))
                p_te_list.append(predict_proba(model, Xte_cat, Xte_num))

                del model
                gc.collect()

            p_cal_raw = np.mean(np.stack(p_cal_list,0),0)
            p_te_raw  = np.mean(np.stack(p_te_list,0),0)

            # ---- calibrate using CAL with trust selector ----
            cal_name, cal_obj, cal_df = select_calibrator_trust(p_cal_raw, y_cal, ece_bins=ece_bins)
            p_te = apply_calibrator(cal_name, cal_obj, p_te_raw)

            # ---- metrics ----
            pr  = average_precision_score(y_te, p_te)
            roc = roc_auc_score(y_te, p_te)
            brier = float(brier_score_loss(y_te, p_te))
            ece = ece_score(y_te, p_te, n_bins=ece_bins)
            te10 = tail_ece(y_te, p_te, top_frac=0.10, n_bins=10)
            te20 = tail_ece(y_te, p_te, top_frac=0.20, n_bins=10)

            print(f"[rep {rep}/{n_repeats} fold {fold}/{n_splits}] "
                  f"Cal={cal_name} PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f} "
                  f"ECE={ece:.4f} Tail10={te10:.4f}")

            rows.append({
                "rep": rep, "fold": fold,
                "calibrator": cal_name,
                "pr_auc": float(pr),
                "roc_auc": float(roc),
                "brier": float(brier),
                "ece": float(ece),
                "tail_ece10": float(te10),
                "tail_ece20": float(te20),
                "pos_rate": float(y_te.mean()),
                "n_test": int(len(y_te)),
            })

            # store pooled (for paper plots)
            all_oof_y.append(y_te.copy())
            all_oof_p.append(p_te.copy())

            # risk & dca for this fold
            all_risk.append(risk_stratification(y_te, p_te))
            all_dca.append(decision_curve(y_te, p_te))

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTDIR, "cv_fold_metrics.csv"), index=False)

    # overall summary table
    summary = df[["pr_auc","roc_auc","brier","ece","tail_ece10","tail_ece20"]].agg(["mean","std"])
    summary.to_csv(os.path.join(OUTDIR, "overall_summary.csv"))
    print("\n==== OVERALL SUMMARY ====")
    display(summary)

    # aggregate pooled preds
    y_all = np.concatenate(all_oof_y)
    p_all = np.concatenate(all_oof_p)

    # aggregated risk & dca
    risk_df = pd.concat(all_risk, axis=0).groupby("top_frac").agg(
        base_rate=("base_rate","mean"),
        risk=("risk","mean"),
        lift=("lift","mean"),
        capture=("capture","mean"),
        n_top=("n_top","mean")
    ).reset_index()
    risk_df.to_csv(os.path.join(OUTDIR, "risk_table.csv"), index=False)

    dca_df = pd.concat(all_dca, axis=0).groupby("threshold").agg(
        nb_model=("nb_model","mean"),
        nb_all=("nb_all","mean"),
        nb_none=("nb_none","mean"),
    ).reset_index()
    dca_df.to_csv(os.path.join(OUTDIR, "decision_curve.csv"), index=False)

    # figures
    plot_calibration(y_all, p_all, n_bins=12, title="TRUST-STROKE Calibration (pooled OOF)")
    savefig("fig_calibration")

    plot_risk_table(risk_df, title="Risk Stratification (Lift vs Top-K%)")
    savefig("fig_risk_lift")

    plot_dca(dca_df, title="Decision Curve Analysis (pooled)")
    savefig("fig_dca")

    print("\nSaved outputs to:", OUTDIR)
    return df, summary, risk_df, dca_df

In [10]:
df_cv, summary, risk_df, dca_df = run_cv_trust_stroke(
    X, y, cat_cols=cat_cols, num_cols=num_cols,
    n_splits=5, n_repeats=2,
    ensemble_M=3,
    seed=SEED,
    ece_bins=15
)

display(df_cv.sort_values("pr_auc", ascending=False).head(10))
display(risk_df)
display(dca_df.head())

[rep 1/2 fold 1/5] Cal=isotonic PR=0.1552 ROC=0.7324 Brier=0.0685 ECE=0.0116 Tail10=0.0118
[rep 1/2 fold 2/5] Cal=isotonic PR=0.1577 ROC=0.7257 Brier=0.0702 ECE=0.0094 Tail10=0.0270
[rep 1/2 fold 3/5] Cal=isotonic PR=0.1434 ROC=0.6659 Brier=0.0720 ECE=0.0162 Tail10=0.0488
[rep 1/2 fold 4/5] Cal=isotonic PR=0.1342 ROC=0.6787 Brier=0.0706 ECE=0.0171 Tail10=0.0009
[rep 1/2 fold 5/5] Cal=isotonic PR=0.1281 ROC=0.6798 Brier=0.0720 ECE=0.0263 Tail10=0.0788
[rep 2/2 fold 1/5] Cal=isotonic PR=0.2007 ROC=0.7662 Brier=0.0665 ECE=0.0160 Tail10=0.0570
[rep 2/2 fold 2/5] Cal=isotonic PR=0.1392 ROC=0.6828 Brier=0.0709 ECE=0.0129 Tail10=0.0851
[rep 2/2 fold 3/5] Cal=isotonic PR=0.1403 ROC=0.6930 Brier=0.0705 ECE=0.0064 Tail10=0.0210
[rep 2/2 fold 4/5] Cal=isotonic PR=0.1295 ROC=0.6756 Brier=0.0732 ECE=0.0224 Tail10=0.0895
[rep 2/2 fold 5/5] Cal=isotonic PR=0.1233 ROC=0.6534 Brier=0.0710 ECE=0.0117 Tail10=0.0275

==== OVERALL SUMMARY ====


,pr_auc,roc_auc,brier,ece,tail_ece10,tail_ece20
mean,0.145156,0.695349,0.070548,0.014997,0.044740,0.036729
std,0.022453,0.034988,0.001894,0.005974,0.031869,0.025963



Saved outputs to: /kaggle/working/paper_outputs


,rep,fold,calibrator,pr_auc,roc_auc,brier,ece,tail_ece10,tail_ece20,pos_rate,n_test
5,2,1,isotonic,0.200669,0.766236,0.066544,0.016004,0.057041,0.048083,0.078176,921
1,1,2,isotonic,0.157689,0.725656,0.070216,0.009387,0.026951,0.015830,0.079262,921
0,1,1,isotonic,0.155226,0.732447,0.068468,0.011571,0.011800,0.000176,0.078176,921
2,1,3,isotonic,0.143357,0.665910,0.071965,0.016182,0.048814,0.032991,0.079262,921
7,2,3,isotonic,0.140312,0.692976,0.070518,0.006407,0.021043,0.034775,0.079262,921
6,2,2,isotonic,0.139246,0.682775,0.070944,0.012946,0.085086,0.048572,0.079262,921
3,1,4,isotonic,0.134192,0.678656,0.070576,0.017118,0.000864,0.004412,0.078261,920
8,2,4,isotonic,0.129529,0.675593,0.073243,0.022416,0.089503,0.074204,0.078261,920
4,1,5,isotonic,0.128085,0.679810,0.072042,0.026268,0.078759,0.075949,0.078261,920
9,2,5,isotonic,0.123254,0.653433,0.070964,0.011676,0.027540,0.032293,0.078261,920


,top_frac,base_rate,risk,lift,capture,n_top
0,0.05,0.078644,0.190749,2.425674,0.122926,46.6
1,0.10,0.078644,0.179231,2.279464,0.229319,92.6
2,0.20,0.078644,0.151648,1.928271,0.386720,184.6
3,0.30,0.078644,0.140263,1.783660,0.535940,276.6
4,0.50,0.078644,0.118969,1.512754,0.756887,460.6


,threshold,nb_model,nb_all,nb_none
0,0.01,0.068157,0.069338,0.0
1,0.02,0.059079,0.059841,0.0
2,0.03,0.051400,0.050149,0.0
3,0.04,0.044898,0.040254,0.0
4,0.05,0.038984,0.030152,0.0
